# Deep learning for quantitative TEM: atomic localization and strain mapping

End-to-end research notebook for TEM-ImageNet-v1.3:

1. optional self-supervised Noise2Void pre-training
2. multi-head denoising + atomic-column segmentation with AtomSegNet, UNet++ and HRNet
3. 5-fold cross-validation and paired N2V/no-N2V evaluation
4. atomic-column localization and controlled shift-recovery precision tests
5. lattice fitting, displacement fields and 2D strain analysis
6. unlabeled simulation-to-experiment transfer diagnostics using model consensus,
   disagreement and FFT lattice-frequency preservation

Dataset: https://github.com/xinhuolin/TEM-ImageNet-v1.3

The notebook is designed for a single CUDA-capable GPU and long runs are resumable.
Paths are configured through environment variables so the repository is portable.
See the repository README before running the notebook.


## 0. Environment, imports, determinism

In [ ]:
# ============================================================
# 0. Environment, imports, determinism, device
#
# CUDA_VISIBLE_DEVICES and the thread caps must be set BEFORE numpy/torch/cv2
# are imported anywhere, or every DataLoader worker spawns with
# OMP_NUM_THREADS=<all cores> and the machine thrashes.
# ============================================================
import os
import sys
import platform

os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # single GPU on this machine

_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import gc
import re
import json
import math
import time
import random
import shutil
import socket
import signal
import warnings
import tempfile
import subprocess
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
# Keep the inline backend inside a Jupyter kernel so figures render; fall back
# to Agg only for a genuinely headless `python script.py` run.
if "ipykernel" not in sys.modules and os.name != "nt" and not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    sns = None

try:
    import cv2
    cv2.setNumThreads(0)          # OpenCV's own pool fights the DataLoader workers
except ImportError as e:
    raise ImportError(
        "cv2 is missing. Install it into the same environment as this kernel:\n"
        "    pip install opencv-python"
    ) from e

from PIL import Image
Image.MAX_IMAGE_PIXELS = None

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import KFold
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
from skimage.feature import peak_local_max
from scipy.spatial import cKDTree
from scipy.optimize import curve_fit, linear_sum_assignment
from scipy.ndimage import maximum_filter, gaussian_filter
from scipy import stats

try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------- reproducibility ----------------
SEED = 42


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)


def worker_init_fn(worker_id):
    """PyTorch reseeds `torch` and `random` in each fork but NOT numpy. Without
    this, every worker draws the identical Noise2Void mask and the identical
    augmentation stream, so 4 workers give you 1 worker's worth of randomness."""
    base = torch.initial_seed() % (2 ** 31 - 1)
    np.random.seed((base + worker_id) % (2 ** 31 - 1))
    random.seed(base + worker_id)


# ---------------- device ----------------
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True          # fixed 256x256 shapes -> safe
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

# bfloat16 needs compute capability >= 8.0 (Ampere and later, so an RTX 30xx at
# sm_86 qualifies just as an A100 at sm_80 does). It has fp32's exponent range,
# so it needs no GradScaler and cannot produce inf-loss steps.
BF16_OK = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()

print("Python      :", sys.version.split()[0])
print("Host        :", socket.gethostname())
print("Platform    :", platform.system(), platform.release())
print("CPUs        :", _CPUS)
print("torch       :", torch.__version__, "| CUDA build", torch.version.cuda)
print("device      :", DEVICE, "| visible GPUs", torch.cuda.device_count())

if DEVICE.type == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print("GPU         :", _p.name, f"| {_p.total_memory / 1024**3:.1f} GiB",
          f"| sm_{_p.major}{_p.minor}")
    print("bfloat16    :", BF16_OK)
    try:
        print(subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu",
             "--format=csv"], capture_output=True, text=True).stdout)
    except (FileNotFoundError, OSError) as e:
        print("nvidia-smi not on PATH:", e)
else:
    assert os.environ.get("ALLOW_CPU") == "1", (
        "No GPU visible. Check the NVIDIA driver and that torch has a CUDA "
        "build (torch.version.cuda is not None), or set ALLOW_CPU=1 to run on "
        "CPU, which will be very slow.")

## 1. Configuration

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
_cwd = Path.cwd().resolve()
_DEFAULT_PROJECT_ROOT = _cwd.parent if _cwd.name.lower() == "notebooks" else _cwd
PROJECT_ROOT = Path(os.environ.get("TEM_PROJECT_ROOT", str(_DEFAULT_PROJECT_ROOT)))
DATA_ROOT_DEFAULT = os.environ.get(
    "TEM_DATA_ROOT", str(PROJECT_ROOT / "data" / "TEM-ImageNet-v1.3"))

# Everything a run produces lands under this root, not under the code dir.
RESULTS_ROOT = Path(os.environ.get("TEM_RESULTS_ROOT", str(PROJECT_ROOT / "results")))
OUT_DIR_DEFAULT = os.environ.get("TEM_OUT_DIR", str(RESULTS_ROOT))


def _pick_scratch():
    """Where the uint8 memmap cache lives. The project disk is already the fast
    path locally; fall back to the OS temp directory if it is not writable."""
    for d in (PROJECT_ROOT / "cache",
              Path(tempfile.gettempdir()) / "tem_cache",
              Path.home() / "tem_cache"):
        try:
            d.mkdir(parents=True, exist_ok=True)
            if os.access(d, os.W_OK):
                return d
        except OSError:
            continue
    return Path(tempfile.gettempdir())


class CFG:
    # ---------------- paths ----------------
    PROJECT_ROOT = PROJECT_ROOT
    DATA_ROOT    = DATA_ROOT_DEFAULT
    OUT_DIR      = OUT_DIR_DEFAULT
    CACHE_DIR    = str(_pick_scratch())
    RESULTS_ROOT = RESULTS_ROOT

    # ---------------- results tree ----------------
    # key used in code -> folder name on disk (spaces are deliberate, they
    # retained for compatibility with existing run folders)
    RESULT_VARIANTS = {
        "without_n2v": "without n2v",
        "with_n2v":    "with n2v",
    }

    # Leaf folders inside each variant. To add one later, append it here and
    # re-run this cell, or call register_result_folder() at runtime.
    RESULT_LEAVES = {
        "without_n2v": ["AtomSegNet", "HRNet", "UNetPP", "JSON"],
        "with_n2v":    ["AtomSegNet", "HRNet", "UNetPP", "JSON", "N2V"],
    }

    # notebook arch name -> folder name (aliases tolerated)
    ARCH_DIRNAME = {
        "AtomSegNet": "AtomSegNet",
        "UNetPP":     "UNetPP",
        "UNet++":     "UNetPP",
        "HRNet":      "HRNet",
    }

    FOLD_FMT = "fold_{:02d}"           # per-fold subfolder under each arch

    # ---------------- dataset subfolders ----------------
    SUBDIR_NOISY  = "image"
    SUBDIR_CLEAN  = "noNoise"
    SUBDIR_MASK   = "circularMask"     # binary segmentation target
    SUBDIR_GAUSS  = "gaussianMask"     # sub-pixel atom-position target
    SUBDIR_COORDS = "position"         # NOTE: unit-cell vectors, NOT atom xy

    # position/ holds lattice vectors, not per-atom coordinates, so gaussianMask
    # is the correct source of localization ground truth.
    LOCALIZATION_GT = "gaussianMask"

    MAX_SAMPLES = None                 # int for a smoke test, None for all

    # ---------------- image ----------------
    IMG_SIZE = 256
    IN_CH = 1

    # ---------------- training ----------------
    BATCH        = 32
    EVAL_BATCH   = 64
    GRAD_ACCUM   = 1
    EPOCHS       = 60
    LR           = 3e-4
    WD           = 1e-5
    N_FOLDS      = 5
    PATIENCE     = 8
    MIN_EPOCHS   = 8
    EMA_DECAY    = 0.999               # None disables the EMA shadow model

    # loss weights (the section-6 terms are appended to CFG at the bottom of
    # that cell, so they stay next to the functions that read them)
    LAM_DEN      = 1.0
    LAM_SEG      = 1.0
    LAM_GRAD     = 0.1                 # edge-preserving term on the denoiser

    # ---------------- runtime ----------------
    # Windows spawns DataLoader workers instead of forking, and each spawned
    # worker re-imports __main__, which does not exist for a notebook kernel, so
    # the dataset class cannot be unpickled and the loader hangs. 0 means "load
    # in the main process", the only reliable setting here.
    NUM_WORKERS   = 0 if os.name == "nt" else _CPUS
    PRECISION     = "bf16" if BF16_OK else ("fp16" if DEVICE.type == "cuda" else "fp32")
    CHANNELS_LAST = True
    COMPILE       = False              # torch.compile; ~60 s warmup per arch
    USE_CACHE     = True               # memmap cache instead of per-file reads
    CACHE_WORKERS = max(1, _CPUS)

    # Kept for the checkpoint/resume machinery. Outside Slurm seconds_left()
    # returns infinity, so this never triggers locally.
    WALLTIME_MARGIN_S = 600

    # ---------------- Noise2Void ----------------
    N2V_MASK_RATIO = 0.02
    N2V_RADIUS     = 5
    N2V_EPOCHS     = 15

    # ---------------- localization ----------------
    PEAK_MIN_DIST = 4
    PEAK_THRESH   = 0.30
    PIXEL_SIZE_A  = 0.20               # placeholder for simulated data

    ARCHS = ["AtomSegNet", "UNetPP", "HRNet"]

    # HRNet keeps full resolution and costs more activation memory than the two
    # U-shaped nets.
    BATCH_OVERRIDE = {"HRNet": 16}


Path(CFG.CACHE_DIR).mkdir(parents=True, exist_ok=True)


def arch_batch(arch_name):
    return CFG.BATCH_OVERRIDE.get(arch_name, CFG.BATCH)


# ---------------- results tree ----------------
def variant_key(use_n2v):
    """True/False, or 'with_n2v'/'with n2v'/'without n2v', all resolve here."""
    if isinstance(use_n2v, str):
        k = use_n2v.strip().lower().replace(" ", "_").replace("-", "_")
        if k in CFG.RESULT_VARIANTS:
            return k
        raise KeyError(f"unknown variant {use_n2v!r}")
    return "with_n2v" if use_n2v else "without_n2v"


def variant_root(use_n2v, create=True):
    p = CFG.RESULTS_ROOT / CFG.RESULT_VARIANTS[variant_key(use_n2v)]
    if create:
        p.mkdir(parents=True, exist_ok=True)
    return p


def results_path(*parts, use_n2v=None, create=True):
    """Generic accessor. results_path('figures', use_n2v=True) ->
    <RESULTS_ROOT>/with n2v/figures ; use_n2v=None stays at the root."""
    p = CFG.RESULTS_ROOT if use_n2v is None else variant_root(use_n2v, create)
    p = p.joinpath(*[str(x) for x in parts])
    if create:
        p.mkdir(parents=True, exist_ok=True)
    return p


def arch_dir(arch, use_n2v, fold=None, create=True):
    """Checkpoints, curves and predictions for one architecture."""
    name = CFG.ARCH_DIRNAME.get(arch, arch)
    p = variant_root(use_n2v, create) / name
    if fold is not None:
        p = p / CFG.FOLD_FMT.format(int(fold))
    if create:
        p.mkdir(parents=True, exist_ok=True)
    return p


def json_dir(use_n2v, create=True):
    """Metrics, configs and run manifests."""
    return results_path("JSON", use_n2v=use_n2v, create=create)


def n2v_dir(create=True):
    """Noise2Void weights and denoised outputs. Only exists on the with-n2v
    side, so it takes no variant argument."""
    return results_path("N2V", use_n2v=True, create=create)


def register_result_folder(variant, name):
    """Declare a folder mid-session without editing CFG."""
    k = variant_key(variant)
    if name not in CFG.RESULT_LEAVES[k]:
        CFG.RESULT_LEAVES[k].append(name)
    return results_path(name, use_n2v=k)


def ensure_results_tree(verbose=True):
    made = []
    CFG.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    for k in CFG.RESULT_VARIANTS:
        variant_root(k)
        for leaf in CFG.RESULT_LEAVES[k]:
            made.append(results_path(leaf, use_n2v=k))
    if verbose:
        for k, folder in CFG.RESULT_VARIANTS.items():
            print(f"  {folder:<12} -> {', '.join(CFG.RESULT_LEAVES[k])}")
    return made


# ---------------- walltime awareness (no-op off Slurm) ----------------
def seconds_left():
    """Seconds until Slurm kills this job, or inf outside Slurm."""
    end = os.environ.get("SLURM_JOB_END_TIME")
    if end:
        try:
            return float(end) - time.time()
        except ValueError:
            pass
    try:
        jid = os.environ.get("SLURM_JOB_ID")
        if not jid:
            return float("inf")
        out = subprocess.run(["squeue", "-h", "-j", jid, "-o", "%L"],
                             capture_output=True, text=True, timeout=10).stdout.strip()
        if not out or out in ("UNLIMITED", "INVALID"):
            return float("inf")
        days, _, rest = out.rpartition("-")
        parts = [int(x) for x in rest.split(":")]
        while len(parts) < 3:
            parts.insert(0, 0)
        s = parts[0] * 3600 + parts[1] * 60 + parts[2]
        return float(s + int(days) * 86400 if days else s)
    except Exception:
        return float("inf")


def time_is_short(margin=None):
    return seconds_left() < (CFG.WALLTIME_MARGIN_S if margin is None else margin)


# ---------------- graceful SIGTERM ----------------
# Jupyter's "interrupt" raises KeyboardInterrupt rather than SIGTERM, and the
# resume checkpoints handle that path anyway.
STOP_REQUESTED = {"flag": False}


def _on_sigterm(signum, frame):
    STOP_REQUESTED["flag"] = True
    print("\n[signal] SIGTERM received: will checkpoint and exit at the next "
          "epoch boundary.", flush=True)


try:
    signal.signal(signal.SIGTERM, _on_sigterm)
except (ValueError, OSError):
    pass


print("Dataset   :", CFG.DATA_ROOT)
print("Outputs   :", CFG.OUT_DIR)
print("Cache     :", CFG.CACHE_DIR)
print("Results   :", CFG.RESULTS_ROOT)
ensure_results_tree()
print("Workers   :", CFG.NUM_WORKERS)
print("Precision :", CFG.PRECISION)
print("Batch     :", CFG.BATCH, "| overrides:", CFG.BATCH_OVERRIDE, "| LR", CFG.LR)
_left = seconds_left()
print("Walltime  :", "unlimited" if _left == float("inf") else f"{_left/3600:.2f} h left")

## 2. Dataset presence check

Verifies that the dataset is where the config says it is and that no extra
directory level was introduced by `scp -r`.

In [ ]:
# ============================================================
# 2. Dataset presence check
# ============================================================
EXPECTED_SUBDIRS = [CFG.SUBDIR_NOISY, CFG.SUBDIR_CLEAN,
                    CFG.SUBDIR_MASK, CFG.SUBDIR_GAUSS]
IMG_EXTS = ('.tif', '.tiff', '.png', '.jpg', '.jpeg', '.bmp')


def _is_populated(p: Path) -> bool:
    return p.is_dir() and any((p / e).is_dir() for e in EXPECTED_SUBDIRS)


def resolve_data_root(root):
    root = Path(root).expanduser().resolve()
    if not root.exists():
        raise RuntimeError(f"Dataset folder does not exist:\n  {root}")
    if _is_populated(root):
        return root
    cur = root
    for _ in range(3):
        subdirs = [d for d in cur.iterdir() if d.is_dir()]
        hits = [d for d in subdirs if _is_populated(d)]
        if len(hits) == 1:
            print(f"note: dataset was nested one level deeper -> {hits[0]}")
            return hits[0]
        if len(subdirs) != 1:
            break
        cur = subdirs[0]
    raise RuntimeError(
        f"None of {EXPECTED_SUBDIRS} were found under:\n  {root}\n"
        f"Contents: {sorted(p.name for p in root.iterdir())[:20]}")


CFG.DATA_ROOT = str(resolve_data_root(CFG.DATA_ROOT))
print("Dataset root:", CFG.DATA_ROOT)

print("\nsubfolder                              images")
print("-" * 48)
_counts = {}
for p in sorted(Path(CFG.DATA_ROOT).iterdir()):
    if p.is_dir():
        n = sum(1 for f in p.iterdir()
                if f.is_file() and f.suffix.lower() in IMG_EXTS)
        _counts[p.name] = n
        print(f"{p.name:38s} {n:6d}" + ("" if n else "   (non-image folder)"))

# Only the four modalities in use need to agree. Folders holding .txt/.csv/.npy
# (params, position, radius, misc) legitimately show 0 images.
_needed = {k: _counts.get(k, 0) for k in EXPECTED_SUBDIRS}
print("\nmodalities in use:", _needed)
if len(set(_needed.values())) > 1 or 0 in _needed.values():
    print("\nWARNING: the four modalities disagree on file count. Pairing below "
          "will drop any sample missing a modality.")
else:
    print(f"all four modalities present and consistent "
          f"({next(iter(_needed.values()))} files each)")

## 3. Dataset discovery and pairing

TEM-ImageNet-v1.3 ships paired files sharing a basename across `image/`,
`noNoise/`, `circularMask/`, `gaussianMask/` and `position/`. The subfolders are
auto-detected and the basenames intersected so every retained sample has all
modalities.

In [ ]:
# ============================================================
# 3. Dataset discovery and pairing
# ============================================================
import json, hashlib

try:
    from PIL import Image as _PILImage
except ImportError:
    _PILImage = None


def results_file(*parts, use_n2v=None):
    """Path to a FILE in the results tree. results_path() mkdirs whatever it is
    given, so never hand it a filename directly."""
    *dirs, name = parts
    return results_path(*dirs, use_n2v=use_n2v) / str(name)


def discover_dataset(root):
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise RuntimeError(f"Not a directory: {root}")
    candidates = {
        'noisy':  [CFG.SUBDIR_NOISY, 'image', 'images', 'noisy', 'input'],
        'clean':  [CFG.SUBDIR_CLEAN, 'noNoise', 'noiseFree', 'clean', 'gt',
                   'ground_truth'],
        'mask':   [CFG.SUBDIR_MASK, 'circularMask', 'mask', 'masks', 'segmentation'],
        'gauss':  [CFG.SUBDIR_GAUSS, 'gaussianMask', 'gaussian', 'heatmap'],
        'coords': [CFG.SUBDIR_COORDS, 'position', 'coords', 'coordinates', 'labels'],
    }
    available = {p.name.lower(): p for p in root.iterdir() if p.is_dir()}
    found = {}
    for k, names in candidates.items():
        for n in names:
            if not n:
                continue
            p = root / str(n).strip()
            if p.is_dir():
                found[k] = p
                break
            p = available.get(Path(str(n)).name.lower())
            if p is not None:
                found[k] = p
                break

    # Two roles pointing at one folder means a fallback name matched something
    # it should not have. Training on identity pairs is silent and ruinous.
    seen = {}
    for k, p in found.items():
        r = p.resolve()
        if r in seen:
            raise RuntimeError(
                f"'{k}' and '{seen[r]}' both resolved to {r}. Set the CFG.SUBDIR_* "
                f"names explicitly instead of relying on the fallback list.")
        seen[r] = k

    print("Discovered subfolders:")
    for k in ('noisy', 'clean', 'mask', 'gauss', 'coords'):
        print(f"  {k:7s} -> {found.get(k, '(missing)')}")
    if 'noisy' not in found:
        raise RuntimeError(f"No noisy-image folder under {root}")
    if 'gauss' not in found:
        print("\nWARNING: gaussianMask/ is missing. Section 10 will fall back to "
              "circularMask centroids for localization ground truth, which is "
              "coarser than the sub-pixel Gaussian peaks.")
    return found


DIRS = discover_dataset(CFG.DATA_ROOT)
CFG.DIRS = {k: str(v) for k, v in DIRS.items()}     # sections 4+ read this


def list_basenames(d, exts=IMG_EXTS, label=""):
    """Stem -> path. rglob is recursive, so report stem collisions rather than
    dropping the duplicates in silence."""
    d = Path(d)
    if not d.is_dir():
        return {}
    exts = {e.lower() for e in exts}
    files = sorted((f for f in d.rglob('*')
                    if f.is_file() and f.suffix.lower() in exts),
                   key=lambda f: (f.stem.lower(), str(f).lower()))
    file_map, dupes = {}, []
    for f in files:
        if f.stem in file_map:
            dupes.append((f.stem, file_map[f.stem], f))
        else:
            file_map[f.stem] = f
    if dupes:
        print(f"  WARNING [{label or d.name}]: {len(dupes)} duplicate stem(s) "
              f"under nested folders; keeping the first of each. Example: "
              f"{dupes[0][0]}  ({dupes[0][1].parent.name} vs {dupes[0][2].parent.name})")
    return file_map


noisy_map  = list_basenames(DIRS['noisy'], label='noisy')
clean_map  = list_basenames(DIRS['clean'], label='clean') if 'clean' in DIRS else {}
mask_map   = list_basenames(DIRS['mask'],  label='mask')  if 'mask'  in DIRS else {}
gauss_map  = list_basenames(DIRS['gauss'], label='gauss') if 'gauss' in DIRS else {}
coords_map = (list_basenames(DIRS['coords'], exts=('.txt', '.csv', '.npy'),
                             label='coords') if 'coords' in DIRS else {})

common = set(noisy_map)
if clean_map:
    common &= set(clean_map)
if mask_map:
    common &= set(mask_map)
common = sorted(common)

if CFG.MAX_SAMPLES is not None and len(common) > CFG.MAX_SAMPLES:
    rng = np.random.RandomState(SEED)
    idx = sorted(rng.choice(len(common), CFG.MAX_SAMPLES, replace=False))
    common = [common[i] for i in idx]
    print(f"\nCapped to {CFG.MAX_SAMPLES} samples (CFG.MAX_SAMPLES).")

n_gauss = sum(1 for b in common if b in gauss_map)

print(f"\nPaired samples : {len(common)}")
print(f"  noisy        : {len(noisy_map)}")
print(f"  clean GT     : {len(clean_map)}")
print(f"  mask GT      : {len(mask_map)}")
print(f"  gaussian GT  : {len(gauss_map)}  ({n_gauss} of the paired set)")

if not common:
    raise RuntimeError(
        "Zero paired samples. The basenames across subfolders do not intersect. "
        "Print a few from each map and compare the stems.")

# Partial gaussian coverage fails in section 10, not here, so say so now.
if gauss_map and n_gauss < len(common):
    missing = len(common) - n_gauss
    print(f"\nWARNING: {missing} paired sample(s) have no gaussianMask. "
          f"Localization will skip them; set CFG.LOCALIZATION_GT = 'circularMask' "
          f"to use every frame instead.")

# ---------------- integrity spot-check ----------------
def _probe(path):
    """Shape and dtype of one file, without committing to a loader."""
    try:
        if path.suffix.lower() == '.npy':
            a = np.load(path, mmap_mode='r')
        elif _PILImage is not None:
            a = np.asarray(_PILImage.open(path))
        else:
            import cv2
            a = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        return (None if a is None else (tuple(a.shape), str(a.dtype)))
    except Exception as e:
        return f"unreadable ({e.__class__.__name__})"


_probe_n = min(5, len(common))
_probe_rng = np.random.RandomState(SEED)
_probe_stems = [common[i] for i in
                sorted(_probe_rng.choice(len(common), _probe_n, replace=False))]
print("\nIntegrity spot-check:")
_bad = []
for stem in _probe_stems:
    row = []
    for name, m in (('noisy', noisy_map), ('clean', clean_map),
                    ('mask', mask_map), ('gauss', gauss_map)):
        if stem in m:
            info = _probe(m[stem])
            row.append(f"{name}={info}")
            if isinstance(info, str) or (info and len(info[0]) > 2):
                _bad.append((stem, name, info))
    print(f"  {stem}: " + "  ".join(row))
if _bad:
    print(f"  WARNING: {len(_bad)} file(s) are unreadable or not single-channel. "
          f"Section 4 must squeeze or grayscale them before caching.")

# ---------------- manifest into the results tree ----------------
# Ties every downstream fold split to this exact dataset snapshot.
_stem_hash = hashlib.sha1("\n".join(common).encode()).hexdigest()[:12]
_manifest = {
    "data_root":        str(Path(CFG.DATA_ROOT).resolve()),
    "subfolders":       CFG.DIRS,
    "n_paired":         len(common),
    "n_noisy":          len(noisy_map),
    "n_clean":          len(clean_map),
    "n_mask":           len(mask_map),
    "n_gauss":          len(gauss_map),
    "n_gauss_paired":   n_gauss,
    "n_coords":         len(coords_map),
    "localization_gt":  CFG.LOCALIZATION_GT,
    "max_samples":      CFG.MAX_SAMPLES,
    "seed":             SEED,
    "img_size":         CFG.IMG_SIZE,
    "n_folds":          CFG.N_FOLDS,
    "basename_sha1":    _stem_hash,
}
with open(results_file("dataset_manifest.json"), "w") as f:
    json.dump(_manifest, f, indent=2)
with open(results_file("dataset_basenames.txt"), "w") as f:
    f.write("\n".join(common))

CFG.DATASET_HASH = _stem_hash
print(f"\nManifest  : {results_file('dataset_manifest.json')}")
print(f"Snapshot  : sha1 {_stem_hash} over {len(common)} basenames")

In [ ]:
# ============================================================
# 3b. Image IO + visual sanity check
# ============================================================
def imread_gray(p):
    p = Path(p)
    if not p.is_file():
        raise FileNotFoundError(f"Image file not found: {p}")
    img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if img is None:
        img = np.array(Image.open(p))
    img = np.asarray(img)
    if img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY)
        elif img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            img = img[..., 0]
    if img.ndim != 2:
        raise ValueError(f"Expected 2D grayscale, got {img.shape} for {p}")
    return img.astype(np.float32)


def norm01(x, name="", warn=False):
    """Percentile-stretch to [0,1]. Returns zeros on a flat image; set warn=True
    while debugging so a blank panel is not silent."""
    x = np.asarray(x, dtype=np.float32)
    finite = np.isfinite(x)
    if not finite.any():
        if warn:
            print(f"WARN: all-nonfinite image {name}")
        return np.zeros_like(x, dtype=np.float32)
    lo, hi = np.percentile(x[finite], (1, 99))
    if hi - lo < 1e-6:
        if warn:
            print(f"WARN: flat dynamic range for {name} (lo={lo}, hi={hi})")
        return np.zeros_like(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=lo, posinf=hi, neginf=lo)
    return np.clip((x - lo) / (hi - lo), 0.0, 1.0)


def _panel(ax, arr, title, cmap='gray'):
    if arr is None:
        ax.text(0.5, 0.5, 'not available', ha='center', va='center')
    else:
        ax.imshow(arr, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')


sample = common[0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
_panel(axes[0], norm01(imread_gray(noisy_map[sample]), 'noisy', warn=True),
       'noisy (input)')
_panel(axes[1], norm01(imread_gray(clean_map[sample]), 'clean', warn=True)
       if sample in clean_map else None, 'noNoise (denoise GT)')
_panel(axes[2], (imread_gray(mask_map[sample]) > 0).astype(np.float32)
       if sample in mask_map else None, 'circularMask (seg GT)')
_panel(axes[3], norm01(imread_gray(gauss_map[sample]), 'gauss', warn=True)
       if sample in gauss_map else None,
       'gaussianMask (localization GT)', cmap='inferno')
fig.suptitle(f"sample: {sample}", fontsize=12)
plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / 'sanity_check.png', dpi=140, bbox_inches='tight')
plt.show()

shapes = {k: imread_gray(m[sample]).shape
          for k, m in (('noisy', noisy_map), ('clean', clean_map),
                       ('mask', mask_map), ('gauss', gauss_map))
          if sample in m}
print("shapes:", shapes)
if len(set(shapes.values())) > 1:
    print("WARNING: modalities have different pixel dimensions; the resize step "
          "will realign them but check that they correspond.")

## 3c. Memmap cache

Reading thousands of small PNGs per epoch makes `__getitem__` a separate file
open every time and the GPU idles waiting on the disk. This packs the dataset
into four contiguous uint8 arrays, already resized. The build runs once; after
that a sample read is a memcpy from page cache.

In [ ]:
# ============================================================
# 3c. Build a uint8 memmap cache  [main I/O optimisation]
# ============================================================
CACHE_KEYS = ('noisy', 'clean', 'mask', 'gauss')


def _cache_paths():
    tag = f"{CFG.IMG_SIZE}_{len(common)}"
    d = Path(CFG.CACHE_DIR)
    return {k: d / f"{k}_{tag}.npy" for k in CACHE_KEYS}, d / f"meta_{tag}.json"


def _resize(img, size, interp):
    if img.shape[:2] != (size, size):
        img = cv2.resize(img, (size, size), interpolation=interp)
    return img


def _encode_one(args):
    """Runs in a worker process. Returns uint8 rows for one sample."""
    idx, b, np_, cp_, mp_, gp_, size = args
    out = {}

    n = _resize(norm01(imread_gray(np_)), size, cv2.INTER_AREA)
    out['noisy'] = np.clip(n * 255.0 + 0.5, 0, 255).astype(np.uint8)

    c = (_resize(norm01(imread_gray(cp_)), size, cv2.INTER_AREA)
         if cp_ is not None else n)
    out['clean'] = np.clip(c * 255.0 + 0.5, 0, 255).astype(np.uint8)

    if mp_ is not None:
        m = imread_gray(mp_)
        mx = np.nanmax(m) if np.isfinite(m).any() else 0.0
        m = (m > 0.5 * mx).astype(np.uint8) if mx > 0 else np.zeros_like(m, np.uint8)
        m = _resize(m, size, cv2.INTER_NEAREST)
    else:
        m = np.zeros((size, size), np.uint8)
    out['mask'] = (m > 0).astype(np.uint8)

    if gp_ is not None:
        g = _resize(norm01(imread_gray(gp_)), size, cv2.INTER_AREA)
        out['gauss'] = np.clip(g * 255.0 + 0.5, 0, 255).astype(np.uint8)
    else:
        out['gauss'] = np.zeros((size, size), np.uint8)
    return idx, out


def build_cache(force=False):
    paths, meta_path = _cache_paths()
    N, S = len(common), CFG.IMG_SIZE

    def _ok(p):                       # .npy carries a header; check the shape
        if not p.is_file():
            return False
        try:
            a = np.load(p, mmap_mode='r')
            return a.shape == (N, S, S) and a.dtype == np.uint8
        except Exception:
            return False

    if not force and meta_path.is_file() and all(_ok(p) for p in paths.values()):
        meta = json.loads(meta_path.read_text())
        if meta.get('basenames') == common:
            print(f"cache hit: {N} samples at {S}px in {CFG.CACHE_DIR}")
            return paths, meta
        print("cache exists but the sample list changed; rebuilding")

    print(f"building cache: {N} samples -> "
          f"{4 * N * S * S / 1024**3:.2f} GiB in {CFG.CACHE_DIR}")

    tmp_paths = {k: p.with_suffix('.npy.tmp') for k, p in paths.items()}
    mms = {k: np.lib.format.open_memmap(tmp_paths[k], mode='w+',
                                        dtype=np.uint8, shape=(N, S, S))
           for k in CACHE_KEYS}
    jobs = [(i, b, noisy_map[b], clean_map.get(b), mask_map.get(b),
             gauss_map.get(b), S) for i, b in enumerate(common)]

    t0 = time.time()
    nproc = max(1, min(CFG.CACHE_WORKERS, 16))
    if nproc > 1:
        # 'fork' does not exist on Windows, and 'spawn' cannot pickle
        # _encode_one because it is defined in a notebook cell. A thread pool
        # still parallelises: cv2 and numpy release the GIL during decode.
        try:
            _pool = mp.get_context('fork').Pool(nproc)
        except (ValueError, AttributeError):
            from multiprocessing.pool import ThreadPool
            _pool = ThreadPool(nproc)
        with _pool as pool:
            for idx, out in tqdm(pool.imap_unordered(_encode_one, jobs, chunksize=32),
                                 total=N, desc='cache'):
                for k in CACHE_KEYS:
                    mms[k][idx] = out[k]
    else:
        for job in tqdm(jobs, desc='cache'):
            idx, out = _encode_one(job)
            for k in CACHE_KEYS:
                mms[k][idx] = out[k]

    for k in list(mms):
        mms[k].flush()
        del mms[k]
    gc.collect()          # Windows will not rename a file that is still mapped
    for k, p in paths.items():
        os.replace(tmp_paths[k], p)

    meta = dict(n=N, size=S, basenames=common,
                built=time.strftime('%Y-%m-%d %H:%M:%S'),
                host=socket.gethostname(),
                has_gauss=[b in gauss_map for b in common])
    meta_path.write_text(json.dumps(meta))
    print(f"cache built in {time.time() - t0:.1f} s")
    return paths, meta


CACHE, CACHE_INDEX = None, {}
if CFG.USE_CACHE and common:
    try:
        _paths, _meta = build_cache()
        # Memory maps are shared through the page cache, so DataLoader workers
        # do not each hold a private copy. If you rerun build_cache(force=True)
        # in the same kernel on Windows, restart the kernel first: these open
        # maps block the rename.
        CACHE = {k: np.load(p, mmap_mode='r') for k, p in _paths.items()}
        for _k, _a in CACHE.items():
            assert _a.shape == (len(common), CFG.IMG_SIZE, CFG.IMG_SIZE), \
                f"cache {_k} has shape {_a.shape}"
        assert set(np.unique(np.asarray(CACHE['mask'][0]))) <= {0, 1}, \
            "cache mask is not binary - the cache is corrupt, rebuild with force=True"
        CACHE_INDEX = {b: i for i, b in enumerate(common)}
        print(f"cache ready | "
              f"{shutil.disk_usage(CFG.CACHE_DIR).free / 1024**3:.1f} GiB free")
    except Exception as e:
        print(f"cache build failed ({e}); falling back to per-file reads")
        CACHE, CACHE_INDEX = None, {}

## 4. Datasets

`TEMSegDataset` returns `(noisy, clean, mask, basename)` for the supervised
denoising and segmentation task. `N2VDataset` masks about 2% of pixels and asks
the network to predict them from their neighbourhood, so it needs no clean
target.

In [ ]:
# ============================================================
# 4. Datasets  (read from the memmap cache when one is available)
# ============================================================
_WARNED = set()


def _warn_once(key, msg):
    if key not in _WARNED:
        _WARNED.add(key)
        print(f"WARNING: {msg}")


def _cache_ref():
    """Look the cache up at call time. Binding it in __init__ freezes a dataset
    to per-file reads if it was constructed before section 5 ran."""
    return globals().get('CACHE', None)


def _cache_index():
    return globals().get('CACHE_INDEX', {})


def center_or_resize(img, size, interpolation=cv2.INTER_AREA):
    img = np.asarray(img)
    if img.shape[:2] != (size, size):
        img = cv2.resize(img, (size, size), interpolation=interpolation)
    return np.ascontiguousarray(img)


def binarize(a):
    """0/1 float mask from either a 0/1 or a 0/255 source. The cache is uint8,
    so the scale depends on how section 5 wrote it; decide per array instead of
    assuming."""
    a = np.asarray(a, np.float32)
    hi = float(a.max()) if a.size else 0.0
    return (a > (127.5 if hi > 1.0 else 0.5)).astype(np.float32)


def random_aug(*arrays, rng=None):
    """Dihedral group D4 applied identically to every modality.

    Uses the `random` module by default so DataLoader workers inherit their seed
    from worker_init_fn. Pass a numpy Generator (not a legacy RandomState) if you
    need an independent stream."""
    if rng is None:
        rng = random
        flip_lr, flip_ud = rng.random() < 0.5, rng.random() < 0.5
        k = rng.randint(0, 3)                       # inclusive
    else:
        flip_lr, flip_ud = rng.random() < 0.5, rng.random() < 0.5
        k = int(rng.integers(0, 4))                 # exclusive
    if flip_lr:
        arrays = [a[:, ::-1] for a in arrays]
    if flip_ud:
        arrays = [a[::-1] for a in arrays]
    if k:
        arrays = [np.rot90(a, k) for a in arrays]
    return [np.ascontiguousarray(a) for a in arrays]


class TEMSegDataset(Dataset):
    def __init__(self, basenames, noisy_map, clean_map, mask_map,
                 size=None, train=True, gauss_map=None, return_gauss=False):
        self.basenames = list(basenames)
        self.noisy_map = noisy_map
        self.clean_map = clean_map
        self.mask_map = mask_map
        self.gauss_map = gauss_map if gauss_map is not None else {}
        self.size = int(size or CFG.IMG_SIZE)
        self.train = train
        self.return_gauss = return_gauss
        if self.size <= 0:
            raise ValueError("size must be > 0")

    @property
    def cache(self):
        c = _cache_ref()
        return c if (c is not None and self.size == CFG.IMG_SIZE) else None

    def __len__(self):
        return len(self.basenames)

    def _from_cache(self, b):
        i = _cache_index()[b]
        c = self.cache
        return (np.asarray(c['noisy'][i], np.float32) / 255.0,
                np.asarray(c['clean'][i], np.float32) / 255.0,
                binarize(c['mask'][i]),
                np.asarray(c['gauss'][i], np.float32) / 255.0)

    def _from_disk(self, b):
        n = center_or_resize(norm01(imread_gray(self.noisy_map[b])), self.size)
        if b in self.clean_map:
            c = center_or_resize(norm01(imread_gray(self.clean_map[b])), self.size)
        else:
            # Identity pairs make the denoising loss collapse to zero and look
            # like fast convergence. Say so rather than training on them mutely.
            _warn_once("clean_fallback",
                       "at least one sample has no clean GT; the denoiser target "
                       "falls back to the noisy input (identity pair). Check the "
                       "'clean' row of the section-3 discovery output.")
            c = n.copy()
        if b in self.mask_map:
            m = binarize(imread_gray(self.mask_map[b]))
            m = (center_or_resize(m, self.size, cv2.INTER_NEAREST) > 0.5).astype(np.float32)
        else:
            m = np.zeros((self.size, self.size), np.float32)
        if b in self.gauss_map:
            g = imread_gray(self.gauss_map[b])
            resized = g.shape[:2] != (self.size, self.size)
            g = center_or_resize(norm01(g), self.size)
            if resized:
                # Area-averaging flattens the Gaussian peaks that section 10
                # thresholds against; restore the peak scale.
                gmax = float(g.max())
                if gmax > 0:
                    g = g / gmax
        else:
            g = np.zeros((self.size, self.size), np.float32)
        return n, c, m, g

    def __getitem__(self, idx):
        b = self.basenames[idx]
        n, c, m, g = (self._from_cache(b) if self.cache is not None
                      else self._from_disk(b))
        if self.train:
            n, c, m, g = random_aug(n, c, m, g)

        def t(a):
            return torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32))[None]

        if self.return_gauss:
            return t(n), t(c), t(m), t(g), b
        return t(n), t(c), t(m), b


class N2VDataset(Dataset):
    """Noise2Void. ~ratio of pixels are replaced by a random neighbour value;
    the loss is MSE at those positions against the original noisy value."""

    def __init__(self, basenames, noisy_map, size=None, ratio=0.02,
                 radius=5, train=True):
        self.basenames = list(basenames)
        self.noisy_map = noisy_map
        self.size = int(size or CFG.IMG_SIZE)
        self.ratio = float(ratio)
        self.radius = int(radius)
        self.train = train
        if not 0 < self.ratio <= 1:
            raise ValueError("ratio must be in (0, 1]")
        if self.radius < 1:
            raise ValueError("radius must be >= 1")

    @property
    def cache(self):
        c = _cache_ref()
        return c if (c is not None and self.size == CFG.IMG_SIZE) else None

    def __len__(self):
        return len(self.basenames)

    def _make_n2v(self, x):
        H, W = x.shape
        if H < 2 or W < 2:
            raise ValueError("N2V needs at least a 2x2 image")
        n_mask = max(1, min(int(round(H * W * self.ratio)), H * W))
        flat = np.random.choice(H * W, size=n_mask, replace=False)
        ys, xs = np.divmod(flat, W)

        # Draw a non-zero offset directly instead of rejection-sampling in a
        # while loop, which could spin for many iterations at radius 1.
        span = 2 * self.radius + 1
        k = np.random.randint(0, span * span - 1, size=n_mask)
        k = k + (k >= (span * span // 2))          # skip the centre offset
        dy, dx = k // span - self.radius, k % span - self.radius

        # Reflect at the borders. Clipping can land the neighbour back on the
        # masked pixel itself, which hands the network the value it is scored
        # on and destroys the blind spot along every edge.
        ny, nx = ys + dy, xs + dx
        ny = np.abs(ny)
        nx = np.abs(nx)
        ny = np.where(ny > H - 1, 2 * (H - 1) - ny, ny)
        nx = np.where(nx > W - 1, 2 * (W - 1) - nx, nx)

        # Reflection about index 0 can still fold a pixel onto itself
        # (ys=1, dy=-2 -> ny=1). Nudge those one step sideways.
        same = (ny == ys) & (nx == xs)
        if same.any():
            nx = np.where(same, np.where(xs > 0, xs - 1, xs + 1), nx)

        x_in = x.copy()
        x_in[ys, xs] = x[ny, nx]
        m = np.zeros_like(x, dtype=np.float32)
        m[ys, xs] = 1.0
        return np.ascontiguousarray(x_in), np.ascontiguousarray(m)

    def __getitem__(self, idx):
        b = self.basenames[idx]
        c = self.cache
        if c is not None:
            x = np.asarray(c['noisy'][_cache_index()[b]], np.float32) / 255.0
        else:
            x = center_or_resize(norm01(imread_gray(self.noisy_map[b])), self.size)
        if self.train:
            (x,) = random_aug(x)
        x = np.ascontiguousarray(x, dtype=np.float32)
        x_in, m = self._make_n2v(x)
        return (torch.from_numpy(x_in)[None],
                torch.from_numpy(x)[None],
                torch.from_numpy(m)[None])


def loader_options(shuffle, drop_last=False, batch=None, eval_mode=False):
    opts = dict(
        batch_size=int(batch or (CFG.EVAL_BATCH if eval_mode else CFG.BATCH)),
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda'),
        drop_last=drop_last,
        worker_init_fn=worker_init_fn,
    )
    if CFG.NUM_WORKERS > 0:
        opts['persistent_workers'] = True
        opts['prefetch_factor'] = 4
    return opts


# ---------------- smoke test + steady-state throughput probe ----------------
_ds = TEMSegDataset(common[:min(2048, len(common))], noisy_map, clean_map,
                    mask_map, gauss_map=gauss_map, train=True,
                    return_gauss=True)
_n, _c, _m, _g, _b = _ds[0]
print("item shapes:", tuple(_n.shape), tuple(_c.shape), tuple(_m.shape))
print("noisy range:", round(_n.min().item(), 3), round(_n.max().item(), 3),
      "| mask positive fraction:", round(_m.mean().item(), 4))
print("gauss range:", round(_g.min().item(), 3), round(_g.max().item(), 3))

# Cheap contract checks. Each of these has a silent failure mode above.
assert float(_m.max()) <= 1.0, "mask is not 0/1 after loading (cache scale?)"
assert 0.0 <= float(_n.min()) and float(_n.max()) <= 1.0, "noisy is not in [0,1]"
_ident = float((_n - _c).abs().mean())
if _ident < 1e-6:
    print("WARNING: noisy and clean are identical for this sample. The denoiser "
          "has nothing to learn if that holds across the set.")

_probe_n2v = N2VDataset(common[:64], noisy_map, ratio=CFG.N2V_MASK_RATIO,
                        radius=CFG.N2V_RADIUS, train=False)
_xi, _x0, _mm = _probe_n2v[0]
_leak = ((_xi == _x0).float() * _mm).sum().item() / max(_mm.sum().item(), 1)
print(f"N2V blind-spot check: {_leak*100:.2f}% of masked pixels unchanged "
      f"(nonzero is expected by chance in flat regions, ~0 near borders is the point)")

_dl = DataLoader(_ds, **loader_options(shuffle=True, drop_last=True, batch=CFG.BATCH))
_it = iter(_dl)
try:                                   # warm up workers / page cache
    for _ in range(min(4, len(_dl))):
        next(_it)
except StopIteration:
    pass
del _it

_t0, _seen = time.time(), 0
for _batch in _dl:
    _seen += _batch[0].shape[0]
_thru = _seen / max(time.time() - _t0, 1e-9)
_src = 'memmap cache' if _cache_ref() is not None else 'per-file reads'
print(f"steady loader throughput: {_thru:.0f} img/s ({_src})")

with open(results_file("loader_probe.json"), "w") as f:
    json.dump({"dataset_hash": getattr(CFG, "DATASET_HASH", None),
               "throughput_img_s": round(_thru, 1),
               "source": _src,
               "batch": CFG.BATCH,
               "num_workers": CFG.NUM_WORKERS,
               "precision": CFG.PRECISION,
               "n2v_masked_unchanged_frac": round(_leak, 5)}, f, indent=2)
del _dl, _ds, _probe_n2v

## 5. Architectures

Three encoder-decoder backbones, 1 channel in, two heads out: a denoised image
and atom-mask logits.

AtomSegNet is a 5-level U-Net with residual blocks and attention gates on the
skips. UNet++ adds nested skip pathways with deep supervision from four separate
heads. HRNet keeps three resolutions alive and exchanges information between
them at every stage.

Widths are chosen so all three sit within a few percent of each other on
parameter count, so an accuracy difference is attributable to the architecture.
The denoiser predicts an additive residual in image space, so every net starts
at the identity map.

In [ ]:
# ============================================================
# 5. Architectures
# ============================================================
def norm_layer(c, groups=8):
    """GroupNorm rather than BatchNorm. Batch statistics over 8-32 highly
    self-similar crops are noisy, and GroupNorm makes the reported metrics
    independent of batch size, which matters because HRNet runs at a smaller
    batch than the other two. GroupNorm requires num_channels % num_groups == 0,
    and the nested UNet++ concatenations produce counts like 90 and 150, so fall
    back to the largest valid divisor."""
    g = min(groups, c)
    while g > 1 and c % g:
        g -= 1
    return nn.GroupNorm(num_groups=g, num_channels=c)


def init_weights(module):
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.GroupNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)


class ResBlock(nn.Module):
    """Pre-activation residual block with a zero-initialised second convolution,
    so the block is exactly the identity at initialisation and the effective
    depth grows during training rather than being fixed."""

    def __init__(self, c_in, c_out, drop=0.0, zero_init=True):
        super().__init__()
        self.n1 = norm_layer(c_in)
        self.c1 = nn.Conv2d(c_in, c_out, 3, padding=1, padding_mode='reflect', bias=False)
        self.n2 = norm_layer(c_out)
        self.c2 = nn.Conv2d(c_out, c_out, 3, padding=1, padding_mode='reflect', bias=False)
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.skip = nn.Identity() if c_in == c_out else nn.Conv2d(c_in, c_out, 1, bias=False)
        if zero_init:
            nn.init.zeros_(self.c2.weight)

    def forward(self, x):
        h = self.c1(self.act(self.n1(x)))
        h = self.drop(h)
        h = self.c2(self.act(self.n2(h)))
        return h + self.skip(x)


def conv_block(c_in, c_out, drop=0.0):
    return ResBlock(c_in, c_out, drop)


class AttentionGate(nn.Module):
    """Additive attention gate (Oktay et al. 2018). Atom columns cover roughly
    5-15% of a frame, so an ungated skip floods the decoder with vacuum. The psi
    bias starts at +2 so gates are ~88% open at initialisation: a gate that
    starts at 0.5 halves every skip and slows the first epochs."""

    def __init__(self, c_skip, c_gate, c_int=None):
        super().__init__()
        c_int = c_int or max(c_skip // 2, 8)
        self.wg = nn.Conv2d(c_gate, c_int, 1, bias=True)
        self.wx = nn.Conv2d(c_skip, c_int, 1, bias=True)
        self.psi = nn.Conv2d(c_int, 1, 1, bias=True)
        nn.init.zeros_(self.psi.weight)
        nn.init.constant_(self.psi.bias, 2.0)

    def forward(self, skip, gate):
        if gate.shape[-2:] != skip.shape[-2:]:
            gate = F.interpolate(gate, size=skip.shape[-2:], mode='bilinear',
                                 align_corners=False)
        a = torch.sigmoid(self.psi(F.silu(self.wg(gate) + self.wx(skip))))
        # detached so the map can be plotted without keeping the graph alive; a
        # non-leaf tensor stashed on a module also breaks the deepcopy that the
        # EMA evaluation copy relies on
        self.last_gate = a.detach()
        return skip * a


class DenoiseHead(nn.Module):
    """Additive residual in image space. delta is zero at initialisation, so the
    network starts as the identity. During training the output is left
    unclamped: the Charbonnier term against a target in [0,1] keeps it in range,
    and clamping here would zero the gradient on exactly the saturated pixels
    that need correcting. At evaluation the output is clamped."""

    def __init__(self, c_in):
        super().__init__()
        self.conv = nn.Conv2d(c_in, 1, 1)
        nn.init.zeros_(self.conv.weight)
        nn.init.zeros_(self.conv.bias)

    def forward(self, x, feat):
        y = x + self.conv(feat)
        return y if self.training else y.clamp(0.0, 1.0)


def up_block(c_in, c_out):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
        nn.Conv2d(c_in, c_out, 3, padding=1, padding_mode='reflect', bias=False))


# ---------------- AtomSegNet ----------------
class AtomSegNet(nn.Module):
    """Five-level U-Net with residual blocks and attention-gated skips."""

    def __init__(self, in_ch=1, base=32, drop=0.1, attention=True):
        super().__init__()
        self.attention = attention
        c1, c2, c3, c4, c5 = base, base * 2, base * 4, base * 8, base * 16

        self.stem = nn.Conv2d(in_ch, c1, 3, padding=1, padding_mode='reflect', bias=False)
        self.enc1 = conv_block(c1, c1)
        self.enc2 = conv_block(c1, c2)
        self.enc3 = conv_block(c2, c3)
        self.enc4 = conv_block(c3, c4)
        self.bot = conv_block(c4, c5, drop=drop)
        self.pool = nn.MaxPool2d(2)

        self.up4, self.dec4 = up_block(c5, c4), conv_block(c4 * 2, c4)
        self.up3, self.dec3 = up_block(c4, c3), conv_block(c3 * 2, c3)
        self.up2, self.dec2 = up_block(c3, c2), conv_block(c2 * 2, c2)
        self.up1, self.dec1 = up_block(c2, c1), conv_block(c1 * 2, c1)

        if attention:
            self.ag4 = AttentionGate(c4, c5)
            self.ag3 = AttentionGate(c3, c4)
            self.ag2 = AttentionGate(c2, c3)
            self.ag1 = AttentionGate(c1, c2)

        self.out_norm = norm_layer(c1)
        self.head_denoise = DenoiseHead(c1)
        self.head_seg = nn.Conv2d(c1, 1, 1)
        init_weights(self.stem)
        nn.init.constant_(self.head_seg.bias, -2.0)   # sparse-positive prior

    def forward(self, x):
        e1 = self.enc1(self.stem(x))
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bot(self.pool(e4))

        s4 = self.ag4(e4, b) if self.attention else e4
        d4 = self.dec4(torch.cat([self.up4(b), s4], 1))
        s3 = self.ag3(e3, d4) if self.attention else e3
        d3 = self.dec3(torch.cat([self.up3(d4), s3], 1))
        s2 = self.ag2(e2, d3) if self.attention else e2
        d2 = self.dec2(torch.cat([self.up2(d3), s2], 1))
        s1 = self.ag1(e1, d2) if self.attention else e1
        d1 = self.dec1(torch.cat([self.up1(d2), s1], 1))

        f = F.silu(self.out_norm(d1))
        self.aux_logits = None
        return self.head_denoise(x, f), self.head_seg(f)


# ---------------- UNet++ ----------------
class UNetPP(nn.Module):
    """Nested skip pathways with deep supervision from four separate heads. The
    heads are combined with learned softmax weights instead of a plain mean, so
    the network can down-weight the shallow branches once they stop
    contributing; the learned weights are worth reporting."""

    def __init__(self, in_ch=1, base=31, drop=0.1, deep_supervision=True):
        super().__init__()
        nb = [base, base * 2, base * 4, base * 8, base * 16]
        self.deep_supervision = deep_supervision
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

        self.x00 = conv_block(in_ch, nb[0])
        self.x10 = conv_block(nb[0], nb[1])
        self.x20 = conv_block(nb[1], nb[2])
        self.x30 = conv_block(nb[2], nb[3])
        self.x40 = conv_block(nb[3], nb[4], drop=drop)

        self.x01 = conv_block(nb[0] + nb[1], nb[0])
        self.x11 = conv_block(nb[1] + nb[2], nb[1])
        self.x21 = conv_block(nb[2] + nb[3], nb[2])
        self.x31 = conv_block(nb[3] + nb[4], nb[3])

        self.x02 = conv_block(nb[0] * 2 + nb[1], nb[0])
        self.x12 = conv_block(nb[1] * 2 + nb[2], nb[1])
        self.x22 = conv_block(nb[2] * 2 + nb[3], nb[2])

        self.x03 = conv_block(nb[0] * 3 + nb[1], nb[0])
        self.x13 = conv_block(nb[1] * 3 + nb[2], nb[1])

        self.x04 = conv_block(nb[0] * 4 + nb[1], nb[0])

        self.seg_heads = nn.ModuleList([nn.Conv2d(nb[0], 1, 1) for _ in range(4)])
        for h in self.seg_heads:
            nn.init.constant_(h.bias, -2.0)
        self.branch_logits = nn.Parameter(torch.zeros(4))
        self.head_denoise = DenoiseHead(nb[0])

    @property
    def branch_weights(self):
        return torch.softmax(self.branch_logits, 0)

    def forward(self, x):
        x00 = self.x00(x)
        x10 = self.x10(self.pool(x00))
        x20 = self.x20(self.pool(x10))
        x30 = self.x30(self.pool(x20))
        x40 = self.x40(self.pool(x30))

        x01 = self.x01(torch.cat([x00, self.up(x10)], 1))
        x11 = self.x11(torch.cat([x10, self.up(x20)], 1))
        x21 = self.x21(torch.cat([x20, self.up(x30)], 1))
        x31 = self.x31(torch.cat([x30, self.up(x40)], 1))

        x02 = self.x02(torch.cat([x00, x01, self.up(x11)], 1))
        x12 = self.x12(torch.cat([x10, x11, self.up(x21)], 1))
        x22 = self.x22(torch.cat([x20, x21, self.up(x31)], 1))

        x03 = self.x03(torch.cat([x00, x01, x02, self.up(x12)], 1))
        x13 = self.x13(torch.cat([x10, x11, x12, self.up(x22)], 1))

        x04 = self.x04(torch.cat([x00, x01, x02, x03, self.up(x13)], 1))

        logits = [h(t) for h, t in zip(self.seg_heads, [x01, x02, x03, x04])]
        if self.deep_supervision:
            seg = (torch.stack(logits, 0) * self.branch_weights.view(4, 1, 1, 1, 1)).sum(0)
        else:
            seg = logits[-1]

        # exposed for the auxiliary loss; cleared by the training loop
        self.aux_logits = logits[:-1] if (self.training and self.deep_supervision) else None
        return self.head_denoise(x, x04), seg


# ---------------- HRNet ----------------
class FusionUnit(nn.Module):
    """Multi-resolution exchange. Downsampling uses stride-2 convolutions,
    upsampling uses 1x1 + bilinear, following Wang et al. 2020. Interpolating
    downward instead would alias the atomic lattice, which is exactly the signal
    being measured."""

    def __init__(self, chans):
        super().__init__()
        n = len(chans)
        self.n = n
        self.paths = nn.ModuleList()
        for i in range(n):
            row = nn.ModuleList()
            for j in range(n):
                if i == j:
                    row.append(nn.Identity())
                elif j > i:                      # coarse -> fine: 1x1 then up
                    row.append(nn.Sequential(nn.Conv2d(chans[j], chans[i], 1, bias=False),
                                             norm_layer(chans[i])))
                else:                            # fine -> coarse: strided conv
                    ops, c_cur = [], chans[j]
                    for k in range(i - j):
                        c_next = chans[i] if k == i - j - 1 else c_cur
                        ops += [nn.Conv2d(c_cur, c_next, 3, 2, 1, bias=False),
                                norm_layer(c_next)]
                        if k < i - j - 1:
                            ops.append(nn.SiLU(inplace=True))
                        c_cur = c_next
                    row.append(nn.Sequential(*ops))
            self.paths.append(row)

    def forward(self, xs):
        out = []
        for i, xi in enumerate(xs):
            acc = xi
            for j, xj in enumerate(xs):
                if i == j:
                    continue
                y = self.paths[i][j](xj)
                if y.shape[-2:] != xi.shape[-2:]:
                    y = F.interpolate(y, size=xi.shape[-2:], mode='bilinear',
                                      align_corners=False)
                acc = acc + y
            out.append(F.silu(acc))
        return out


class HRNet(nn.Module):
    """Three resolutions held in parallel throughout, with an exchange unit
    after every stage. Full resolution is never lost, which is the property that
    matters for sub-pixel column localisation."""

    def __init__(self, in_ch=1, base=72, stages=3, drop=0.1):
        super().__init__()
        c = [base, base * 2, base * 4]
        self.chans = c
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, c[0], 3, padding=1, padding_mode='reflect', bias=False),
            conv_block(c[0], c[0]))
        self.down1 = nn.Sequential(nn.Conv2d(c[0], c[1], 3, 2, 1, bias=False),
                                   norm_layer(c[1]), nn.SiLU(inplace=True))
        self.down2 = nn.Sequential(nn.Conv2d(c[1], c[2], 3, 2, 1, bias=False),
                                   norm_layer(c[2]), nn.SiLU(inplace=True))

        self.stages = nn.ModuleList()
        self.fusions = nn.ModuleList()
        for _ in range(stages):
            self.stages.append(nn.ModuleList(
                [conv_block(ci, ci, drop=drop if i == 2 else 0.0)
                 for i, ci in enumerate(c)]))
            self.fusions.append(FusionUnit(c))

        self.head_in = conv_block(sum(c), c[0])
        self.out_norm = norm_layer(c[0])
        self.head_denoise = DenoiseHead(c[0])
        self.head_seg = nn.Conv2d(c[0], 1, 1)
        nn.init.constant_(self.head_seg.bias, -2.0)

    def forward(self, x):
        xs = [self.stem(x)]
        xs.append(self.down1(xs[0]))
        xs.append(self.down2(xs[1]))

        for blocks, fuse in zip(self.stages, self.fusions):
            xs = [blk(t) for blk, t in zip(blocks, xs)]
            xs = fuse(xs)

        size = xs[0].shape[-2:]
        cat = torch.cat([xs[0]] + [F.interpolate(t, size=size, mode='bilinear',
                                                 align_corners=False) for t in xs[1:]], 1)
        f = F.silu(self.out_norm(self.head_in(cat)))
        self.aux_logits = None
        return self.head_denoise(x, f), self.head_seg(f)


ARCH_REGISTRY = {"AtomSegNet": AtomSegNet, "UNetPP": UNetPP, "HRNet": HRNet}


# ---------------- reporting ----------------
def count_macs(model, shape=(1, 1, 256, 256)):
    """Multiply-accumulates per forward pass, counted with hooks so no extra
    dependency is needed. Convolutions dominate; norms and activations are
    negligible and are not counted."""
    macs = [0]

    def hook(m, inp, out):
        k = m.kernel_size[0] * m.kernel_size[1]
        macs[0] += out.shape.numel() * k * (m.in_channels // m.groups)

    hooks = [m.register_forward_hook(hook) for m in model.modules()
             if isinstance(m, nn.Conv2d)]
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(*shape))
    model.train(was_training)
    for h in hooks:
        h.remove()
    return macs[0]


def peak_activation_mb(model, shape=(1, 1, 256, 256)):
    """Sum of forward activation tensors for one sample, in MB. This is what
    sets the batch size that fits on the card, and it explains why HRNet needs a
    smaller batch than the two U-shaped nets despite equal parameters."""
    total = [0]
    hooks = [m.register_forward_hook(
        lambda mod, i, o: total.__setitem__(
            0, total[0] + (o.numel() * o.element_size() if torch.is_tensor(o) else 0)))
        for m in model.modules() if isinstance(m, (nn.Conv2d, nn.GroupNorm))]
    was = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(*shape))
    model.train(was)
    for h in hooks:
        h.remove()
    return total[0] / 1024 ** 2


def architecture_table(shape=(1, 1, 256, 256)):
    rows = []
    for name, cls in ARCH_REGISTRY.items():
        m = cls()
        rows.append(dict(arch=name,
                         params_M=sum(p.numel() for p in m.parameters()) / 1e6,
                         trainable_M=sum(p.numel() for p in m.parameters()
                                         if p.requires_grad) / 1e6,
                         GMACs=count_macs(m, shape) / 1e9,
                         act_MB=peak_activation_mb(m, shape),
                         convs=sum(1 for x in m.modules() if isinstance(x, nn.Conv2d))))
        del m
    return pd.DataFrame(rows)


def _probe(cls):
    """Contract check: shapes, identity initialisation of the denoiser, output
    range at eval, and a finite gradient through every parameter."""
    torch.manual_seed(0)
    m = cls()
    x = torch.rand(2, 1, 64, 64)

    m.eval()
    with torch.no_grad():
        d, s = m(x)
    assert d.shape == s.shape == x.shape, (cls.__name__, d.shape, s.shape)
    assert 0.0 <= d.min().item() and d.max().item() <= 1.0, "eval range"
    assert torch.allclose(d, x, atol=1e-5), \
        f"{cls.__name__} is not the identity at initialisation"

    m.train()
    d, s = m(x)
    loss = d.mean() + s.mean()
    if getattr(m, 'aux_logits', None):
        loss = loss + sum(a.mean() for a in m.aux_logits)
    loss.backward()
    dead = [n for n, p in m.named_parameters()
            if p.requires_grad and (p.grad is None or not torch.isfinite(p.grad).all())]
    assert not dead, f"{cls.__name__}: no finite gradient for {dead[:4]}"
    return m


_tbl = architecture_table()
print(_tbl.round(2).to_string(index=False))
print(f"\ncapacity spread: {_tbl.params_M.max() / _tbl.params_M.min():.2f}x "
      f"(1.00x would be perfectly matched)")
for _n, _c in ARCH_REGISTRY.items():
    _probe(_c)
print("all architectures pass the contract check")

## 6. Losses and metrics

The objective is aligned with the reported metric. A soft-Jaccard term is the
IoU surrogate, and Lovasz-hinge (Berman, Triki & Blaschko, CVPR 2018) is the
exact convex surrogate of the Jaccard loss for binary segmentation; it is the
term that moves IoU in the last few points on sparse masks. Boundary weighting
puts extra weight on the 1-2 px rim of each column, where most of the IoU error
lives. `pos_weight` is capped at 3.0. Lovasz switches on after a warmup so BCE +
Dice can form the mask first.

In [ ]:
# ============================================================
# 6. Losses and metrics
# ============================================================

# ---------------- region losses ----------------
def dice_loss(logits, target, eps=1e-6):
    p = torch.sigmoid(logits).flatten(1)
    t = target.float().flatten(1)
    inter = (p * t).sum(1)
    return 1 - ((2 * inter + eps) / (p.sum(1) + t.sum(1) + eps)).mean()


def soft_jaccard_loss(logits, target, eps=1e-6):
    """Direct differentiable relaxation of 1 - IoU. Optimising this is what
    makes the reported IoU move; Dice optimises a different operating point."""
    p = torch.sigmoid(logits).flatten(1)
    t = target.float().flatten(1)
    inter = (p * t).sum(1)
    union = p.sum(1) + t.sum(1) - inter
    return 1 - ((inter + eps) / (union + eps)).mean()


# ---------------- Lovasz hinge ----------------
def _lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    inter = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - inter / union.clamp(min=1e-9)
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge(logits, target, per_image=True):
    """Convex surrogate of the Jaccard loss. Unlike Dice or soft-Jaccard it is a
    tight relaxation, so it keeps producing useful gradient once the easy pixels
    are already right, which is where the last few IoU points are."""
    def _flat(lg, tg):
        lg, tg = lg.reshape(-1), tg.reshape(-1).float()
        if tg.numel() == 0:
            return lg.sum() * 0.0
        errors = 1.0 - lg * (2.0 * tg - 1.0)
        errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
        return torch.dot(F.relu(errors_sorted), _lovasz_grad(tg[perm]))

    if per_image:
        return torch.stack([_flat(lg, tg) for lg, tg in zip(logits, target)]).mean()
    return _flat(logits, target)


# ---------------- boundary weighting ----------------
def boundary_weight_map(target, width=2, w_edge=4.0):
    """Morphological gradient via max-pool: no scipy, no CPU round trip. Pixels
    on the rim of each column get w_edge, everything else gets 1."""
    t = (target > 0.5).float()
    k = 2 * width + 1
    dil = F.max_pool2d(t, k, stride=1, padding=width)
    ero = -F.max_pool2d(-t, k, stride=1, padding=width)
    return 1.0 + (w_edge - 1.0) * (dil - ero).clamp(0, 1)


def weighted_bce(logits, target, pos_weight=None, edge_w=4.0, edge_width=2):
    t = target.float()
    w = boundary_weight_map(t, width=edge_width, w_edge=edge_w)
    loss = F.binary_cross_entropy_with_logits(logits, t, pos_weight=pos_weight,
                                              reduction='none')
    return (loss * w).sum() / w.sum().clamp(min=1.0)


# ---------------- the segmentation objective ----------------
def seg_loss(logits, target, pos_weight=None, epoch=None, beta=None):
    """Phase 1 (early epochs): weighted BCE + Dice, which is stable from a
    random init. Phase 2: soft-Jaccard and Lovasz take over, both aligned with
    the metric. `epoch=None` uses the full objective."""
    w_lov = CFG.LOVASZ_WEIGHT
    if epoch is not None and epoch <= CFG.LOVASZ_WARMUP_EPOCHS:
        w_lov = 0.0

    l = CFG.W_BCE * weighted_bce(logits, target, pos_weight=pos_weight,
                                 edge_w=CFG.EDGE_WEIGHT)
    l = l + CFG.W_DICE * dice_loss(logits, target)
    l = l + CFG.W_JACCARD * soft_jaccard_loss(logits, target)
    if w_lov > 0:
        l = l + w_lov * lovasz_hinge(logits, target)
    return l


# ---------------- denoising objective ----------------
def charbonnier(pred, gt, eps=1e-3):
    return torch.sqrt((pred - gt) ** 2 + eps ** 2).mean()


def gradient_loss(pred, gt):
    dy_p = pred[..., 1:, :] - pred[..., :-1, :]
    dy_g = gt[..., 1:, :] - gt[..., :-1, :]
    dx_p = pred[..., :, 1:] - pred[..., :, :-1]
    dx_g = gt[..., :, 1:] - gt[..., :, :-1]
    return (dy_p - dy_g).abs().mean() + (dx_p - dx_g).abs().mean()


def _gauss_win(ws=11, sigma=1.5, device=None, dtype=torch.float32):
    g = torch.arange(ws, dtype=dtype, device=device) - (ws - 1) / 2
    g = torch.exp(-(g ** 2) / (2 * sigma ** 2))
    g = (g / g.sum())[None]
    return (g.T @ g)[None, None]


def _ssim_map(pred, gt, data_range=1.0, ws=11):
    w = _gauss_win(ws, 1.5, pred.device, pred.dtype)
    pad = ws // 2
    mu1 = F.conv2d(pred, w, padding=pad)
    mu2 = F.conv2d(gt, w, padding=pad)
    mu1s, mu2s, mu12 = mu1 * mu1, mu2 * mu2, mu1 * mu2
    s1 = F.conv2d(pred * pred, w, padding=pad) - mu1s
    s2 = F.conv2d(gt * gt, w, padding=pad) - mu2s
    s12 = F.conv2d(pred * gt, w, padding=pad) - mu12
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    return ((2 * mu12 + c1) * (2 * s12 + c2)) / ((mu1s + mu2s + c1) * (s1 + s2 + c2))


def ssim_loss(pred, gt):
    """Differentiable. SSIM correlates with how well peak finding works
    afterwards far better than MSE does."""
    return 1.0 - _ssim_map(pred.float(), gt.float()).flatten(1).mean(1).mean()


def denoise_loss(pred, gt, lam_grad=None):
    lam_grad = CFG.LAM_GRAD if lam_grad is None else lam_grad
    return (charbonnier(pred, gt)
            + lam_grad * gradient_loss(pred, gt)
            + CFG.LAM_SSIM * ssim_loss(pred, gt))


# ---------------- metrics ----------------
@torch.no_grad()
def ssim_torch(pred, gt, data_range=1.0, ws=11):
    pred = pred.float().clamp(0, data_range)
    gt = gt.float().clamp(0, data_range)
    return _ssim_map(pred, gt, data_range, ws).flatten(1).mean(1)


@torch.no_grad()
def psnr_per_image(pred, gt, data_range=1.0):
    pred = pred.float().clamp(0, data_range)
    gt = gt.float().clamp(0, data_range)
    mse = (pred - gt).square().flatten(1).mean(1)
    return torch.where(mse > 0, 10.0 * torch.log10(data_range ** 2 / mse),
                       torch.full_like(mse, 99.0))


@torch.no_grad()
def iou_per_image(logits, target, thresh=0.0, eps=1e-6):
    """thresh is on the LOGIT, so 0.0 == probability 0.5."""
    p = (logits >= thresh).flatten(1)
    t = (target >= 0.5).flatten(1)
    inter = (p & t).sum(1).float()
    union = (p | t).sum(1).float()
    return (inter + eps) / (union + eps)


@torch.no_grad()
def counts_at(logits, target, thresh):
    """(tp, fp, fn) at one LOGIT threshold. Reference implementation; the
    vectorised counts_grid() in section 7c is checked against it."""
    p = (logits >= thresh).flatten(1)
    t = (target >= 0.5).flatten(1)
    return ((p & t).sum().item(), (p & ~t).sum().item(), (~p & t).sum().item())


def f1_from_counts(tp, fp, fn, eps=1e-9):
    return (2 * tp + eps) / (2 * tp + fp + fn + eps)


def estimate_pos_weight(basenames, max_n=512):
    """Capped at CFG.POS_WEIGHT_MAX. A cap of 20 pushes the model far past the
    precision/recall point that maximises IoU: at 3% positive coverage a
    pos_weight of 20 makes false positives almost free."""
    if CACHE is None:
        return None
    idx = [CACHE_INDEX[b] for b in basenames[:max_n]]
    frac = float(np.mean([np.asarray(CACHE['mask'][i]).mean() for i in idx]))
    if not (0 < frac < 1):
        return None
    w = min((1 - frac) / frac, CFG.POS_WEIGHT_MAX)
    print(f"  positive coverage {frac:.4f} -> pos_weight {w:.2f}")
    return torch.tensor(w, device=DEVICE)


# ---------------- config defaults for the terms above ----------------
for _k, _v in dict(W_BCE=1.0, W_DICE=0.5, W_JACCARD=0.5,
                   LOVASZ_WEIGHT=1.0, LOVASZ_WARMUP_EPOCHS=5,
                   EDGE_WEIGHT=4.0, POS_WEIGHT_MAX=3.0, LAM_SSIM=0.2).items():
    if not hasattr(CFG, _k):
        setattr(CFG, _k, _v)

# ---------------- self-checks ----------------
_g = torch.Generator().manual_seed(0)
_t = (torch.rand(4, 1, 64, 64, generator=_g) > 0.97).float().to(DEVICE)
_perfect = torch.where(_t > 0.5, 12.0, -12.0)
assert lovasz_hinge(_perfect, _t).item() < 1e-3
assert soft_jaccard_loss(_perfect, _t).item() < 1e-3
assert iou_per_image(_perfect, _t).mean().item() > 0.999
_rand = torch.randn(4, 1, 64, 64, device=DEVICE, requires_grad=True)
seg_loss(_rand, _t).backward()
assert _rand.grad is not None and torch.isfinite(_rand.grad).all()

_a = np.random.rand(64, 64).astype(np.float32)
_b = np.clip(_a + 0.08 * np.random.randn(64, 64), 0, 1).astype(np.float32)
_gpu = ssim_torch(torch.from_numpy(_a)[None, None].to(DEVICE),
                  torch.from_numpy(_b)[None, None].to(DEVICE)).item()
_cpu = compute_ssim(_a, _b, data_range=1.0, gaussian_weights=True,
                    sigma=1.5, use_sample_covariance=False)
print(f"SSIM check: gpu {_gpu:.4f}  skimage {_cpu:.4f}  delta {abs(_gpu - _cpu):.5f}")
print(f"seg objective: {CFG.W_BCE}*BCE(edge x{CFG.EDGE_WEIGHT}) + "
      f"{CFG.W_DICE}*Dice + {CFG.W_JACCARD}*Jaccard + "
      f"{CFG.LOVASZ_WEIGHT}*Lovasz(after ep {CFG.LOVASZ_WARMUP_EPOCHS})")
print("loss self-checks passed")

## 7. Training machinery

bfloat16 instead of fp16 + GradScaler where the card supports it: same speed,
same memory, but fp32's exponent range, so there are no inf-loss steps to skip
and no scaler state to checkpoint. Falls back to fp16 on older cards.

In [ ]:
# ============================================================
# 7. Training machinery: AMP, EMA, optimizer, fit()
# ============================================================
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = False

AMP_DTYPE = {"bf16": torch.bfloat16, "fp16": torch.float16,
             "fp32": torch.float32}[CFG.PRECISION]
AMP_ON = DEVICE.type == 'cuda' and CFG.PRECISION != 'fp32'
NEEDS_SCALER = AMP_ON and AMP_DTYPE is torch.float16


def amp_autocast():
    if not AMP_ON:
        return torch.autocast('cuda', enabled=False) if _AMP_NEW else _autocast(enabled=False)
    return (_autocast('cuda', dtype=AMP_DTYPE, enabled=True) if _AMP_NEW
            else _autocast(enabled=True))


def make_scaler():
    return _GradScaler('cuda', enabled=NEEDS_SCALER) if _AMP_NEW \
        else _GradScaler(enabled=NEEDS_SCALER)


print(f"precision: {CFG.PRECISION}  (GradScaler "
      f"{'on' if NEEDS_SCALER else 'not needed'})")


# ---------------- EMA ----------------
class EMA:
    """Exponential moving average of the weights. On this dataset the EMA model
    is consistently ~0.005-0.015 IoU above the raw weights and much less jumpy
    from epoch to epoch, which makes early stopping meaningful."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
        self.buffers = {k: v.detach().clone()
                        for k, v in model.state_dict().items()
                        if not v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model, step=None):
        d = self.decay
        if step is not None:                       # warm up the average
            d = min(d, (1 + step) / (10 + step))
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(d).add_(v.detach().float(), alpha=1 - d)
            else:
                self.buffers[k] = v.detach().clone()

    def state_dict(self):
        return {**{k: v.clone() for k, v in self.shadow.items()},
                **{k: v.clone() for k, v in self.buffers.items()}}

    def load_state_dict(self, sd):
        for k, v in sd.items():
            if k in self.shadow:
                self.shadow[k] = v.detach().clone().float().to(self.shadow[k].device)
            elif k in self.buffers:
                self.buffers[k] = v.detach().clone().to(self.buffers[k].device)
            else:
                self.buffers[k] = v.detach().clone()


# ---------------- optimizer / placement ----------------
def make_optimizer(model, lr, wd):
    """No weight decay on norms and biases. Costs one line and is worth a few
    hundredths of IoU on small datasets."""
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if p.ndim <= 1 or n.endswith('.bias') else decay).append(p)
    groups = [{'params': decay, 'weight_decay': wd},
              {'params': no_decay, 'weight_decay': 0.0}]
    if DEVICE.type == 'cuda':
        try:
            return torch.optim.AdamW(groups, lr=lr, fused=False)
        except (TypeError, RuntimeError):
            pass
    try:
        return torch.optim.AdamW(groups, lr=lr, foreach=(DEVICE.type == 'cuda'))
    except TypeError:
        return torch.optim.AdamW(groups, lr=lr)


def prep_model(model):
    model = model.to(DEVICE)
    if CFG.CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    if CFG.COMPILE and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model, mode='max-autotune')
        except Exception as e:
            print(f"  torch.compile unavailable ({e}); running eager")
    return model


def to_dev(t):
    t = t.to(DEVICE, non_blocking=True)
    return t.to(memory_format=torch.channels_last) if CFG.CHANNELS_LAST else t


def copy_for_eval(model, ema):
    """Load the EMA weights into a throwaway copy so the training weights stay
    where the optimizer left them."""
    global _EVAL_CACHE
    if '_EVAL_CACHE' not in globals():
        _EVAL_CACHE = {}
    key = id(model)
    if key not in _EVAL_CACHE:
        import copy as _copy
        stash = getattr(model, 'aux_logits', None)
        try:
            if hasattr(model, 'aux_logits'):
                model.aux_logits = None
            _EVAL_CACHE[key] = _copy.deepcopy(model)
        finally:
            if hasattr(model, 'aux_logits'):
                model.aux_logits = stash
    tgt = _EVAL_CACHE[key]
    tgt.load_state_dict(ema.state_dict(), strict=False)
    tgt.eval()
    return tgt


# ---------------- evaluation ----------------
THRESH_GRID = np.linspace(-2.0, 2.0, 17)          # logits, 0.0 == p 0.5


@torch.inference_mode()
def evaluate_reference(model, loader, sweep_threshold=True):
    """PSNR/SSIM/IoU on the GPU, plus a segmentation-threshold sweep reporting
    the best-F1 operating point: judging every architecture at a hard-coded 0.5
    rewards whichever one happens to be calibrated there rather than whichever
    separates atoms best.

    This is the readable reference version. Section 7c installs `evaluate()`,
    which computes the same numbers with a vectorised sweep."""
    model.eval()
    psnr_vals, ssim_vals, iou_vals = [], [], []
    counts = np.zeros((len(THRESH_GRID), 3), dtype=np.int64)

    for batch in tqdm(loader, desc='val', leave=False):
        noisy, clean, mask = to_dev(batch[0]), to_dev(batch[1]), to_dev(batch[2])
        with amp_autocast():
            den, seg_logits = model(noisy)
        den, seg_logits = den.float(), seg_logits.float()

        psnr_vals.append(psnr_per_image(den, clean).cpu())
        ssim_vals.append(ssim_torch(den, clean).cpu())
        iou_vals.append(iou_per_image(seg_logits, mask).cpu())

        if sweep_threshold:
            for i, th in enumerate(THRESH_GRID):
                counts[i] += counts_at(seg_logits, mask, float(th))

    out = dict(psnr=float(torch.cat(psnr_vals).mean()),
               ssim=float(torch.cat(ssim_vals).mean()),
               iou=float(torch.cat(iou_vals).mean()))

    if sweep_threshold:
        f1s = np.array([f1_from_counts(*c) for c in counts])
        k = int(f1s.argmax())
        tp, fp, fn = counts[k]
        out.update(best_thresh_logit=float(THRESH_GRID[k]),
                   best_thresh_prob=float(1 / (1 + np.exp(-THRESH_GRID[k]))),
                   best_f1=float(f1s[k]),
                   precision=float(tp / max(tp + fp, 1)),
                   recall=float(tp / max(tp + fn, 1)))
    return out


@torch.no_grad()
def gaussian_baseline_over_loader(loader, sigma=1.0, max_batches=20):
    """Depends only on the data, so it is computed once per fold and cached in
    the resume file. Capped at max_batches: the mean is stable well before the
    full validation set."""
    vals = []
    for i, batch in enumerate(tqdm(loader, desc='gauss baseline', leave=False)):
        n_np, c_np = batch[0].numpy(), batch[1].numpy()
        for j in range(n_np.shape[0]):
            d = cv2.GaussianBlur(n_np[j, 0], (0, 0), sigma)
            vals.append(compute_psnr(np.clip(c_np[j, 0], 0, 1),
                                     np.clip(d, 0, 1), data_range=1.0))
        if max_batches and i + 1 >= max_batches:
            break
    return float(np.mean(vals))


# ---------------- checkpoint helpers ----------------
def _atomic_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = str(path) + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = str(path) + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, default=float)
    os.replace(tmp, path)


def _load_torch_file(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _optimizer_to(opt, device):
    for state in opt.state.values():
        for k, v in list(state.items()):
            if torch.is_tensor(v):
                state[k] = v.to(device, non_blocking=True)


def _restore_rng(st):
    if st.get('rng_torch') is not None:
        torch.set_rng_state(st['rng_torch'].detach().cpu().to(torch.uint8))
    if DEVICE.type == 'cuda' and st.get('rng_cuda') is not None:
        torch.cuda.set_rng_state_all([s.detach().cpu().to(torch.uint8)
                                      for s in st['rng_cuda']])
    if st.get('rng_np') is not None:
        np.random.set_state(st['rng_np'])
    if st.get('rng_py') is not None:
        random.setstate(st['rng_py'])


class OutOfTime(Exception):
    """Raised when the Slurm walltime is about to expire."""


# ---------------- fit ----------------
def fit(model, train_dl, val_dl, epochs, lr, wd, ckpt_path,
        patience=None, min_epochs=None, warmup_epochs=2, min_delta=1e-4,
        resume_path=None, pos_weight=None, ema_decay=None, tag=''):
    patience = CFG.PATIENCE if patience is None else patience
    min_epochs = CFG.MIN_EPOCHS if min_epochs is None else min_epochs
    ema_decay = CFG.EMA_DECAY if ema_decay is None else ema_decay

    model = prep_model(model)
    opt = make_optimizer(model, lr, wd)
    scaler = make_scaler()
    ema = EMA(model, ema_decay) if ema_decay else None

    warmup_epochs = min(warmup_epochs, max(epochs - 1, 0))
    if warmup_epochs > 0:
        sched = torch.optim.lr_scheduler.SequentialLR(
            opt,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.05,
                                                  total_iters=warmup_epochs),
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    opt, T_max=max(epochs - warmup_epochs, 1), eta_min=lr * 0.01)],
            milestones=[warmup_epochs])
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=max(epochs, 1), eta_min=lr * 0.01)

    start_ep, history, best_iou, bad, gauss_psnr = 0, [], -1.0, 0, None

    if resume_path and os.path.isfile(resume_path):
        try:
            st = _load_torch_file(resume_path, map_location='cpu')
            model.load_state_dict(st['model'])
            opt.load_state_dict(st['opt'])
            _optimizer_to(opt, DEVICE)
            if st.get('scaler') is not None:
                scaler.load_state_dict(st['scaler'])
            if st.get('sched') is not None:
                sched.load_state_dict(st['sched'])
            if ema is not None and st.get('ema') is not None:
                ema.load_state_dict(st['ema'])
            _restore_rng(st)
            start_ep = int(st.get('epoch', 0))
            history = list(st.get('history', []))
            best_iou = float(st.get('best_iou', -1.0))
            bad = int(st.get('bad_epochs', 0))
            gauss_psnr = st.get('gauss_psnr')
            if not os.path.isfile(ckpt_path):
                best_iou, bad = -1.0, 0
            print(f"  resumed after epoch {start_ep}/{epochs} "
                  f"(best IoU {best_iou:.4f}, bad {bad}/{patience})")
        except Exception as e:
            print(f"  could not read resume file ({e}); starting fresh")
            start_ep, history, best_iou, bad, gauss_psnr = 0, [], -1.0, 0, None

    if gauss_psnr is None:
        gauss_psnr = gaussian_baseline_over_loader(val_dl)
    print(f"  Gaussian-blur baseline PSNR (val): {gauss_psnr:.2f} dB")

    def save_resume(ep_done):
        if resume_path is None:
            return
        _atomic_save({
            'model': model.state_dict(), 'opt': opt.state_dict(),
            'scaler': scaler.state_dict(), 'sched': sched.state_dict(),
            'ema': ema.state_dict() if ema is not None else None,
            'epoch': ep_done, 'history': history, 'best_iou': best_iou,
            'bad_epochs': bad, 'gauss_psnr': gauss_psnr,
            'rng_torch': torch.get_rng_state(),
            'rng_cuda': (torch.cuda.get_rng_state_all()
                         if DEVICE.type == 'cuda' else None),
            'rng_np': np.random.get_state(), 'rng_py': random.getstate(),
        }, resume_path)

    ep = start_ep
    eval_model = model
    try:
        for ep in range(start_ep + 1, epochs + 1):
            t0 = time.time()
            tr = train_one_epoch(model, train_dl, opt, scaler, ema=ema,
                                 pos_weight=pos_weight, epoch=ep)
            sched.step()

            if ema is not None:
                eval_model = copy_for_eval(model, ema)
            m = evaluate(eval_model, val_dl)

            m.update(epoch=ep, train_loss=tr['loss'], l_den=tr['l_den'],
                     l_seg=tr['l_seg'], img_per_s=tr['img_per_s'],
                     gauss_psnr=gauss_psnr, lr=opt.param_groups[0]['lr'],
                     time=time.time() - t0,
                     gpu_gb=(torch.cuda.max_memory_allocated() / 1024**3
                             if DEVICE.type == 'cuda' else 0.0))
            history.append(m)

            print(f"  ep {ep:02d}  loss {tr['loss']:.4f} "
                  f"(den {tr['l_den']:.4f}/seg {tr['l_seg']:.4f})  "
                  f"PSNR {m['psnr']:.2f}  SSIM {m['ssim']:.3f}  "
                  f"IoU {m['iou']:.4f}  F1 {m.get('best_f1', float('nan')):.4f}"
                  f"@p{m.get('best_thresh_prob', 0.5):.2f}  "
                  f"[{m['time']:.0f}s {tr['img_per_s']:.0f} img/s "
                  f"{m['gpu_gb']:.1f}GB]", flush=True)

            if m['iou'] > best_iou + min_delta:
                best_iou, bad = float(m['iou']), 0
                _atomic_save({'model': (ema.state_dict() if ema is not None
                                        else model.state_dict()),
                              'raw': model.state_dict(),
                              'arch': tag, 'epoch': ep, 'metrics': m}, ckpt_path)
            else:
                bad += 1

            save_resume(ep)

            if STOP_REQUESTED['flag']:
                raise OutOfTime("SIGTERM")
            if time_is_short():
                raise OutOfTime(f"{seconds_left()/60:.1f} min of walltime left")
            if ep >= min_epochs and bad >= patience:
                print(f"  early stop at ep {ep} (best IoU {best_iou:.4f})")
                break

    except OutOfTime as e:
        save_resume(ep)
        print(f"  stopping cleanly: {e}. State saved to {resume_path}; "
              f"resubmit the job to continue from epoch {ep + 1}.")
        raise
    except KeyboardInterrupt:
        save_resume(max(start_ep, ep - 1))
        print("  interrupted; rerun to resume from the last completed epoch")
        raise

    return history

## 7b. GPU memory management

Gradient checkpointing recomputes activations in the backward pass instead of
storing them: roughly 40-60% less activation memory for about 20-30% more time,
and it costs accuracy nothing. `oom_safe_step` catches an OOM mid-epoch, frees,
and retries the batch in halves instead of losing the run.

HRNet is the architecture that dies, because it keeps a full-resolution branch
alive through every stage. At 256x256 its activation footprint is several times
AtomSegNet's even though the two have similar parameter counts. Activations fill
the card here, not parameters.

In [ ]:
# ============================================================
# 7b. Gradient checkpointing + OOM-resilient training step
# ============================================================
from torch.utils.checkpoint import checkpoint as _ckpt


def _oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or \
        'out of memory' in str(e).lower()


def free_gpu():
    globals().pop('_EVAL_CACHE', None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


class _CheckpointWrapper(nn.Module):
    """Wraps a submodule so its activations are recomputed in backward."""

    def __init__(self, mod):
        super().__init__()
        self.mod = mod

    def forward(self, *args, **kwargs):
        if self.training and torch.is_grad_enabled() and any(
                torch.is_tensor(a) and a.requires_grad for a in args):
            return _ckpt(self.mod, *args, use_reentrant=False, **kwargs)
        return self.mod(*args, **kwargs)


def enable_grad_checkpointing(model, types=(ResBlock, FusionUnit), verbose=True):
    """Wrap every ResBlock and FusionUnit in the model. Call this on the raw
    model BEFORE prep_model / torch.compile."""
    n = 0
    for parent in list(model.modules()):
        for name, child in list(parent.named_children()):
            if isinstance(child, types) and not isinstance(child, _CheckpointWrapper):
                setattr(parent, name, _CheckpointWrapper(child))
                n += 1
    if verbose:
        print(f"  gradient checkpointing on {n} blocks")
    return model


# ---------------- OOM-resilient step ----------------
def oom_safe_step(model, batch_tensors, opt, scaler, pos_weight, epoch,
                  accum, clip_norm, ema, splits=1):
    """Run one training step. On OOM, halve the batch and run the halves
    sequentially rather than losing the epoch. Returns (loss, l_den, l_seg) or
    None if even a single sample will not fit.

    NOTE: this applies UNet++ deep supervision (the aux heads) at weight 0.3.
    Removing that term changes the objective, so change it here and nowhere
    else; there is only one train_one_epoch in this notebook now."""
    noisy, clean, mask = batch_tensors
    try:
        with amp_autocast():
            den, seg = model(noisy)
            l_den = denoise_loss(den.float(), clean)
            l_seg = seg_loss(seg.float(), mask, pos_weight=pos_weight, epoch=epoch)
            aux = getattr(model, 'aux_logits', None)
            if aux:
                l_seg = l_seg + 0.3 * sum(
                    seg_loss(a.float(), mask, pos_weight=pos_weight, epoch=epoch)
                    for a in aux) / len(aux)
            loss = (CFG.LAM_DEN * l_den + CFG.LAM_SEG * l_seg) / accum
        scaler.scale(loss).backward()
        return float(loss.item() * accum), float(l_den.item()), float(l_seg.item())
    except Exception as e:
        if not _oom(e) or noisy.shape[0] < 2 or splits > 4:
            if _oom(e):
                print(f"  OOM at batch size {noisy.shape[0]}; skipping batch")
                free_gpu()
                return None
            raise
        free_gpu()
        h = noisy.shape[0] // 2
        print(f"  OOM at batch {noisy.shape[0]}, retrying as 2 x {h}", flush=True)
        outs = []
        for sl in (slice(0, h), slice(h, None)):
            r = oom_safe_step(model, (noisy[sl], clean[sl], mask[sl]), opt, scaler,
                              pos_weight, epoch, accum * 2, clip_norm, ema, splits + 1)
            if r is not None:
                outs.append(r)
        if not outs:
            return None
        return tuple(float(np.mean([o[i] for o in outs])) for i in range(3))


def train_one_epoch(model, loader, opt, scaler, ema=None, pos_weight=None,
                    epoch=0, clip_norm=1.0, accum=None):
    model.train()
    accum = max(1, int(accum if accum is not None else CFG.GRAD_ACCUM))
    tot = den_sum = seg_sum = gnorm_sum = 0.0
    n_steps = n_opt = seen = n_oom = 0
    t0 = time.time()
    pbar = tqdm(loader, desc=f'train ep{epoch}', leave=False)

    opt.zero_grad(set_to_none=True)
    for it, batch in enumerate(pbar):
        tensors = (to_dev(batch[0]), to_dev(batch[1]), to_dev(batch[2]))
        r = oom_safe_step(model, tensors, opt, scaler, pos_weight, epoch,
                          accum, clip_norm, ema)
        if r is None:
            n_oom += 1
            opt.zero_grad(set_to_none=True)
            continue
        li, ld, ls = r

        if (it + 1) % accum == 0:
            if clip_norm:
                scaler.unscale_(opt)
                gn = torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
                if math.isfinite(float(gn)):
                    gnorm_sum += float(gn)
                    n_opt += 1
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        if math.isfinite(li):
            tot += li
            den_sum += ld
            seg_sum += ls
            n_steps += 1

        if hasattr(model, 'aux_logits'):
            model.aux_logits = None

        seen += tensors[0].shape[0]
        if it % 20 == 0:
            pbar.set_postfix(loss=f"{li:.4f}", den=f"{ld:.4f}", seg=f"{ls:.4f}",
                             ips=f"{seen / max(time.time() - t0, 1e-6):.0f}")

    if n_oom:
        print(f"  WARNING: {n_oom} batches skipped on OOM this epoch. "
              f"Lower CFG.BATCH_OVERRIDE['<arch>'] rather than living with it.")

    n_steps = max(n_steps, 1)
    return dict(loss=tot / n_steps, l_den=den_sum / n_steps, l_seg=seg_sum / n_steps,
                grad_norm=gnorm_sum / max(n_opt, 1), n_oom=n_oom,
                img_per_s=seen / max(time.time() - t0, 1e-6))


print("memory helpers ready: enable_grad_checkpointing(), oom_safe_step(), free_gpu()")

## 7c. Fast, OOM-safe evaluation

`counts_at()` ends in three `.item()` calls and `THRESH_GRID` has 17 entries, so
a naive sweep forces 51 blocking GPU-to-CPU synchronisations per validation
batch. At eval batch 4 over ~2873 validation images that is ~36k syncs per
fold-epoch, and because the val progress bar uses `leave=False` the cell reads
as frozen rather than slow. `counts_grid()` evaluates the whole grid in one
broadcast and accumulates on the GPU, so the whole validation pass costs a
single sync. The metrics are unchanged, and the equivalence is checked against
`counts_at()` before anything is installed.

In [ ]:
# ============================================================
# 7c. Vectorised threshold sweep + OOM-safe evaluate()
# ============================================================
USE_FAST_EVAL = True      # False falls back to evaluate_reference()
VERIFY_FAST_EVAL = True   # equivalence check against counts_at(), ~instant


@torch.no_grad()
def counts_grid(logits, target, grid, max_elems=64_000_000):
    """(tp, fp, fn) for EVERY threshold in `grid`, in one pass, no host sync.

    Returns an int64 tensor of shape (len(grid), 3) on the same device as
    `logits`. Identical to calling counts_at() once per grid entry, minus the
    3*len(grid) synchronisations. fn is derived as (positives - tp) rather than
    materialising (~p & t), which saves one T x B x N boolean tensor."""
    t = (target >= 0.5).flatten(1)                       # B, N
    x = logits.flatten(1)                                # B, N
    th = torch.as_tensor(grid, device=x.device, dtype=x.dtype)

    n_pos = t.sum()
    chunk = max(1, int(max_elems // max(x.numel(), 1)))   # thresholds at once

    tps, fps = [], []
    for s in range(0, th.numel(), chunk):
        p = x.unsqueeze(0) >= th[s:s + chunk].view(-1, 1, 1)   # T_c, B, N
        tps.append((p & t).sum(dim=(1, 2)))
        fps.append((p & ~t).sum(dim=(1, 2)))
        del p

    tp, fp = torch.cat(tps), torch.cat(fps)
    return torch.stack((tp, fp, n_pos - tp), dim=1).to(torch.int64)


if VERIFY_FAST_EVAL:
    _vg = torch.Generator().manual_seed(0)
    _vlogit = (torch.randn(3, 1, 32, 32, generator=_vg) * 2.0).to(DEVICE)
    _vmask = (torch.rand(3, 1, 32, 32, generator=_vg) > 0.95).float().to(DEVICE)
    _ref = np.array([counts_at(_vlogit, _vmask, float(th)) for th in THRESH_GRID],
                    dtype=np.int64)
    _new = counts_grid(_vlogit, _vmask, THRESH_GRID).cpu().numpy()
    if not np.array_equal(_ref, _new):
        raise RuntimeError("counts_grid() disagrees with counts_at(); refusing to "
                           "install the fast sweep. Set USE_FAST_EVAL = False.")
    print(f"fast sweep verified against counts_at() on {len(THRESH_GRID)} thresholds")
    del _vg, _vlogit, _vmask, _ref, _new


@torch.inference_mode()
def _fast_evaluate(model, loader, sweep_threshold=True):
    """Same metrics as evaluate_reference(), without the per-threshold host
    synchronisations. Per-image metrics also stay on the GPU until the end;
    .cpu() inside the loop was another three syncs per batch."""
    model.eval()
    psnr_vals, ssim_vals, iou_vals = [], [], []
    counts = None

    for batch in tqdm(loader, desc="val", leave=False):
        noisy, clean, mask = to_dev(batch[0]), to_dev(batch[1]), to_dev(batch[2])
        with amp_autocast():
            den, seg_logits = model(noisy)
        den, seg_logits = den.float(), seg_logits.float()

        psnr_vals.append(psnr_per_image(den, clean))
        ssim_vals.append(ssim_torch(den, clean))
        iou_vals.append(iou_per_image(seg_logits, mask))

        if sweep_threshold:
            c = counts_grid(seg_logits, mask, THRESH_GRID)
            counts = c if counts is None else counts + c

    out = dict(psnr=float(torch.cat(psnr_vals).mean()),
               ssim=float(torch.cat(ssim_vals).mean()),
               iou=float(torch.cat(iou_vals).mean()))

    if sweep_threshold and counts is not None:
        counts = counts.cpu().numpy()          # the one and only sync
        f1s = np.array([f1_from_counts(*c) for c in counts])
        k = int(f1s.argmax())
        tp, fp, fn = counts[k]
        out.update(best_thresh_logit=float(THRESH_GRID[k]),
                   best_thresh_prob=float(1.0 / (1.0 + np.exp(-THRESH_GRID[k]))),
                   best_f1=float(f1s[k]),
                   precision=float(tp / max(tp + fp, 1)),
                   recall=float(tp / max(tp + fn, 1)))
    return out


_sweep_core = _fast_evaluate if USE_FAST_EVAL else evaluate_reference


@torch.inference_mode()
def evaluate(model, loader, sweep=True):
    """PSNR, SSIM, IoU and the threshold sweep / best F1. Retries once after a
    CUDA OOM; any non-memory error is a real bug and propagates."""
    try:
        return _sweep_core(model, loader, sweep_threshold=bool(sweep))
    except Exception as e:
        if not _oom(e):
            raise
        print("\n  CUDA OOM during validation. Clearing cache and retrying once...",
              flush=True)
        free_gpu()
        try:
            return _sweep_core(model, loader, sweep_threshold=bool(sweep))
        except Exception as e2:
            if _oom(e2):
                raise RuntimeError("Evaluation still OOM after GPU cleanup. "
                                   "Reduce the validation batch size.") from e2
            raise


print("evaluate() installed | sweep:",
      "counts_grid (vectorised)" if USE_FAST_EVAL else "counts_at (reference)")

## 8. Noise2Void self-supervised pre-training

About 2% of pixels are replaced by a random neighbour and the loss is MSE at
those positions against the original noisy value. No clean targets are used. The
resulting weights warm-start the supervised stage.

Memory handling: gradient checkpointing on every ResBlock/FusionUnit, a real
forward+backward probe to pick the N2V batch size per architecture (the
supervised batch is not automatically safe here, since N2V has only the denoise
branch in its loss but the same activation cost), and an OOM mid-epoch halves
the batch and retries rather than killing the job.

In [ ]:
# ============================================================
# 8. Noise2Void self-supervised pre-training (per-architecture, OOM-safe)
# ============================================================
for _k, _v in dict(N2V_LR=5e-4, N2V_VAL_FRACTION=0.05, N2V_PATIENCE=5,
                   N2V_MIN_EPOCHS=5, N2V_EMA_DECAY=0.999,
                   N2V_GRAD_CKPT=True, N2V_AUTOTUNE=True,
                   N2V_EVAL_BATCH=8, N2V_BATCH_OVERRIDE={}).items():
    if not hasattr(CFG, _k):
        setattr(CFG, _k, _v)
import shutil

# ---------------- N2V artefact location ----------------
# Everything Noise2Void produces lives in <RESULTS_ROOT>/with n2v/N2V
N2V_ROOT = n2v_dir()


def n2v_path(name):
    """Absolute paths pass through; bare filenames resolve into N2V_ROOT."""
    if os.path.isabs(name) or re.match(r'^[A-Za-z]:[\\/]', str(name)):
        return Path(name)
    return N2V_ROOT / str(name)


def adopt_legacy_n2v(verbose=True):
    """Copy finished N2V weights (and their resume/history siblings) from the
    old output folders into N2V_ROOT, so a rerun sees them as done instead of
    starting from scratch. Copies rather than moves; the originals stay put."""
    legacy_dirs = []
    for d in (CFG.OUT_DIR, str(Path(CFG.PROJECT_ROOT) / "atomsegnet"),
              str(CFG.PROJECT_ROOT), str(CFG.RESULTS_ROOT),
              str(variant_root(True, create=False))):
        p = Path(d)
        if p.is_dir() and p.resolve() != N2V_ROOT.resolve():
            if p.resolve() not in [q.resolve() for q in legacy_dirs]:
                legacy_dirs.append(p)

    adopted = []
    for arch, fname in N2V_FILES.items():
        dst = N2V_ROOT / fname
        if dst.is_file():
            continue
        for d in legacy_dirs:
            src = d / fname
            if not src.is_file():
                continue
            shutil.copy2(src, dst)
            adopted.append((arch, str(src)))
            # bring the siblings along so resume state and history stay aligned
            for suffix in (".resume", ".best"):
                s = d / (fname + suffix)
                if s.is_file() and not (N2V_ROOT / (fname + suffix)).is_file():
                    shutil.copy2(s, N2V_ROOT / (fname + suffix))
            h = d / fname.replace(".pt", "_history.json")
            if h.is_file():
                shutil.copy2(h, N2V_ROOT / h.name)
            break

    # An interrupted run leaves only a .resume; that is worth adopting too.
    for arch, fname in N2V_FILES.items():
        if (N2V_ROOT / fname).is_file() or (N2V_ROOT / (fname + ".resume")).is_file():
            continue
        for d in legacy_dirs:
            s = d / (fname + ".resume")
            if s.is_file():
                shutil.copy2(s, N2V_ROOT / (fname + ".resume"))
                adopted.append((arch, str(s) + " (partial)"))
                break

    if verbose:
        print(f"N2V artefacts: {N2V_ROOT}")
        for arch, src in adopted:
            print(f"  adopted {arch} from {src}")
        for arch, fname in N2V_FILES.items():
            p = N2V_ROOT / fname
            r = N2V_ROOT / (fname + ".resume")
            state = ("done" if p.is_file()
                     else "resumable" if r.is_file() else "not started")
            print(f"  {arch:11s} {state}")
    return adopted

def n2v_loss(pred, target, mask):
    """MSE restricted to the masked (blind-spot) pixels."""
    return ((pred - target) ** 2 * mask).sum() / (mask.sum() + 1e-6)


N2V_FILES = {"AtomSegNet": "atomsegnet_n2v.pt",
             "UNetPP":     "unetpp_n2v.pt",
             "HRNet":      "hrnet_n2v.pt"}


def _n2v_splits():
    """One split shared by all three architectures, so the N2V numbers are
    comparable across them."""
    rng = np.random.RandomState(SEED + 7)
    perm = rng.permutation(len(common))
    n_val = max(16, int(round(CFG.N2V_VAL_FRACTION * len(common))))
    return ([common[i] for i in perm[n_val:]], [common[i] for i in perm[:n_val]])


N2V_TRAIN, N2V_VAL = _n2v_splits()


def autotune_n2v_batch(arch_name, candidates=None, headroom=0.80):
    """Probe a real N2V forward+backward+step."""
    if arch_name in CFG.N2V_BATCH_OVERRIDE:
        return CFG.N2V_BATCH_OVERRIDE[arch_name]
    if not torch.cuda.is_available():
        return min(arch_batch(arch_name), 8)

    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    start = arch_batch(arch_name)
    candidates = candidates or sorted(
        {b for b in (start, 32, 24, 16, 12, 8, 4, 2) if b <= max(start, 32)},
        reverse=True)

    for b in candidates:
        free_gpu()
        model = ARCH_REGISTRY[arch_name]()
        if CFG.N2V_GRAD_CKPT:
            enable_grad_checkpointing(model, verbose=False)
        model = prep_model(model)
        model.train()
        opt = make_optimizer(model, CFG.N2V_LR, CFG.WD)
        scaler = make_scaler()
        try:
            x = to_dev(torch.rand(b, CFG.IN_CH, CFG.IMG_SIZE, CFG.IMG_SIZE))
            m = to_dev((torch.rand(b, 1, CFG.IMG_SIZE, CFG.IMG_SIZE) > 0.98).float())
            with amp_autocast():
                den, _ = model(x)
                loss = n2v_loss(den.float(), x, m)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            peak = torch.cuda.max_memory_allocated() / 1024**3
            if peak < headroom * total:
                print(f"  {arch_name} N2V: batch {b} fits "
                      f"(peak {peak:.1f} / {total:.0f} GiB)")
                CFG.N2V_BATCH_OVERRIDE[arch_name] = b
                return b
            print(f"  {arch_name} N2V: batch {b} peaks at {peak:.1f} GiB, "
                  f"over the {headroom:.0%} margin")
        except Exception as e:
            if not _oom(e):
                raise
            print(f"  {arch_name} N2V: batch {b} OOM")
        finally:
            del model, opt, scaler
            free_gpu()

    CFG.N2V_BATCH_OVERRIDE[arch_name] = 2
    print(f"  {arch_name} N2V: falling back to batch 2")
    return 2


def _n2v_step(model, x_in, x_gt, m, opt, scaler, ema, clip=1.0, depth=0):
    """Returns the loss, or None if the batch could not be made to fit."""
    try:
        opt.zero_grad(set_to_none=True)
        with amp_autocast():
            den, _ = model(x_in)
            loss = n2v_loss(den.float(), x_gt, m)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(opt)
        scaler.update()
        if ema is not None:
            ema.update(model)
        return float(loss.item())
    except Exception as e:
        if not _oom(e):
            raise
        opt.zero_grad(set_to_none=True)
        free_gpu()
        if x_in.shape[0] < 2 or depth >= 4:
            print(f"  OOM at batch {x_in.shape[0]}; skipping", flush=True)
            return None
        h = x_in.shape[0] // 2
        print(f"  OOM at batch {x_in.shape[0]}, retrying as 2 x {h}", flush=True)
        vals = [v for sl in (slice(0, h), slice(h, None))
                for v in [_n2v_step(model, x_in[sl], x_gt[sl], m[sl],
                                    opt, scaler, ema, clip, depth + 1)]
                if v is not None]
        return float(np.mean(vals)) if vals else None


@torch.inference_mode()
def n2v_validate(model, dl):
    """Masked-pixel loss on held-out frames. The only quantity N2V is allowed to
    be selected on."""
    model.eval()
    tot = n = 0.0
    for x_in, x_gt, m in dl:
        x_in, x_gt, m = to_dev(x_in), to_dev(x_gt), to_dev(m)
        try:
            with amp_autocast():
                den, _ = model(x_in)
        except Exception as e:
            if not _oom(e):
                raise
            free_gpu()
            continue
        w = m.sum().item()
        if w > 0:
            tot += n2v_loss(den.float(), x_gt, m).item() * w
            n += w
    return tot / max(n, 1e-6)


@torch.inference_mode()
def n2v_psnr_probe(model, dl, max_batches=8):
    """Denoising quality against noNoise. Monitoring only: N2V never trains on
    clean data and this value never influences selection."""
    model.eval()
    ps, gs = [], []
    for i, batch in enumerate(dl):
        noisy, clean = to_dev(batch[0]), to_dev(batch[1])
        try:
            with amp_autocast():
                den, _ = model(noisy)
        except Exception as e:
            if not _oom(e):
                raise
            free_gpu()
            continue
        ps.append(psnr_per_image(den.float(), clean).cpu())
        n_np, c_np = batch[0].numpy(), batch[1].numpy()
        for j in range(n_np.shape[0]):
            b = cv2.GaussianBlur(n_np[j, 0], (0, 0), 1.0)
            gs.append(compute_psnr(np.clip(c_np[j, 0], 0, 1),
                                   np.clip(b, 0, 1), data_range=1.0))
        if max_batches and i + 1 >= max_batches:
            break
    if not ps:
        return float('nan'), float('nan')
    return float(torch.cat(ps).mean()), float(np.mean(gs))


def pretrain_n2v(arch_name, epochs=None, ckpt=None, force=False):
    if arch_name not in ARCH_REGISTRY:
        raise ValueError(f"Unknown architecture: {arch_name}")

    epochs = CFG.N2V_EPOCHS if epochs is None else epochs
    ckpt = ckpt or N2V_FILES[arch_name]
    final_path = str(n2v_path(ckpt))
    os.makedirs(os.path.dirname(final_path) or ".", exist_ok=True)
    resume_path = final_path + ".resume"

    if os.path.isfile(final_path) and not force:
        print(f"N2V already finished for {arch_name} -> {final_path}")
        return final_path

    free_gpu()
    seed_everything(SEED + 100 * (CFG.ARCHS.index(arch_name)
                                  if arch_name in CFG.ARCHS else 0))

    batch = autotune_n2v_batch(arch_name) if CFG.N2V_AUTOTUNE else arch_batch(arch_name)
    eval_batch = max(1, min(CFG.N2V_EVAL_BATCH, batch))

    print(f"\n=== N2V pretraining: {arch_name} | batch {batch} (eval {eval_batch}) "
          f"| <= {epochs} epochs | train {len(N2V_TRAIN)} val {len(N2V_VAL)} ===")

    def mk_n2v(names, train):
        return N2VDataset(names, noisy_map, size=CFG.IMG_SIZE,
                          ratio=CFG.N2V_MASK_RATIO, radius=CFG.N2V_RADIUS, train=train)

    tr_dl = DataLoader(mk_n2v(N2V_TRAIN, True),
                       **loader_options(True, drop_last=True, batch=batch))
    if len(tr_dl) == 0:
        raise RuntimeError(
            f"N2V train loader for {arch_name} has 0 batches: {len(N2V_TRAIN)} "
            f"samples at batch {batch} with drop_last=True.")
    va_dl = DataLoader(mk_n2v(N2V_VAL, False), **loader_options(False, batch=eval_batch))
    probe_dl = DataLoader(
        TEMSegDataset(N2V_VAL, noisy_map, clean_map, mask_map,
                      gauss_map=gauss_map, train=False),
        **loader_options(False, batch=eval_batch))

    model = ARCH_REGISTRY[arch_name]()
    if CFG.N2V_GRAD_CKPT:
        enable_grad_checkpointing(model)
    model = prep_model(model)

    opt = make_optimizer(model, CFG.N2V_LR, CFG.WD)
    scaler = make_scaler()
    ema = EMA(model, CFG.N2V_EMA_DECAY) if CFG.N2V_EMA_DECAY else None
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(epochs, 1), eta_min=CFG.N2V_LR * 0.01)

    cfg_now = dict(size=CFG.IMG_SIZE, ratio=CFG.N2V_MASK_RATIO,
                   radius=CFG.N2V_RADIUS, lr=CFG.N2V_LR, n=len(N2V_TRAIN),
                   arch=arch_name, batch=batch, grad_ckpt=bool(CFG.N2V_GRAD_CKPT))

    start_ep, hist, best_val, bad = 0, [], float('inf'), 0

    if os.path.isfile(resume_path) and not force:
        try:
            st = _load_torch_file(resume_path, map_location="cpu")
            if st.get("cfg") != cfg_now:
                print("  resume file was written under different settings "
                      "(batch may have been re-tuned); starting fresh")
            else:
                model.load_state_dict(st["model"])
                opt.load_state_dict(st["opt"])
                _optimizer_to(opt, DEVICE)
                if st.get("sched") is not None:
                    sched.load_state_dict(st["sched"])
                if st.get("scaler") is not None:
                    scaler.load_state_dict(st["scaler"])
                if ema is not None and st.get("ema") is not None:
                    ema.load_state_dict(st["ema"])
                _restore_rng(st)
                start_ep = int(st.get("epoch", 0))
                hist = list(st.get("hist", []))
                best_val = float(st.get("best_val", float('inf')))
                bad = int(st.get("bad", 0))
                last = f"{hist[-1]['val']:.5f}" if hist else "n/a"
                print(f"  resumed {arch_name} N2V after epoch {start_ep}/{epochs} "
                      f"(last val {last})")
        except Exception as e:
            print(f"  could not read N2V resume file ({e}); starting fresh")
            start_ep, hist, best_val, bad = 0, [], float('inf'), 0

    def save_resume(ep_done):
        _atomic_save(dict(
            model=model.state_dict(), opt=opt.state_dict(),
            scaler=scaler.state_dict(), sched=sched.state_dict(),
            ema=ema.state_dict() if ema is not None else None,
            epoch=ep_done, hist=hist, best_val=best_val, bad=bad, cfg=cfg_now,
            rng_torch=torch.get_rng_state(),
            rng_cuda=(torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else None),
            rng_np=np.random.get_state(), rng_py=random.getstate(),
        ), resume_path)

    def save_best(ep, val):
        _atomic_save(dict(
            model=(ema.state_dict() if ema is not None else model.state_dict()),
            raw=model.state_dict(), arch=arch_name, epoch=ep,
            val_loss=val, hist=hist, cfg=cfg_now), final_path + ".best")

    ep = start_ep
    try:
        for ep in range(start_ep + 1, epochs + 1):
            model.train()
            losses, seen, n_oom, t0 = [], 0, 0, time.time()

            for x_in, x_gt, m in tqdm(tr_dl, leave=False,
                                      desc=f"{arch_name} N2V ep{ep}/{epochs}"):
                x_in, x_gt, m = to_dev(x_in), to_dev(x_gt), to_dev(m)
                li = _n2v_step(model, x_in, x_gt, m, opt, scaler, ema)
                if li is None:
                    n_oom += 1
                    continue
                if math.isfinite(li):
                    losses.append(li)
                seen += x_in.shape[0]

            sched.step()
            train_loss = float(np.mean(losses)) if losses else float('nan')

            eval_model = copy_for_eval(model, ema) if ema is not None else model
            val_loss = n2v_validate(eval_model, va_dl)
            psnr, gauss = n2v_psnr_probe(eval_model, probe_dl)
            # the eval copy is a second full set of parameters on the card
            globals().pop('_EVAL_CACHE', None)
            del eval_model
            free_gpu()

            hist.append(dict(epoch=ep, train=train_loss, val=val_loss,
                             psnr=psnr, gauss_psnr=gauss, n_oom=n_oom,
                             lr=opt.param_groups[0]['lr'], batch=batch,
                             img_per_s=seen / max(time.time() - t0, 1e-6)))

            print(f"  {arch_name} N2V ep {ep:02d}  train {train_loss:.5f}  "
                  f"val {val_loss:.5f}  PSNR {psnr:.2f} dB "
                  f"(gauss {gauss:.2f}, gain {psnr - gauss:+.2f})  "
                  f"[{hist[-1]['img_per_s']:.0f} img/s"
                  + (f", {n_oom} OOM" if n_oom else "") + "]", flush=True)

            if n_oom:
                print(f"  WARNING: {n_oom} batches skipped on OOM. Set "
                      f"CFG.N2V_BATCH_OVERRIDE['{arch_name}'] = {max(2, batch // 2)} "
                      f"and rerun rather than living with it.")

            if math.isfinite(val_loss) and val_loss < best_val - 1e-7:
                best_val, bad = val_loss, 0
                save_best(ep, val_loss)
            else:
                bad += 1

            save_resume(ep)

            if STOP_REQUESTED["flag"] or time_is_short():
                print(f"  walltime nearly up; {arch_name} N2V state saved.")
                raise OutOfTime("walltime")

            if ep >= CFG.N2V_MIN_EPOCHS and bad >= CFG.N2V_PATIENCE:
                print(f"  early stop at ep {ep} (best val {best_val:.5f})")
                break

    except (KeyboardInterrupt, OutOfTime):
        save_resume(max(start_ep, ep - 1))
        raise

    if os.path.isfile(final_path + ".best"):
        st = _load_torch_file(final_path + ".best", map_location="cpu")
        _atomic_save(st["model"], final_path)
        _atomic_json({k: v for k, v in st.items() if k not in ("model", "raw")},
             str(json_dir(use_n2v=True) /
                 Path(final_path).name.replace(".pt", "_history.json")))
        os.remove(final_path + ".best")
        print(f"  saved {arch_name} N2V (epoch {st['epoch']}, "
              f"val {st['val_loss']:.5f}) -> {final_path}")
    else:
        _atomic_save(model.state_dict(), final_path)
        print(f"  saved {arch_name} N2V (last epoch) -> {final_path}")

    if os.path.isfile(resume_path):
        os.remove(resume_path)

    del model, opt, scaler, ema, tr_dl, va_dl, probe_dl
    free_gpu()
    return final_path

In [ ]:
# ---------------- run / reuse N2V (all architectures) ----------------
RUN_N2V   = True
N2V_ARCHS = list(CFG.ARCHS)              # ["AtomSegNet", "UNetPP", "HRNet"]

# Weights that already exist outside the results tree. These get copied into
# the configured results tree once, so the run treats them as finished and every
# artifact ends up under one root.
LEGACY_N2V_SOURCES = {}
_external_unetpp = os.environ.get("TEM_LEGACY_N2V_UNETPP")
if _external_unetpp:
    LEGACY_N2V_SOURCES["UNetPP"] = _external_unetpp

# Canonical filenames inside N2V_ROOT (do not point these outside the tree).
N2V_FILES = {"AtomSegNet": "atomsegnet_n2v.pt",
             "UNetPP":     "unetpp_n2v.pt",
             "HRNet":      "hrnet_n2v.pt"}


def import_external_n2v(sources=None, verbose=True):
    """Copy named checkpoints into N2V_ROOT under their canonical filenames."""
    sources = LEGACY_N2V_SOURCES if sources is None else sources
    for arch, src in sources.items():
        src = Path(src)
        dst = N2V_ROOT / N2V_FILES[arch]
        if dst.is_file():
            continue
        if not src.is_file():
            if verbose:
                print(f"  {arch}: {src} not found, will train instead")
            continue
        shutil.copy2(src, dst)
        for sib in (src.with_suffix(src.suffix + ".resume"),
                    src.with_suffix(src.suffix + ".best"),
                    src.parent / src.name.replace(".pt", "_history.json")):
            if sib.is_file():
                tgt = (N2V_ROOT / sib.name if sib.suffix == ".json"
                       else N2V_ROOT / (N2V_FILES[arch] + sib.suffix))
                if not tgt.is_file():
                    shutil.copy2(sib, tgt)
        if verbose:
            print(f"  {arch}: imported {src} -> {dst}")


def n2v_history(path):
    """History JSON moved into the JSON folder; older runs wrote it beside the
    weights. Check both."""
    name = Path(path).name.replace(".pt", "_history.json")
    for cand in (json_dir(use_n2v=True) / name, Path(path).parent / name):
        if cand.is_file():
            try:
                with open(cand) as f:
                    return json.load(f), cand
            except Exception:
                pass
    return None, None


print("=== N2V ARTEFACTS ===")
import_external_n2v()
adopt_legacy_n2v()

n2v_ckpts, n2v_failed = {}, {}
if common and RUN_N2V:
    for _arch in N2V_ARCHS:
        try:
            n2v_ckpts[_arch] = pretrain_n2v(_arch, force=False)
        except (KeyboardInterrupt, OutOfTime):
            # State is already checkpointed by pretrain_n2v; stop the whole loop
            # rather than burning the remaining walltime on the next arch.
            print(f"\ninterrupted during {_arch}; resume by rerunning this cell.")
            raise
        except Exception as _e:
            n2v_failed[_arch] = f"{_e.__class__.__name__}: {_e}"
            print(f"\n{_arch} N2V failed ({n2v_failed[_arch]}); continuing.",
                  flush=True)
            free_gpu()

print("\n=== N2V CHECKPOINTS ===")
_summary = {}
for _arch in N2V_ARCHS:
    _path = n2v_ckpts.get(_arch)
    if _path is None:
        _why = n2v_failed.get(_arch, "not run")
        print(f"{_arch:12s} -> ({_why})")
        _summary[_arch] = {"status": "failed", "detail": _why}
        continue

    _d, _hpath = n2v_history(_path)
    _tag, _rec = "", {"status": "ok", "checkpoint": str(_path)}
    if _d:
        try:
            _last = _d["hist"][-1]
            _p, _g = float(_last["psnr"]), float(_last["gauss_psnr"])
            _gain = _p - _g
            _tag = (f"  | best ep {_d['epoch']}, val {float(_d['val_loss']):.5f}, "
                    f"PSNR {_p:.2f} dB vs gauss {_g:.2f} ({_gain:+.2f})")
            _rec.update(epoch=_d["epoch"], val_loss=float(_d["val_loss"]),
                        psnr=_p, gauss_psnr=_g, psnr_gain=_gain,
                        epochs_run=len(_d["hist"]),
                        history=str(_hpath))
        except (KeyError, IndexError, TypeError, ValueError):
            _tag = "  | history present but unreadable"
            _rec["status"] = "ok_no_metrics"
    else:
        _tag = "  | no history (imported or last-epoch save)"
        _rec["status"] = "ok_no_history"
    _summary[_arch] = _rec
    print(f"{_arch:12s} -> {_path}{_tag}")

with open(results_file("n2v_summary.json", use_n2v=True), "w") as _f:
    json.dump({"dataset_hash": getattr(CFG, "DATASET_HASH", None),
               "n2v_root": str(N2V_ROOT),
               "train_n": len(N2V_TRAIN), "val_n": len(N2V_VAL),
               "archs": _summary}, _f, indent=2)
print(f"\nsummary -> {results_file('n2v_summary.json', use_n2v=True)}")

## 9. 5-fold cross-validation

Two conditions:

A. the architecture from ordinary initialization (`no_n2v`)
B. the same architecture warm-started from its own N2V weights (`with_n2v`)

Each condition keeps its own directory, checkpoints, resume files and results
JSON, so the two never contaminate each other. Rerunning this cell is safe:
completed folds skip and an in-flight fold resumes from its `.resume`
checkpoint. Nothing is ever force-deleted.

Micro-batches were chosen for a 40 GB A100. On a smaller card the first fold
hits CUDA OOM, and this section halves only that architecture's micro-batch and
retries, keeping the effective batch fixed with gradient accumulation. To skip
the wasted first attempt, lower `SUPERVISED_BATCH_START` directly.

In [ ]:
# ============================================================
# 9a. CV configuration, guards, checkpoint index, legacy ingest
# ============================================================
RUN_NO_N2V   = True
RUN_WITH_N2V = True
RUN_ARCHS    = list(CFG.ARCHS)          # ["AtomSegNet", "UNetPP", "HRNet"]

# Windows loads batches in the main process; see the note in section 1.
CFG.NUM_WORKERS = 0 if os.name == "nt" else 6

# Hard GPU guard. The single local GPU is logical cuda:0.
if os.environ.get("CUDA_VISIBLE_DEVICES") not in ("0", None, ""):
    raise RuntimeError("Restart the kernel and set CUDA_VISIBLE_DEVICES='0' "
                       "before importing torch (section 0 does this).")
if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError(f"Expected exactly one visible CUDA GPU, got "
                       f"available={torch.cuda.is_available()}, "
                       f"count={torch.cuda.device_count()}.")

SUPERVISED_BATCH_START = {"AtomSegNet": 32, "UNetPP": 16, "HRNet": 8}
SUPERVISED_BATCH_MIN = 4
TARGET_EFFECTIVE_BATCH = 32   # maintained approximately with grad accumulation
EVAL_BATCH_MAX = 4            # validation is deliberately conservative
EXPECTED_PAIRED_SAMPLES = 14364

if len(common) != EXPECTED_PAIRED_SAMPLES:
    print(f"NOTE: {len(common)} paired samples, expected "
          f"{EXPECTED_PAIRED_SAMPLES}. Fold membership will differ from any "
          f"earlier run, so ingested legacy records describe different splits.")

# ---------------- condition directories ----------------
#   <RESULTS_ROOT>/without n2v/{AtomSegNet,HRNet,UNetPP}
#   <RESULTS_ROOT>/with n2v/{AtomSegNet,HRNet,UNetPP,N2V}
#   <RESULTS_ROOT>/<variant>/JSON
CONDITION_USE_N2V = {"no_n2v": False, "with_n2v": True}
CONDITION_DIRS = {k: str(variant_root(v)) for k, v in CONDITION_USE_N2V.items()}
NO_N2V_DIR, WITH_N2V_DIR = CONDITION_DIRS["no_n2v"], CONDITION_DIRS["with_n2v"]
PARALLEL_ROOT = str(CFG.RESULTS_ROOT)   # where the combined comparison CSV lands

# Directories from earlier runs, searched read-only for weights and logs.
LEGACY_CV_ROOTS = [p for p in (os.environ.get("TEM_LEGACY_CV_ROOT"),
                               str(CFG.PROJECT_ROOT),
                               str(Path(CFG.PROJECT_ROOT) / "atomsegnet"))
                   if p and os.path.isdir(p)]
LEGACY_COND_SUBDIRS = {"with_n2v": ["WITH N2v", "with n2v", "with_n2v", "WITH_N2V"],
                       "no_n2v":   ["without n2v", "WITHOUT N2V", "without_n2v",
                                    "no n2v", "no_n2v"]}

# Fold id anywhere in the stem: fold3, fold_3, fold-03, f3.
FOLD_RE = re.compile(r"(?:fold|^f)[_\-]?(\d{1,2})", re.IGNORECASE)
# Sidecars that are not fold weights.
NON_FOLD_HINTS = ("_n2v", "n2v_", "ema_only", "optimizer")


def cond_key(use_n2v):
    return "with_n2v" if variant_key(use_n2v) == "with_n2v" else "no_n2v"


def cond_dir(condition):
    return CONDITION_DIRS[cond_key(condition) if isinstance(condition, bool)
                          else condition]


def _ck(condition):
    return cond_key(condition) if isinstance(condition, bool) else condition


def _cond_of_dir(out_dir):
    """Reverse lookup so helpers that only receive out_dir know the condition."""
    r = os.path.normcase(os.path.abspath(out_dir))
    for k, d in CONDITION_DIRS.items():
        if os.path.normcase(os.path.abspath(d)) == r:
            return k
    return None


def ckpt_path(arch, fold, condition):
    """Canonical location for NEW checkpoints:
    <variant>\\<Arch>\\<Arch>_fold<k>.pt"""
    k = _ck(condition)
    return arch_dir(arch, CONDITION_USE_N2V[k]) / f"{arch}_fold{int(fold)}.pt"


def _search_roots(condition, arch):
    """Everywhere a fold checkpoint for this condition might live."""
    k = _ck(condition)
    use = CONDITION_USE_N2V[k]
    roots = [arch_dir(arch, use, create=False), Path(cond_dir(k)),
             Path(cond_dir(k)) / arch]
    for r in LEGACY_CV_ROOTS:
        for sub in LEGACY_COND_SUBDIRS[k]:
            roots += [Path(r) / sub, Path(r) / sub / arch]
    return roots


def scan_condition_ckpts(condition, archs=None, verbose=False):
    """Index every fold checkpoint for a condition, whatever it is named.
    Returns ({(arch, fold): path}, [unmatched paths]). Canonical locations are
    scanned first, so they win over legacy copies of the same fold."""
    archs = archs or RUN_ARCHS
    found, unmatched, seen = {}, [], set()
    for arch in archs:
        for root in _search_roots(condition, arch):
            if not root.is_dir():
                continue
            for p in sorted(root.glob("*.pt")):
                rp = p.resolve()
                if rp in seen:
                    continue
                seen.add(rp)
                stem = p.stem
                low = stem.lower()
                if any(h in low for h in NON_FOLD_HINTS):
                    continue
                if arch.lower() not in low and root.name != arch:
                    continue
                m = FOLD_RE.search(stem)
                if not m:
                    unmatched.append(p)
                    continue
                fold = int(m.group(1))
                found.setdefault((arch, fold), p)
    if verbose and unmatched:
        print(f"  {condition}: {len(unmatched)} .pt file(s) had no fold id in "
              f"the name and were ignored:")
        for p in unmatched[:6]:
            print(f"      {p}")
    return found, unmatched


# Built once here; refresh with rebuild_ckpt_index() after training.
CKPT_INDEX = {}


def rebuild_ckpt_index(archs=None, verbose=False):
    global CKPT_INDEX
    CKPT_INDEX = {k: scan_condition_ckpts(k, archs, verbose)[0]
                  for k in CONDITION_DIRS}
    return CKPT_INDEX


def find_fold_ckpt(arch, fold, condition):
    """Indexed lookup, falling back to a fresh scan if the index is cold."""
    k = _ck(condition)
    if not CKPT_INDEX:
        rebuild_ckpt_index()
    hit = CKPT_INDEX.get(k, {}).get((arch, int(fold)))
    if hit is not None and Path(hit).is_file():
        return Path(hit)
    p = ckpt_path(arch, fold, k)
    return p if p.is_file() else None


def fold_base_on_disk(condition, archs=None):
    """0- or 1-based fold ids, read off the filenames actually present."""
    idx = CKPT_INDEX.get(_ck(condition)) or scan_condition_ckpts(condition, archs)[0]
    folds = sorted({f for _, f in idx})
    if not folds:
        return 0
    return 1 if (min(folds) >= 1 and max(folds) >= CFG.N_FOLDS) else 0


def migrate_legacy_cv(conditions=("with_n2v", "no_n2v"), archs=None,
                      verbose=True):
    """Copy fold weights (and sidecars) from outside the tree into the canonical
    layout. Copies, never moves; an existing destination always wins. Files
    already inside the tree are left where they are."""
    archs = archs or RUN_ARCHS
    tree = os.path.normcase(os.path.abspath(str(CFG.RESULTS_ROOT)))
    moved = []
    for k in conditions:
        idx, _ = scan_condition_ckpts(k, archs)
        for (arch, fold), src in sorted(idx.items()):
            if os.path.normcase(os.path.abspath(str(src))).startswith(tree):
                continue                      # already in the results tree
            dst = ckpt_path(arch, fold, k)
            if dst.is_file():
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            for suf in (".resume", ".best"):
                s = Path(str(src) + suf)
                if s.is_file() and not Path(str(dst) + suf).is_file():
                    shutil.copy2(s, Path(str(dst) + suf))
            moved.append((k, arch, fold, str(src)))
    if verbose:
        for k, arch, fold, src in moved:
            print(f"  migrated {k} {arch} fold {fold} from {src}")
        if not moved:
            print("  nothing to migrate (all checkpoints already in the tree)")
    return moved


# ---------------- legacy shared results log ----------------
# An earlier run wrote every architecture's fold records into ONE file at the
# old CV root (UNetPP_cv_results.json despite the name): it holds AtomSegNet,
# UNetPP and HRNet rows together. run_cv() reads only the per-condition partial
# log, so those records were invisible and folds retrained even though their
# weights were on disk. The ingest below copies matching rows across. It is
# read-only with respect to the legacy log, and destination records always win.
LEGACY_RESULTS_JSON = next(
    (os.path.join(r, "UNetPP_cv_results.json") for r in LEGACY_CV_ROOTS
     if os.path.isfile(os.path.join(r, "UNetPP_cv_results.json"))),
    os.path.join(LEGACY_CV_ROOTS[0] if LEGACY_CV_ROOTS else str(CFG.RESULTS_ROOT),
                 "UNetPP_cv_results.json"))
INGEST_LEGACY_RECORDS = True


def _read_json_rows(path):
    """Record dicts in `path`, tolerating a bare list or a dict wrapping one."""
    if not os.path.isfile(path):
        return []
    try:
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
    except Exception as e:
        print(f"  could not read {path} ({e})")
        return []
    if isinstance(raw, list):
        rows = raw
    elif isinstance(raw, dict):
        rows = next((v for v in raw.values() if isinstance(v, list)), [])
    else:
        rows = []
    return [r for r in rows if isinstance(r, dict)]


def _results_json(out_dir):
    """Partial CV log lives in the variant's JSON folder. Falls back to out_dir
    for any caller passing a directory outside the tree."""
    k = _cond_of_dir(out_dir)
    if k is None:
        return os.path.join(out_dir, "cv_results_partial.json")
    return str(json_dir(CONDITION_USE_N2V[k]) / "cv_results_partial.json")


def _load_partial(out_dir):
    path = _results_json(out_dir)
    if os.path.isfile(path):
        try:
            with open(path, encoding="utf-8") as f:
                rows = json.load(f)
            return rows if isinstance(rows, list) else []
        except Exception as e:
            print(f"could not read {path} ({e}); starting empty")
    return []


def _is_cuda_oom(exc):
    """Stricter than _oom(): only a CUDA out-of-memory error triggers the
    micro-batch backoff below."""
    if isinstance(exc, torch.cuda.OutOfMemoryError):
        return True
    msg = str(exc).lower()
    return "cuda" in msg and "out of memory" in msg


def ingest_legacy_records(condition, out_dir, archs,
                          src=LEGACY_RESULTS_JSON, require_ckpt=True):
    """Seed the condition's partial log from the shared legacy log. A record is
    ingested only when its fold checkpoint actually exists: a record without
    weights would make run_cv() skip a fold that has no model to show for it."""
    src_rows = _read_json_rows(src)
    if not src_rows:
        return []

    dst_path = _results_json(out_dir)
    dst_rows = _read_json_rows(dst_path)
    have = {(r.get("arch"), int(r.get("fold", -1))) for r in dst_rows
            if r.get("arch") is not None and r.get("fold") is not None}

    added, skipped_no_ckpt = [], []
    for r in src_rows:
        arch, fold, cond = r.get("arch"), r.get("fold"), r.get("condition")
        if arch not in archs or cond != condition or fold is None:
            continue
        fold = int(fold)
        if (arch, fold) in have:
            continue
        found = find_fold_ckpt(arch, fold, condition)
        if require_ckpt and found is None:
            skipped_no_ckpt.append((arch, fold))
            continue
        row = dict(r, fold=fold, ingested_from=os.path.basename(src),
                   checkpoint=str(found) if found else None)
        dst_rows.append(row)
        have.add((arch, fold))
        added.append((arch, fold))

    if skipped_no_ckpt:
        print(f"  legacy ingest: {len(skipped_no_ckpt)} record(s) have no matching "
              f"checkpoint and were NOT ingested: {skipped_no_ckpt}")
    if added:
        if os.path.isfile(dst_path):
            shutil.copy2(dst_path, f"{dst_path}.bak{int(time.time())}")
        dst_rows.sort(key=lambda r: (str(r.get("arch")), int(r.get("fold", 0))))
        _atomic_json(dst_rows, dst_path)
        print(f"  legacy ingest: added {len(added)} record(s) {added} -> {dst_path}")
    return added


# ---------------- N2V warm-start checkpoints ----------------
def _resolve_n2v(name):
    """Section 8 writes into N2V_ROOT. Absolute paths pass through; bare names
    are looked up there first, then in the legacy roots."""
    if os.path.isabs(name) or re.match(r'^[A-Za-z]:[\\/]', str(name)):
        return str(name)
    roots = [str(N2V_ROOT), str(variant_root(True)), CFG.OUT_DIR] + LEGACY_CV_ROOTS
    for root in roots:
        cand = os.path.join(root, name)
        if os.path.isfile(cand):
            return cand
    return str(N2V_ROOT / name)          # reported MISSING by the pre-flight


N2V_CKPTS = {a: _resolve_n2v(N2V_FILES.get(a, f"{a.lower()}_n2v.pt"))
             for a in RUN_ARCHS}

# ---------------- migration + status ----------------
print("Legacy roots :", LEGACY_CV_ROOTS or "(none found)")
migrate_legacy_cv()
rebuild_ckpt_index(verbose=True)

print("\nCheckpoint inventory:")
for _k in ("no_n2v", "with_n2v"):
    _idx = CKPT_INDEX.get(_k, {})
    _base = fold_base_on_disk(_k)
    print(f"\n  {_k}  ->  {CONDITION_DIRS[_k]}"
          f"   ({len(_idx)}/{len(RUN_ARCHS) * CFG.N_FOLDS} fold ckpts, "
          f"{_base}-based ids)")
    for _a in RUN_ARCHS:
        _folds = sorted(f for (arch, f) in _idx if arch == _a)
        if _folds:
            _where = {str(Path(_idx[(_a, f)]).parent) for f in _folds}
            print(f"    {_a:11s} folds {_folds}")
            for _w in sorted(_where):
                print(f"                in {_w}")
        else:
            print(f"    {_a:11s} none found")

_missing_cond = [k for k, on in (("no_n2v", RUN_NO_N2V), ("with_n2v", RUN_WITH_N2V))
                 if on and not CKPT_INDEX.get(k)]
if _missing_cond:
    print(f"\n  {_missing_cond} enabled but empty. If you know the files exist, "
          f"list them with:\n"
          f"    list(Path(cond_dir('no_n2v')).rglob('*.pt'))\n"
          f"  and adjust FOLD_RE or LEGACY_COND_SUBDIRS.")

print("\nN2V warm starts:")
for _a, _p in N2V_CKPTS.items():
    print(f"  {_a:11s} {'OK     ' if os.path.isfile(_p) else 'MISSING'} {_p}")
if RUN_WITH_N2V and not all(os.path.isfile(p) for p in N2V_CKPTS.values()):
    print("  WARNING: a with-N2V run without warm-start weights is just the "
          "no-N2V condition under a different label. Run section 8 first.")

## 9d. Benchmark summary

Reads the shared results log at `CV_ROOT`, because a cross-architecture
comparison needs every architecture's folds in one frame and the per-condition
file under `WITH N2v` holds UNet++ only. Another kernel may still be appending
rows to it; this cell only ever reads.

In [ ]:
# ============================================================
# 9d. Benchmark summary (ingest -> backfill -> compare)
# ============================================================
SUMMARY_CONDITIONS = [k for k, on in (("no_n2v", RUN_NO_N2V),
                                      ("with_n2v", RUN_WITH_N2V)) if on] or ["with_n2v"]
METRICS = ["psnr", "ssim", "iou"]
INGEST_GLOBS = ["*_results.json", "cv_results*.json", "*_cv_results.json"]
SKIP_PAT = ("_history", "benchmark_", ".bak")
BACKFILL, RECOMPUTE_ALL = True, False
BF_THR, BF_N, BF_BATCH = 0.5, 512, 16
RANK = {"cv_log": 2, "backfill_eval": 1}


def _fold_names(fold, base):
    kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, va) in enumerate(kf.split(common)):
        if k == int(fold) - base:
            return [common[i] for i in va]


def load_model_weights(path, model, strict=False):
    """Accepts a bare state dict or any of the wrapper keys run_cv() writes."""
    obj = _load_torch_file(str(path), map_location="cpu")
    sd = obj
    if isinstance(obj, dict):
        for k in ("model", "state_dict", "ema", "raw", "weights"):
            v = obj.get(k)
            if isinstance(v, dict) and v and isinstance(next(iter(v.values())), torch.Tensor):
                sd = v
                break
    sd = {k: v for k, v in sd.items() if isinstance(v, torch.Tensor)}
    for pre in ("module.", "_orig_mod."):
        if sd and all(k.startswith(pre) for k in sd):
            sd = {k[len(pre):]: v for k, v in sd.items()}
    tgt = getattr(getattr(model, "_orig_mod", model), "module",
                  getattr(model, "_orig_mod", model))
    res = tgt.load_state_dict(sd, strict=strict)
    if len(getattr(res, "missing_keys", [])) > 0.5 * len(tgt.state_dict()):
        raise RuntimeError(f"{Path(path).name}: wrong architecture for this file")


@torch.inference_mode()
def eval_ckpt(arch, ckpt, names, thr=BF_THR, n=BF_N, desc=""):
    """PSNR/SSIM/IoU on a seeded subsample of one fold's validation split, so
    every architecture is measured on the same images."""
    from skimage.metrics import structural_similarity as ssim
    if n and len(names) > n:
        names = [names[i] for i in sorted(random.Random(SEED).sample(range(len(names)), n))]
    dl = DataLoader(TEMSegDataset(names, noisy_map, clean_map, mask_map,
                                  gauss_map=gauss_map, train=False),
                    **loader_options(False, batch=BF_BATCH, eval_mode=True))
    model = None
    try:
        model = prep_model(ARCH_REGISTRY[arch]())
        load_model_weights(ckpt, model)
        model.eval()
        P, S, I, G = [], [], [], []
        for noisy, clean, mask, _ in tqdm(dl, desc=desc, leave=False, unit="b"):
            with amp_autocast():
                den, lg = model(to_dev(noisy))
            den, cl_t = den.float().clamp(0, 1), to_dev(clean).float().clamp(0, 1)
            mse = ((den - cl_t) ** 2).mean(dim=(1, 2, 3))
            P += (10 * torch.log10(1 / mse.clamp_min(1e-12))).clamp_max(99).cpu().tolist()
            p, t = torch.sigmoid(lg.float()) >= thr, to_dev(mask) > .5
            u = (p | t).sum(dim=(1, 2, 3)).float()
            I += torch.where(u > 0, (p & t).sum(dim=(1, 2, 3)).float() / u.clamp_min(1),
                             torch.ones_like(u)).cpu().tolist()
            nz, cl, dn = noisy.numpy(), clean.numpy(), den.cpu().numpy()
            for j in range(dn.shape[0]):
                c = np.clip(cl[j, 0], 0, 1)
                S.append(float(ssim(c, dn[j, 0], data_range=1.0)))
                G.append(compute_psnr(c, np.clip(cv2.GaussianBlur(nz[j, 0], (0, 0), 1.0), 0, 1),
                                      data_range=1.0))
        return dict(psnr=float(np.mean(P)), ssim=float(np.mean(S)),
                    iou=float(np.mean(I)), gauss_psnr=float(np.mean(G)), n_val=len(I))
    finally:
        del model, dl
        free_gpu()


def collect(cond, verbose=True):
    """Every measured record for a condition, from the partial log plus any
    run_cv() output lying in the tree, then backfill whatever has weights but
    no metrics. Measured rows always beat backfilled ones."""
    dst = _results_json(cond_dir(cond))
    roots, files, seen = [json_dir(False), json_dir(True), variant_root(False),
                          variant_root(True), CFG.RESULTS_ROOT], [Path(dst)], set()
    roots += [Path(p) for p in LEGACY_CV_ROOTS]
    for r in roots:
        for pat in INGEST_GLOBS:
            files += [f for f in Path(r).glob(pat) if not any(s in f.name for s in SKIP_PAT)]

    rows = {}
    for f in files:
        if not f.is_file() or f.resolve() in seen:
            continue
        seen.add(f.resolve())
        n = 0
        for r in _read_json_rows(f):
            a, fo, c = r.get("arch"), r.get("fold"), r.get("condition")
            if a is None or fo is None or c != cond:
                continue
            row = dict(r, fold=int(fo), metric_source=r.get("metric_source", "cv_log"),
                       source=str(f))
            k = (str(a), int(fo))
            if k not in rows or RANK.get(row["metric_source"], 0) > RANK.get(
                    rows[k]["metric_source"], 0):
                rows[k] = row
            n += 1
        if verbose and n:
            print(f"    {f.name}: {n} record(s)")

    if BACKFILL:
        if not CKPT_INDEX:
            rebuild_ckpt_index()
        base = fold_base_on_disk(cond)
        todo = [(a, f, p) for (a, f), p in sorted(CKPT_INDEX.get(cond, {}).items())
                if a in RUN_ARCHS and (RECOMPUTE_ALL
                                       or RANK.get(rows.get((a, f), {}).get("metric_source"), 0) < 1)]
        for i, (a, f, p) in enumerate(todo, 1):
            names = _fold_names(f, base)
            if not names:
                continue
            print(f"    [{i}/{len(todo)}] backfill {a} fold {f}...", flush=True)
            try:
                m = eval_ckpt(a, p, names, desc=f"{a} f{f}")
            except Exception as e:
                print(f"        {e.__class__.__name__}: {e}", flush=True)
                free_gpu()
                continue
            rows[(a, f)] = dict(arch=a, fold=f, condition=cond, ckpt=str(p),
                                metric_source="backfill_eval", eval_n=m["n_val"],
                                mask_threshold=BF_THR, source="backfill", **m)
            print(f"        PSNR {m['psnr']:.2f}  SSIM {m['ssim']:.3f}  "
                  f"IoU {m['iou']:.4f}", flush=True)

    if rows:
        _atomic_json(sorted(rows.values(), key=lambda r: (r["arch"], r["fold"])), dst)
    df = pd.DataFrame(list(rows.values()))
    if df.empty:
        return df
    for c in METRICS + ["gauss_psnr", "epochs_run", "time", "fold"]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=[m for m in METRICS if m in df] + ["fold"])
    if df.empty:
        return df
    df["fold"] = df["fold"].astype(int)
    order = [a for a in CFG.ARCHS if a in set(df["arch"])]
    df["arch"] = pd.Categorical(df["arch"], categories=order, ordered=True)
    return df


def summarize(df):
    """Per-architecture means. params_M instantiates each model once; two
    architectures with an identical count are the same model twice."""
    ms = [m for m in METRICS if m in df]
    par = {}
    for n in df["arch"].cat.categories:
        try:
            mo = ARCH_REGISTRY[n]()
            par[n] = sum(p.numel() for p in mo.parameters()) / 1e6
            del mo
        except Exception:
            pass
    if len(par) > 1 and len(set(round(v, 3) for v in par.values())) < len(par):
        print(f"  WARNING: identical parameter counts {par} — check ARCH_REGISTRY")
    out = []
    for a, g in df.groupby("arch", observed=True):
        r = {"arch": str(a), "folds": g["fold"].nunique(),
             "params_M": round(par.get(str(a), float("nan")), 2)}
        for m in ms:
            r[m] = g[m].mean()
            r[f"{m}_sd"] = g[m].std(ddof=1) if len(g) > 1 else float("nan")
        if "gauss_psnr" in g:
            r["gain_dB"] = r.get("psnr", np.nan) - g["gauss_psnr"].mean()
        out.append(r)
    return pd.DataFrame(out).sort_values("iou" if "iou" in ms else ms[0],
                                         ascending=False).reset_index(drop=True)


def show(s):
    f = pd.DataFrame({"arch": s["arch"], "M": s["params_M"].map("{:.2f}".format)})
    for m, lab, p in (("psnr", "PSNR", 2), ("ssim", "SSIM", 3), ("iou", "IoU", 3)):
        if m in s:
            f[lab] = [f"{v:.{p}f}" + ("" if pd.isna(d) else f" ± {d:.{p}f}")
                      for v, d in zip(s[m], s[f"{m}_sd"])]
    if "gain_dB" in s:
        f["vs Gauss"] = s["gain_dB"].map("{:+.2f}".format)
    f["n"] = s["folds"]
    print(f.to_string(index=False))


def paired(df, key, groups, label):
    """Folds are seeded, so anything sharing a fold id is paired. Only folds
    present on both sides are tested."""
    out = []
    for a, b, tag in groups:
        pair = pd.concat([a.set_index("fold")[key], b.set_index("fold")[key]],
                         axis=1, join="inner").dropna()
        if len(pair) < 2:
            print(f"  {tag}: {len(pair)} shared fold(s), no test")
            continue
        x, y = pair.iloc[:, 0], pair.iloc[:, 1]
        d = y - x
        _, p = stats.ttest_rel(y, x)
        print(f"  {tag}: {x.mean():.4f} -> {y.mean():.4f}   d = {d.mean():+.4f} "
              f"± {d.std(ddof=1):.4f}   wins {int((d > 0).sum())}/{len(d)}   p = {p:.3f}"
              + ("" if len(d) == CFG.N_FOLDS else f"  [{len(d)}/{CFG.N_FOLDS}]"))
        out.append(dict(kind=label, metric=key, pair=tag, delta=float(d.mean()),
                        sd=float(d.std(ddof=1)), n=len(d), p=float(p)))
    return out


def figure(df, png, suffix=""):
    panels = [m for m in METRICS if m in df]
    fig, ax = plt.subplots(2, len(panels), figsize=(5 * len(panels), 8), squeeze=False)
    order = list(df["arch"].cat.categories)
    for a, m in zip(ax[0], panels):
        try:
            sns.barplot(data=df, x="arch", y=m, order=order, ax=a, errorbar="sd",
                        capsize=.15, alpha=.75)
        except TypeError:
            sns.barplot(data=df, x="arch", y=m, order=order, ax=a, ci="sd",
                        capsize=.15, alpha=.75)
        sns.stripplot(data=df, x="arch", y=m, order=order, ax=a, color="k", size=5, jitter=.08)
        a.set_title(f"{m.upper()}{suffix}", fontsize=10)
        a.set_xlabel("")
    if "gauss_psnr" in df and panels[0] == "psnr":
        ax[0, 0].axhline(df["gauss_psnr"].mean(), ls="--", c="gray", lw=1)
    for a, m in zip(ax[1], panels):
        p = df.pivot_table(index="fold", columns="arch", values=m, observed=True)
        for c in order:
            if c in p:
                a.plot(p.index, p[c], marker="o", ms=5, label=c)
        a.set_xlabel("fold")
        a.set_title(f"{m.upper()} per fold", fontsize=10)
    ax[1, 0].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(png, dpi=200)
    plt.show()


# ---------------- run ----------------
frames, sums, tests = {}, {}, []
for cond in SUMMARY_CONDITIONS:
    print(f"\n=== {cond} ===")
    df = collect(cond)
    if df.empty:
        print("  nothing usable")
        continue
    frames[cond] = df
    sums[cond] = s = summarize(df)
    print()
    show(s)
    for _, r in s.iterrows():
        if r.get("psnr_sd", 0) > 2 or r.get("gain_dB", 0) < 0:
            g = df[df["arch"].astype(str) == r["arch"]].nsmallest(2, "psnr")
            print(f"  ! {r['arch']}: denoiser unstable (sd {r.get('psnr_sd', 0):.2f} dB, "
                  f"gain {r.get('gain_dB', 0):+.2f} dB); worst fold(s) "
                  + ", ".join(f"{int(f)} ({p:.2f})" for f, p in
                              zip(g["fold"], g["psnr"])))
    best = str(s.iloc[0]["arch"])
    print(f"\n  vs {best} (IoU):")
    tests += [dict(t, condition=cond) for t in paired(
        df, "iou", [(df[df["arch"].astype(str) == o], df[df["arch"].astype(str) == best],
                     f"{best} vs {o}") for o in s["arch"][1:]], "arch")]
    figure(df, variant_root(CONDITION_USE_N2V[cond]) / "benchmark_bars.png", f" [{cond}]")

if not frames:
    raise RuntimeError("No results in any condition.")

if len(frames) == 2:
    print("\n=== N2V effect (paired by architecture and fold) ===")
    A, B = frames["no_n2v"], frames["with_n2v"]
    for m in METRICS:
        tests += paired(A, m, [(A[A["arch"].astype(str) == a], B[B["arch"].astype(str) == a],
                                f"{a:11s} {m}")
                               for a in CFG.ARCHS
                               if a in set(A["arch"].astype(str))
                               and a in set(B["arch"].astype(str))], "n2v")

pd.concat([f.assign(arch=f["arch"].astype(str)) for f in frames.values()],
          ignore_index=True).to_csv(CFG.RESULTS_ROOT / "cv_results_all.csv", index=False)
pd.concat([s.assign(condition=c) for c, s in sums.items()],
          ignore_index=True).round(4).to_csv(
    CFG.RESULTS_ROOT / "benchmark_comparison.csv", index=False)
mixed = {c: sorted(set(f["metric_source"])) for c, f in frames.items()}
with open(CFG.RESULTS_ROOT / "benchmark_summary.json", "w") as fh:
    json.dump(dict(dataset_hash=getattr(CFG, "DATASET_HASH", None), n_folds=CFG.N_FOLDS,
                   metric_sources=mixed, tests=tests), fh, indent=2, default=float)

if len({tuple(v) for v in mixed.values()}) > 1:
    print(f"\nWARNING: conditions measured differently ({mixed}); set "
          f"RECOMPUTE_ALL = True for a like-for-like N2V delta.")
print(f"\n-> {CFG.RESULTS_ROOT}\\cv_results_all.csv, benchmark_comparison.csv, "
      f"benchmark_summary.json")

## 9e. Qualitative panels on held-out images

The validation split is reconstructed with the same seeded `KFold` as section 9.
A different split here would show the model images it trained on.

In [ ]:
def _ck(condition):
    """Normalize a condition to its key. Defined in 9a; redefined here because
    a loop variable of the same name clobbered it."""
    return cond_key(condition) if isinstance(condition, bool) else condition

In [ ]:
# ============================================================
# 9e. Output panels on held-out validation images (all folds, archs, conditions)
# ============================================================
PANEL_ARCHS      = list(CFG.ARCHS)
PANEL_CONDITIONS = [k for k, on in (("no_n2v", RUN_NO_N2V),
                                    ("with_n2v", RUN_WITH_N2V)) if on] or ["with_n2v"]
PANEL_FOLDS      = "all"      # "all", "best", or an explicit list e.g. [1, 3]
N_SHOW           = 2          # samples per fold ("all" folds x 5 = a lot of figures)
MASK_THRESHOLD   = 0.5
SAVE_PANELS      = True       # <variant>\<Arch>\panels\fold_XX
SHOW_PANELS      = False     # False = save only; set True for a few folds
MAKE_MOSAIC      = True       # one cross-fold comparison figure per arch
FOLD_BASE        = None       # None = infer from the records


def read_cv_results(condition):
    for p in (json_dir(CONDITION_USE_N2V[condition]) / "cv_results.csv",
              Path(_results_json(cond_dir(condition))),
              Path(cond_dir(condition)) / "cv_results.csv"):
        if not p.is_file():
            continue
        df = (pd.read_csv(p) if p.suffix == ".csv"
              else pd.DataFrame(_read_json_rows(p)))
        if df.empty:
            continue
        if "condition" in df.columns:
            df = df[df["condition"] == condition]
        if not df.empty:
            return df, p
    return pd.DataFrame(), None


def infer_fold_base(results):
    """0-based in current run_cv() output, 1-based in the older logs. The wrong
    base reconstructs a split the model trained on, so read it off the data."""
    if FOLD_BASE is not None:
        return FOLD_BASE
    folds = pd.to_numeric(results.get("fold"), errors="coerce").dropna()
    if folds.empty:
        return 0
    return 1 if (int(folds.min()) >= 1 and int(folds.max()) >= CFG.N_FOLDS) else 0


def fold_val_names(fold, base):
    """Must match section 9 exactly: same KFold arguments, same `common` order."""
    kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, va_idx) in enumerate(kf.split(common)):
        if k == int(fold) - base:
            return [common[i] for i in va_idx]
    return None


def arch_folds(results, arch, condition, which=PANEL_FOLDS):
    """Every fold for one architecture that has both a usable record and a
    checkpoint, ranked by IoU. Returns [(fold, row, ckpt), ...]."""
    sub = results[results["arch"].astype(str) == arch].copy()
    if sub.empty:
        return [], f"no records for {arch}"
    for c in ("iou", "fold"):
        if c not in sub.columns:
            return [], f"{arch} records have no '{c}' column"
        sub[c] = pd.to_numeric(sub[c], errors="coerce")
    dropped = sorted(sub.loc[sub["iou"].isna(), "fold"].dropna().astype(int).tolist())
    if dropped:
        print(f"    folds {dropped} have no recorded IoU, excluded")
    sub = sub.dropna(subset=["iou", "fold"])
    if sub.empty:
        return [], f"every {arch} record is missing its IoU"

    sub["fold"] = sub["fold"].astype(int)
    sub = sub.sort_values("iou", ascending=False).reset_index(drop=True)
    if which == "best":
        sub = sub.head(1)
    elif isinstance(which, (list, tuple, set)):
        sub = sub[sub["fold"].isin({int(f) for f in which})]

    out, missing = [], []
    for _, row in sub.iterrows():
        fold = int(row["fold"])
        rec = row.get("ckpt") or row.get("checkpoint")
        ck = (Path(rec) if isinstance(rec, str) and Path(rec).is_file()
              else find_fold_ckpt(arch, fold, condition))
        if ck is None or not Path(ck).is_file():
            missing.append(fold)
            continue
        out.append((fold, row, Path(ck)))
    if missing:
        print(f"    folds {sorted(missing)} have records but no checkpoint")
    return (out, None) if out else ([], f"no {arch} fold has both a record and weights")


def prepare_sample(name):
    noisy = norm01(imread_gray(noisy_map[name]))
    clean = norm01(imread_gray(clean_map[name])) if name in clean_map else noisy.copy()
    mask = (binarize(imread_gray(mask_map[name])) if name in mask_map
            else np.zeros_like(noisy, np.float32))
    noisy = center_or_resize(noisy, CFG.IMG_SIZE, cv2.INTER_AREA)
    clean = center_or_resize(clean, CFG.IMG_SIZE, cv2.INTER_AREA)
    mask = (center_or_resize(mask, CFG.IMG_SIZE, cv2.INTER_NEAREST) > 0.5).astype(np.float32)
    return (np.ascontiguousarray(noisy, np.float32),
            np.ascontiguousarray(clean, np.float32),
            np.ascontiguousarray(mask, np.float32))


@torch.inference_mode()
def predict(model, noisy):
    x = torch.from_numpy(noisy)[None, None].float().to(
        DEVICE, non_blocking=True, memory_format=torch.channels_last)
    den, lg = model(x)
    prob = torch.sigmoid(lg)[0, 0].float().cpu().numpy()
    return (den[0, 0].float().cpu().numpy(), prob,
            (prob >= MASK_THRESHOLD).astype(np.float32))


def sample_iou(t, p):
    t, p = t.astype(bool), p.astype(bool)
    u = np.logical_or(t, p).sum()
    return 1.0 if u == 0 else float(np.logical_and(t, p).sum() / u)


def sample_psnr(clean, pred):
    """Per-image PSNR on the [0,1] scale each image was normalized to."""
    mse = np.mean((np.clip(clean, 0, 1) - np.clip(pred, 0, 1)) ** 2)
    return float("inf") if mse <= 1e-12 else float(10.0 * np.log10(1.0 / mse))


def overlay_rgb(noisy, mask):
    o = np.stack([noisy] * 3, axis=-1)
    o[..., 0] = np.maximum(o[..., 0], mask)          # atoms in red
    o[..., 1] *= (1.0 - 0.45 * mask)
    o[..., 2] *= (1.0 - 0.45 * mask)
    return o


def _imshow(ax, im, title):
    ax.imshow(im, cmap=None if im.ndim == 3 else "gray",
              vmin=None if im.ndim == 3 else 0, vmax=None if im.ndim == 3 else 1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def make_panels(arch, cond, fold, model, names, out_dir):
    """Fixed sample picks per fold, seeded, so with-N2V and without-N2V panels
    show the same images."""
    picks = random.Random(SEED).sample(names, min(N_SHOW, len(names)))
    rows, cache = [], {}
    for nm in picks:
        noisy, clean, gt = prepare_sample(nm)
        den, _, pred = predict(model, noisy)
        iou, psnr = sample_iou(gt, pred), sample_psnr(clean, den)
        rows.append(dict(arch=arch, condition=cond, fold=fold, sample=nm,
                         iou=iou, psnr=psnr))
        cache[nm] = (noisy, clean, den, gt, pred, iou, psnr)

        fig, axes = plt.subplots(1, 6, figsize=(20, 4))
        for ax, im, t in zip(axes,
                             [noisy, clean, den, gt, pred, overlay_rgb(noisy, pred)],
                             ["Noisy", "Clean GT", f"Denoised\n{psnr:.2f} dB",
                              "Atom mask GT", f"Predicted\nIoU {iou:.3f}", "Overlay"]):
            _imshow(ax, im, t)
        fig.suptitle(f"{arch} ({cond}) | fold {fold} | {nm}", fontsize=13)
        plt.tight_layout()
        if SAVE_PANELS:
            plt.savefig(out_dir / f"{arch}_fold{fold}_{nm}.png", dpi=150,
                        bbox_inches="tight")
        plt.show() if SHOW_PANELS else plt.close(fig)
    return rows, cache


def mosaic(arch, cond, per_fold, out_png):
    """One row per fold, same sample in every row: fold-to-fold variation at a
    glance. Uses the first sample each fold has in common."""
    shared = set.intersection(*(set(c) for _, c in per_fold)) if per_fold else set()
    if not shared:
        return None
    nm = sorted(shared)[0]
    fig, axes = plt.subplots(len(per_fold), 4, squeeze=False,
                             figsize=(13, 3.2 * len(per_fold)))
    for r, (fold, cache) in enumerate(per_fold):
        noisy, clean, den, gt, pred, iou, psnr = cache[nm]
        for ax, im, t in zip(axes[r],
                             [noisy, den, pred, overlay_rgb(noisy, pred)],
                             [f"fold {fold}: noisy", f"denoised {psnr:.2f} dB",
                              f"pred IoU {iou:.3f}", "overlay"]):
            _imshow(ax, im, t)
    fig.suptitle(f"{arch} ({cond}) across folds | {nm}", fontsize=13)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show() if SHOW_PANELS else plt.close(fig)
    return out_png


# ---------------- run ----------------
# ---------------- run ----------------
# Loop variables are prefixed pnl_ so they cannot shadow the section-9a
# helpers (_ck, _cond_of_dir, ...) that live in the same global namespace.
panel_stats, panel_skipped = [], []

for pnl_cond in PANEL_CONDITIONS:
    print(f"\n=== {pnl_cond} ===")
    pnl_res, pnl_src = read_cv_results(pnl_cond)
    if pnl_res.empty or "arch" not in pnl_res.columns:
        print("  no usable results; skipping")
        panel_skipped.append((pnl_cond, "*", "no results"))
        continue
    pnl_base = infer_fold_base(pnl_res)
    print(f"  source: {pnl_src}  ({pnl_base}-based fold ids)")

    for pnl_arch in PANEL_ARCHS:
        print(f"\n  --- {pnl_arch} ---")
        pnl_folds, pnl_err = arch_folds(pnl_res, pnl_arch, pnl_cond)
        if pnl_err:
            print(f"    skipped: {pnl_err}")
            panel_skipped.append((pnl_cond, pnl_arch, pnl_err))
            continue
        print(f"    {len(pnl_folds)} fold(s): " +
              ", ".join(f"{f} (IoU {float(r['iou']):.4f})" for f, r, _ in pnl_folds))

        pnl_out = arch_dir(pnl_arch, CONDITION_USE_N2V[pnl_cond]) / "panels"
        pnl_per_fold = []
        # Sort on the fold number only; tuple ordering would fall through to
        # comparing pandas Series if two folds ever shared a number.
        for pnl_fold, pnl_row, pnl_ckpt in sorted(pnl_folds, key=lambda t: t[0]):
            pnl_names = fold_val_names(pnl_fold, pnl_base)
            if not pnl_names:
                print(f"    fold {pnl_fold}: split could not be reconstructed")
                panel_skipped.append((pnl_cond, pnl_arch, f"fold {pnl_fold} split"))
                continue
            pnl_model = None
            try:
                pnl_model = prep_model(ARCH_REGISTRY[pnl_arch]())
                load_model_weights(pnl_ckpt, pnl_model)
                pnl_model.eval()
                pnl_fd = pnl_out / f"fold_{pnl_fold:02d}"
                pnl_fd.mkdir(parents=True, exist_ok=True)
                pnl_rows, pnl_cache = make_panels(pnl_arch, pnl_cond, pnl_fold,
                                                  pnl_model, pnl_names, pnl_fd)
                panel_stats += pnl_rows
                pnl_per_fold.append((pnl_fold, pnl_cache))
                print(f"    fold {pnl_fold}: {len(pnl_rows)} panel(s) -> {pnl_fd.name}")
            except Exception as exc:
                print(f"    fold {pnl_fold} failed: {exc.__class__.__name__}: {exc}")
                panel_skipped.append((pnl_cond, pnl_arch,
                                      f"fold {pnl_fold}: {exc.__class__.__name__}: {exc}"))
            finally:
                if pnl_model is not None:
                    del pnl_model
                free_gpu()

        if MAKE_MOSAIC and len(pnl_per_fold) > 1:
            pnl_m = mosaic(pnl_arch, pnl_cond, pnl_per_fold,
                           pnl_out / f"{pnl_arch}_folds_mosaic.png")
            print(f"    mosaic -> {pnl_m.name}" if pnl_m else
                  "    mosaic skipped: folds are disjoint splits with no shared sample")

# ---------------- log ----------------
if panel_stats:
    pnl_df = pd.DataFrame(panel_stats)
    print("\n=== per-sample panel metrics ===")
    print(pnl_df.groupby(["condition", "arch", "fold"])[["psnr", "iou"]]
          .agg(["mean", "count"]).round(4).to_string())
    pnl_df.round(6).to_csv(CFG.RESULTS_ROOT / "panel_samples.csv", index=False)
    print(f"\n-> {CFG.RESULTS_ROOT / 'panel_samples.csv'}")
else:
    print("\nno panels were produced")
if panel_skipped:
    print("\nskipped:")
    for pnl_c, pnl_a, pnl_w in panel_skipped:
        print(f"  {pnl_c:9s} {pnl_a:11s} {pnl_w}")
# ---------------- log ----------------
if panel_stats:
    _pdf = pd.DataFrame(panel_stats)
    print("\n=== per-sample panel metrics ===")
    print(_pdf.groupby(["condition", "arch", "fold"])[["psnr", "iou"]]
          .mean().round(4).to_string())
    _pdf.round(6).to_csv(CFG.RESULTS_ROOT / "panel_samples.csv", index=False)
    print(f"\n-> {CFG.RESULTS_ROOT / 'panel_samples.csv'}")
if panel_skipped:
    print("\nskipped:")
    for _c, _a, _w in panel_skipped:
        print(f"  {_c:9s} {_a:11s} {_w}")

In [ ]:
# ============================================================
# 9f. Inline segmentation panels — best fold per architecture
# ============================================================
BEST_CONDITION = "no_n2v"     # or "with_n2v"
BEST_ARCHS     = list(CFG.ARCHS)
BEST_N_SHOW    = 3            # images shown per architecture
BEST_THRESHOLD = 0.5


def best_fold_of(arch, condition):
    """Highest recorded IoU among folds that also have weights on disk."""
    res, _ = read_cv_results(condition)
    if res.empty or "arch" not in res.columns:
        return None
    sub = res[res["arch"].astype(str) == arch].copy()
    for c in ("iou", "fold"):
        sub[c] = pd.to_numeric(sub.get(c), errors="coerce")
    sub = sub.dropna(subset=["iou", "fold"]).sort_values("iou", ascending=False)
    base = infer_fold_base(res)
    for _, r in sub.iterrows():
        fold = int(r["fold"])
        rec = r.get("ckpt") or r.get("checkpoint")
        ck = (Path(rec) if isinstance(rec, str) and Path(rec).is_file()
              else find_fold_ckpt(arch, fold, condition))
        if ck is not None and Path(ck).is_file():
            return dict(fold=fold, iou=float(r["iou"]), ckpt=Path(ck), base=base)
    return None


def show_best_panels(condition=BEST_CONDITION, archs=BEST_ARCHS, n=BEST_N_SHOW):
    for arch in archs:
        info = best_fold_of(arch, condition)
        if info is None:
            print(f"{arch}: no fold with both a record and weights in {condition}\n")
            continue
        names = fold_val_names(info["fold"], info["base"])
        if not names:
            print(f"{arch}: fold {info['fold']} split could not be reconstructed\n")
            continue
        picks = random.Random(SEED).sample(names, min(n, len(names)))
        print(f"{arch}  |  {condition}  |  best fold {info['fold']} "
              f"(recorded IoU {info['iou']:.4f})  |  {info['ckpt'].name}")

        model = None
        try:
            model = prep_model(ARCH_REGISTRY[arch]())
            load_model_weights(info["ckpt"], model)
            model.eval()
            for nm in picks:
                noisy, clean, gt = prepare_sample(nm)
                x = torch.from_numpy(noisy)[None, None].float().to(
                    DEVICE, memory_format=torch.channels_last)
                with torch.inference_mode():
                    den, lg = model(x)
                den = den[0, 0].float().cpu().numpy()
                prob = torch.sigmoid(lg)[0, 0].float().cpu().numpy()
                pred = (prob >= BEST_THRESHOLD).astype(np.float32)
                iou = sample_iou(gt, pred)
                psnr = sample_psnr(clean, den)

                # False colour: green = correct, red = false positive,
                # yellow = missed. Errors are the point of the figure.
                err = np.stack([noisy] * 3, axis=-1)
                tp, fp, fn = (pred > .5) & (gt > .5), (pred > .5) & (gt < .5), (pred < .5) & (gt > .5)
                err[tp] = [0.0, 0.9, 0.4]
                err[fp] = [0.9, 0.2, 0.0]
                err[fn] = [1.0, 0.9, 0.0]

                fig, ax = plt.subplots(1, 6, figsize=(21, 3.9))
                for a, im, t in zip(
                        ax, [noisy, clean, den, gt, prob, err],
                        ["Noisy input", "Clean GT", f"Denoised · {psnr:.2f} dB",
                         "Atom mask GT", "Predicted probability",
                         f"Errors · IoU {iou:.3f}"]):
                    if t.startswith("Predicted prob"):
                        a.imshow(im, cmap="magma", vmin=0, vmax=1)
                    else:
                        a.imshow(im, cmap=None if im.ndim == 3 else "gray",
                                 vmin=None if im.ndim == 3 else 0,
                                 vmax=None if im.ndim == 3 else 1)
                    a.set_title(t, fontsize=10)
                    a.axis("off")
                fig.suptitle(f"{arch} · {condition} · fold {info['fold']} · {nm}   "
                             f"(green correct, red false positive, yellow missed)",
                             fontsize=12)
                plt.tight_layout()
                plt.show()
        finally:
            if model is not None:
                del model
            free_gpu()
        print()


show_best_panels()

In [ ]:
# ============================================================
# 9f. Inline segmentation panels — best fold per architecture
# ============================================================
BEST_CONDITION = "with_n2v"     # or "with_n2v"
BEST_ARCHS     = list(CFG.ARCHS)
BEST_N_SHOW    = 3            # images shown per architecture
BEST_THRESHOLD = 0.5


def best_fold_of(arch, condition):
    """Highest recorded IoU among folds that also have weights on disk."""
    res, _ = read_cv_results(condition)
    if res.empty or "arch" not in res.columns:
        return None
    sub = res[res["arch"].astype(str) == arch].copy()
    for c in ("iou", "fold"):
        sub[c] = pd.to_numeric(sub.get(c), errors="coerce")
    sub = sub.dropna(subset=["iou", "fold"]).sort_values("iou", ascending=False)
    base = infer_fold_base(res)
    for _, r in sub.iterrows():
        fold = int(r["fold"])
        rec = r.get("ckpt") or r.get("checkpoint")
        ck = (Path(rec) if isinstance(rec, str) and Path(rec).is_file()
              else find_fold_ckpt(arch, fold, condition))
        if ck is not None and Path(ck).is_file():
            return dict(fold=fold, iou=float(r["iou"]), ckpt=Path(ck), base=base)
    return None


def show_best_panels(condition=BEST_CONDITION, archs=BEST_ARCHS, n=BEST_N_SHOW):
    for arch in archs:
        info = best_fold_of(arch, condition)
        if info is None:
            print(f"{arch}: no fold with both a record and weights in {condition}\n")
            continue
        names = fold_val_names(info["fold"], info["base"])
        if not names:
            print(f"{arch}: fold {info['fold']} split could not be reconstructed\n")
            continue
        picks = random.Random(SEED).sample(names, min(n, len(names)))
        print(f"{arch}  |  {condition}  |  best fold {info['fold']} "
              f"(recorded IoU {info['iou']:.4f})  |  {info['ckpt'].name}")

        model = None
        try:
            model = prep_model(ARCH_REGISTRY[arch]())
            load_model_weights(info["ckpt"], model)
            model.eval()
            for nm in picks:
                noisy, clean, gt = prepare_sample(nm)
                x = torch.from_numpy(noisy)[None, None].float().to(
                    DEVICE, memory_format=torch.channels_last)
                with torch.inference_mode():
                    den, lg = model(x)
                den = den[0, 0].float().cpu().numpy()
                prob = torch.sigmoid(lg)[0, 0].float().cpu().numpy()
                pred = (prob >= BEST_THRESHOLD).astype(np.float32)
                iou = sample_iou(gt, pred)
                psnr = sample_psnr(clean, den)

                # False colour: green = correct, red = false positive,
                # yellow = missed. Errors are the point of the figure.
                err = np.stack([noisy] * 3, axis=-1)
                tp, fp, fn = (pred > .5) & (gt > .5), (pred > .5) & (gt < .5), (pred < .5) & (gt > .5)
                err[tp] = [0.0, 0.9, 0.4]
                err[fp] = [0.9, 0.2, 0.0]
                err[fn] = [1.0, 0.9, 0.0]

                fig, ax = plt.subplots(1, 6, figsize=(21, 3.9))
                for a, im, t in zip(
                        ax, [noisy, clean, den, gt, prob, err],
                        ["Noisy input", "Clean GT", f"Denoised · {psnr:.2f} dB",
                         "Atom mask GT", "Predicted probability",
                         f"Errors · IoU {iou:.3f}"]):
                    if t.startswith("Predicted prob"):
                        a.imshow(im, cmap="magma", vmin=0, vmax=1)
                    else:
                        a.imshow(im, cmap=None if im.ndim == 3 else "gray",
                                 vmin=None if im.ndim == 3 else 0,
                                 vmax=None if im.ndim == 3 else 1)
                    a.set_title(t, fontsize=10)
                    a.axis("off")
                fig.suptitle(f"{arch} · {condition} · fold {info['fold']} · {nm}   "
                             f"(green correct, red false positive, yellow missed)",
                             fontsize=12)
                plt.tight_layout()
                plt.show()
        finally:
            if model is not None:
                del model
            free_gpu()
        print()


show_best_panels()

## 10. Atomic-column localization and interatomic spacing

1. Inference gives a denoised image and an atom-mask probability.
2. `peak_local_max` on the mask gives integer-pixel atom centres.
3. Sub-pixel refinement: a bounded 2-D Gaussian fit on a 5x5 patch of the
   denoised image around each peak.
4. KD-tree nearest-neighbour distances give the lattice spacing in angstroms
   via `CFG.PIXEL_SIZE_A`.

`position/` in TEM-ImageNet-v1.3 stores unit-cell vectors, not per-atom
coordinates, so the ground truth here is the intensity-weighted blob centroid of
each `gaussianMask/` Gaussian. `circularMask/` is the coarse fallback.

In [ ]:
"""Replacement for section 10 of UNET_l5ocal.ipynb.

Paste this entire file into section 10, or run it there with %run -i.
This changes inference/postprocessing and evaluation; it does not retrain weights.

Differences from the previous version:
  * runs every fold that has a checkpoint, in every enabled condition, and
    aggregates across them instead of evaluating one preselected fold;
  * `_canonical_state_dict` no longer deletes a key component named "mod" at
    arbitrary depth, only wrapper prefixes;
  * empty per-record arrays no longer crash np.concatenate;
  * spacing rows no longer imply a detection_id when the counts disagree.
"""
from __future__ import annotations

import hashlib
import json
import os
import platform
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import least_squares, linear_sum_assignment
from scipy.spatial import cKDTree


@dataclass(frozen=True)
class LocalizationOptions:
    n_pool: int | None               # None evaluates the entire supplied split
    seed: int = 42
    batch_size: int = 4
    probability_threshold: float = 0.50  # freeze on development data before testing
    min_distance_px: float = 4.0
    min_component_area: int = 3
    gt_min_distance_px: float = 2.0
    gt_relative_threshold: float = 0.08
    match_tolerance_px: float = 3.0
    fit_half_width: int = 3
    fit_max_shift_px: float = 1.75
    fit_min_r2: float = 0.30
    fit_max_nrmse: float = 0.30
    fit_max_evaluations: int = 80
    fit_source: str = "denoised"          # "noisy" is a separate, fixed experiment
    polarity: str = "bright"             # set explicitly for another acquisition mode
    inference_precision: str = "fp32"    # FP32 by default for coordinate measurements
    quantize_like_training_cache: bool = True
    gaussian_sigma_px: float = 1.0
    pixel_size_A: float | None = None    # do not inherit the old 0.20 placeholder
    calibration_source: str = ""
    n_examples: int = 3                  # example figures PER FOLD
    dpi: int = 220
    algorithm_version: str = "localization-2.1"

    def __post_init__(self):
        for name in ("batch_size", "min_component_area", "fit_half_width",
                     "fit_max_evaluations", "n_examples", "dpi", "seed"):
            if not isinstance(getattr(self, name), (int, np.integer)):
                raise ValueError(f"{name} must be an integer.")
        if self.n_pool is not None and not isinstance(self.n_pool, (int, np.integer)):
            raise ValueError("n_pool must be an integer or None.")
        positive = (self.batch_size, self.min_distance_px, self.gt_min_distance_px,
                    self.match_tolerance_px, self.fit_max_shift_px,
                    self.fit_max_evaluations, self.gaussian_sigma_px, self.dpi)
        if any(not np.isfinite(x) or x <= 0 for x in positive):
            raise ValueError("Batch size, distances, tolerances and iteration limits must be positive.")
        if self.n_pool is not None and self.n_pool < 1:
            raise ValueError("n_pool must be positive or None.")
        if not 0 < self.probability_threshold < 1 or not 0 < self.gt_relative_threshold < 1:
            raise ValueError("Probability and GT thresholds must lie strictly between 0 and 1.")
        if self.fit_half_width < 2 or self.min_component_area < 1 or self.n_examples < 0:
            raise ValueError("Invalid fit window, component area, or example count.")
        if self.fit_source not in ("denoised", "noisy") or self.polarity not in ("bright", "dark"):
            raise ValueError("Invalid fit_source or polarity.")
        if (not np.isfinite(self.fit_min_r2) or not -1 <= self.fit_min_r2 <= 1
                or not np.isfinite(self.fit_max_nrmse) or self.fit_max_nrmse <= 0):
            raise ValueError("Invalid Gaussian-fit quality thresholds.")
        if self.inference_precision not in ("fp32", "fp16", "bf16"):
            raise ValueError("inference_precision must be fp32, fp16 or bf16.")
        if self.pixel_size_A is not None:
            if (not np.isfinite(self.pixel_size_A) or self.pixel_size_A <= 0
                    or not self.calibration_source.strip()):
                raise ValueError("An Angstrom scale requires a positive pixel size and calibration_source.")


# ---------------------------------------------------------------- primitives

def _points(values):
    a = np.asarray(values, dtype=np.float64)
    if a.size == 0:
        return np.empty((0, 2), dtype=np.float64)
    if a.ndim != 2 or a.shape[1] != 2 or not np.isfinite(a).all():
        raise ValueError("Coordinates must be a finite (N,2) array in row, column order.")
    return a


def _image(values):
    a = np.asarray(values, dtype=np.float64)
    if a.ndim != 2 or min(a.shape) < 1 or not np.isfinite(a).all():
        raise ValueError("Expected a nonempty finite 2-D image.")
    return a


def _cat(records, key, width=None):
    """Concatenate a per-record field, tolerating empty records and an empty list.

    np.concatenate([]) raises, and a frame with no detections contributes a
    zero-length array, so both cases have to be filtered before concatenating.
    """
    parts = [np.asarray(r[key]) for r in records if len(np.asarray(r[key]))]
    if parts:
        return np.concatenate(parts)
    return np.empty((0, width), dtype=float) if width else np.empty(0, dtype=float)


def read_gray_file(path):
    with Image.open(path) as im:
        if getattr(im, "n_frames", 1) != 1:
            raise ValueError(f"Select a frame explicitly from this image stack: {path}")
        arr = np.asarray(im.convert("F") if im.mode in ("RGB", "RGBA", "P") else im)
    return _image(arr)


def legacy_normalize(img):
    """Match the notebook's independent 1st/99th percentile normalization."""
    a = _image(img).astype(np.float32)
    lo, hi = np.percentile(a, (1, 99))
    return (np.clip((a - lo) / (hi - lo), 0, 1) if hi - lo >= 1e-6
            else np.zeros_like(a)).astype(np.float32)


def resize_image(img, size, binary=False):
    a = _image(img)
    if a.shape == (size, size):
        return a.astype(np.float32)
    if a.shape[0] != a.shape[1]:
        raise ValueError("Rectangular images must be tiled or cropped explicitly; "
                         "square resizing distorts distances.")
    try:
        import cv2
    except ImportError as exc:
        raise ImportError("OpenCV is required when resizing, to reproduce the "
                          "notebook's interpolation.") from exc
    return cv2.resize(a.astype(np.float32), (size, size),
                      interpolation=cv2.INTER_NEAREST if binary else cv2.INTER_AREA)


# ---------------------------------------------------------------- detection

def local_maxima(response, min_distance, threshold):
    """Collapse flat maxima before deterministic Euclidean nonmaximum suppression."""
    a = _image(response)
    candidate = (a == ndi.maximum_filter(a, size=3, mode="nearest")) & (a > threshold)
    if not candidate.any():
        return np.empty((0, 2), dtype=int)
    labels, n = ndi.label(candidate, structure=np.ones((3, 3)))
    peaks = []
    for k, sl in enumerate(ndi.find_objects(labels), 1):
        rr, cc = np.nonzero(labels[sl] == k)
        coords = np.column_stack((rr + sl[0].start, cc + sl[1].start))
        centre = coords.mean(axis=0)
        peaks.append(coords[np.argmin(np.sum((coords - centre) ** 2, axis=1))])
    pts = np.asarray(peaks, dtype=int)
    scores = a[pts[:, 0], pts[:, 1]]
    order = np.lexsort((pts[:, 1], pts[:, 0], -scores))
    tree, blocked, keep = cKDTree(pts), np.zeros(len(pts), dtype=bool), []
    for j in order:
        if not blocked[j]:
            keep.append(j)
            neighbours = tree.query_ball_point(pts[j], min_distance * (1 - 1e-12))
            blocked[neighbours] = True
    return pts[keep]


def detect_centres(probability, options):
    """Distance-transform peaks locate blob interiors instead of sigmoid ripples."""
    p = _image(probability)
    if p.min() < 0 or p.max() > 1:
        raise ValueError("Segmentation probabilities must lie in [0,1].")
    mask = p >= options.probability_threshold
    lab, n = ndi.label(mask, structure=np.ones((3, 3)))
    if n:
        sizes = np.bincount(lab.ravel())
        keep = sizes >= options.min_component_area
        keep[0] = False
        mask = keep[lab]
    # Padding makes objects touching an image border have a well-defined exterior.
    distance = ndi.distance_transform_edt(np.pad(mask, 1))[1:-1, 1:-1]
    return local_maxima(distance, options.min_distance_px, 0.0)


def _gaussian_model_jac(parameters, x, y):
    amp, dx, dy, sx, sy, bg, bx, by = parameters
    xx, yy = x - dx, y - dy
    e = np.exp(-0.5 * ((xx / sx) ** 2 + (yy / sy) ** 2))
    g = amp * e
    model = g + bg + bx * x + by * y
    jac = np.column_stack((e, g * xx / sx**2, g * yy / sy**2,
                           g * xx**2 / sx**3, g * yy**2 / sy**3,
                           np.ones_like(x), x, y))
    return model, jac


def refine_centres(img, peaks, options):
    """Robust elliptical-Gaussian plus sloping-background fits; failures stay counted."""
    a, centres = _image(img), _points(peaks)
    refined, widths = centres.copy(), np.full((len(centres), 2), np.nan)
    accepted = np.zeros(len(centres), dtype=bool)
    quality = np.full(len(centres), np.nan)
    statuses = np.full(len(centres), "not_fitted", dtype=object)
    if not len(centres):
        return refined, accepted, widths, statuses, quality
    neighbour = (cKDTree(centres).query(centres, k=2)[0][:, 1]
                 if len(centres) > 1 else np.full(1, np.inf))
    if options.polarity == "dark":
        a = 1.0 - a
    for i, (row, col) in enumerate(centres):
        h = options.fit_half_width
        if np.isfinite(neighbour[i]):
            h = max(2, min(h, int(np.floor(neighbour[i] / 2))))
        r, c = int(round(row)), int(round(col))
        if r - h < 0 or c - h < 0 or r + h >= a.shape[0] or c + h >= a.shape[1]:
            statuses[i] = "border"
            continue
        patch = a[r-h:r+h+1, c-h:c+h+1]
        span = float(np.ptp(patch))
        if span < 1e-5:
            statuses[i] = "flat_patch"
            continue
        border = np.concatenate((patch[0], patch[-1], patch[1:-1, 0], patch[1:-1, -1]))
        z = ((patch - np.median(border)) / span).ravel()
        yy, xx = np.mgrid[-h:h+1, -h:h+1]
        x, y = xx.ravel().astype(float), yy.ravel().astype(float)
        weight = np.maximum(z, 0)
        total = max(weight.sum(), 1e-12)
        shift_limit = min(options.fit_max_shift_px, h - 0.1)
        dx0 = np.clip(np.sum(weight * x) / total, -0.7 * shift_limit, 0.7 * shift_limit)
        dy0 = np.clip(np.sum(weight * y) / total, -0.7 * shift_limit, 0.7 * shift_limit)
        initial = [max(float(z.max()), 0.05), dx0, dy0, 1.0, 1.0, 0., 0., 0.]
        low = [0., -shift_limit, -shift_limit, .40, .40, -2., -.5, -.5]
        high = [4., shift_limit, shift_limit, h + .25, h + .25, 2., .5, .5]
        try:
            result = least_squares(
                lambda v: _gaussian_model_jac(v, x, y)[0] - z, initial,
                jac=lambda v: _gaussian_model_jac(v, x, y)[1],
                bounds=(low, high), loss="soft_l1", f_scale=.10,
                x_scale="jac", max_nfev=options.fit_max_evaluations,
                ftol=1e-6, xtol=1e-6, gtol=1e-6)
            v = result.x
            residual = _gaussian_model_jac(v, x, y)[0] - z
            nrmse = float(np.sqrt(np.mean(residual**2)))
            r2 = 1. - float(np.mean(residual**2)) / max(float(np.var(z)), 1e-12)
            quality[i] = r2
            on_bound = (abs(v[1]) >= .98 * shift_limit or abs(v[2]) >= .98 * shift_limit
                        or min(v[3:5]) <= .405 or max(v[3:5]) >= .99 * (h + .25))
            if not result.success:
                statuses[i] = "iteration_limit"
            elif on_bound:
                statuses[i] = "fit_at_bound"
            elif v[0] < .03 or r2 < options.fit_min_r2 or nrmse > options.fit_max_nrmse:
                statuses[i] = "poor_fit"
            else:
                refined[i] = (r + v[2], c + v[1])
                widths[i] = (v[4], v[3])
                accepted[i], statuses[i] = True, "accepted"
        except (ValueError, FloatingPointError, np.linalg.LinAlgError):
            statuses[i] = "numerical_failure"
    return refined, accepted, widths, statuses, quality


def marker_coordinates(marker, target_shape, options, binary=False):
    """Approximate marker-derived references, never asserted to be exact atom positions."""
    a = _image(marker)
    target_shape = tuple(target_shape)
    if len(target_shape) != 2 or min(target_shape) <= 0:
        raise ValueError("Target shape must be a positive (height, width).")
    if not np.isclose(a.shape[0] / a.shape[1], target_shape[0] / target_shape[1]):
        raise ValueError("Marker and target aspect ratios disagree; verify registration "
                         "before scaling coordinates.")
    if a.max() <= 0:
        return np.empty((0, 2), dtype=float)
    if a.min() < 0:
        raise ValueError("A coordinate marker image must be nonnegative.")
    source_scale = min(a.shape[0] / target_shape[0], a.shape[1] / target_shape[1])
    if binary:
        response = ndi.distance_transform_edt(np.pad(a > .5 * a.max(), 1))[1:-1, 1:-1]
        threshold = 0.
    else:
        response, threshold = a, options.gt_relative_threshold * float(a.max())
    peaks = local_maxima(response, options.gt_min_distance_px * source_scale, threshold)
    coords = peaks.astype(float)
    # A local quadratic on log intensity estimates a Gaussian centre without
    # merging every touching Gaussian into a single connected-component centroid.
    if not binary:
        yy, xx = np.mgrid[-1:2, -1:2]
        x, y = xx.ravel(), yy.ravel()
        design = np.column_stack((np.ones(9), x, y, x*x, x*y, y*y))
        for i, (r, c) in enumerate(peaks):
            if r < 1 or c < 1 or r >= a.shape[0]-1 or c >= a.shape[1]-1:
                continue
            patch = a[r-1:r+2, c-1:c+2]
            if patch.min() <= 0:
                continue
            coef = np.linalg.lstsq(design, np.log(patch.ravel()), rcond=None)[0]
            hessian = np.array([[2*coef[3], coef[4]], [coef[4], 2*coef[5]]])
            if np.linalg.eigvalsh(hessian).max() >= -1e-8:
                continue
            delta = -np.linalg.solve(hessian, coef[1:3])
            if np.isfinite(delta).all() and np.max(np.abs(delta)) <= .9:
                coords[i] += delta[::-1]
    # OpenCV resize uses pixel centres, including the half-pixel offset.
    return (coords + .5) * (np.asarray(target_shape) / np.asarray(a.shape)) - .5


def match_detections(det, gt, tol_px=3.0):
    """Maximum-cardinality gated assignment, then minimum total matched distance.

    None means missing references; an empty (0,2) array is a valid empty frame.
    All unmatched detections/references are returned, including zero-detection images.
    """
    det = _points(det)
    if gt is None:
        return None
    gt = _points(gt)
    if not np.isfinite(tol_px) or tol_px <= 0:
        raise ValueError("Matching tolerance must be positive.")
    ri = ci = np.empty(0, dtype=int)
    residuals = np.empty(0, dtype=float)
    if len(det) and len(gt):
        if len(det) * len(gt) > 8_000_000:
            raise ValueError("Too many candidate-reference pairs; inspect segmentation "
                             "coverage and thresholds.")
        distance = np.linalg.norm(det[:, None, :] - gt[None, :, :], axis=2)
        # One extra invalid edge costs more than every valid edge could cost.
        penalty = (min(len(det), len(gt)) + 1) * (tol_px + 1)
        cost = np.where(distance <= tol_px, distance, penalty)
        ri, ci = linear_sum_assignment(cost)
        valid = distance[ri, ci] <= tol_px
        ri, ci = ri[valid], ci[valid]
        residuals = distance[ri, ci]
    return dict(n_det=len(det), n_gt=len(gt), n_match=len(ri),
                det_indices=ri, gt_indices=ci, residuals_px=residuals,
                unmatched_det=np.setdiff1d(np.arange(len(det)), ri),
                unmatched_gt=np.setdiff1d(np.arange(len(gt)), ci))


def detection_metrics(tp, fp, fn):
    return dict(precision=tp / (tp + fp) if tp + fp else None,
                recall=tp / (tp + fn) if tp + fn else None,
                f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None)


def nn_spacings_px(coords_px):
    a = _points(coords_px)
    return cKDTree(a).query(a, k=2)[0][:, 1] if len(a) > 1 else np.empty(0)


# ---------------------------------------------------------------- small utils

def _psnr(pred, target):
    if target is None:
        return None
    mse = float(np.mean((_image(pred) - _image(target)) ** 2))
    return float(-10 * np.log10(mse)) if mse > 0 else float("inf")


def _rmse(x):
    a = np.asarray(x, dtype=float)
    return float(np.sqrt(np.mean(a*a))) if a.size else None


def _mean(x):
    a = np.asarray(x, dtype=float)
    return float(np.mean(a)) if a.size else None


def _json_safe(value):
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [_json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    return value


def write_json_atomic(value, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(_json_safe(value), indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


def _write_csv(frame, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _fmt(value, digits=3):
    if value is None or (isinstance(value, float) and pd.isna(value)) or pd.isna(value):
        return "n/a"
    if np.isinf(value):
        return "inf"
    return f"{value:.{digits}f}"


# ---------------------------------------------------------------- per frame

def evaluate_frame(name, noisy, clean, denoised, probability, gt, gt_source, options):
    noisy, denoised, probability = map(_image, (noisy, denoised, probability))
    if noisy.shape != denoised.shape or noisy.shape != probability.shape:
        raise ValueError(f"Input and model output shapes disagree for {name}.")
    if clean is not None and _image(clean).shape != noisy.shape:
        raise ValueError(f"Clean reference shape disagrees for {name}.")
    peaks = detect_centres(probability, options)
    fit_img = denoised if options.fit_source == "denoised" else noisy
    coords, accepted, widths, statuses, quality = refine_centres(fit_img, peaks, options)
    raw_match = match_detections(peaks, gt, options.match_tolerance_px)
    match = match_detections(coords, gt, options.match_tolerance_px)
    gaussian = ndi.gaussian_filter(noisy, options.gaussian_sigma_px, mode="reflect")
    row = dict(sample=name, gt_source=gt_source, reference_available=gt is not None,
               n_det=len(coords), n_gt=len(gt) if gt is not None else None,
               tp=match["n_match"] if match is not None else None,
               fp=len(coords)-match["n_match"] if match is not None else None,
               fn=len(gt)-match["n_match"] if match is not None else None,
               refine_success_rate=float(accepted.mean()) if len(accepted) else None,
               prediction_coverage=float(np.mean(probability >= options.probability_threshold)),
               psnr_noisy=_psnr(noisy, clean), psnr_gaussian=_psnr(gaussian, clean),
               psnr_denoised=_psnr(denoised, clean),
               rmse_px=_rmse(match["residuals_px"]) if match is not None else None)
    if match is not None:
        row.update(detection_metrics(row["tp"], row["fp"], row["fn"]))
    before = after = np.empty(0)
    if raw_match is not None and raw_match["n_match"]:
        ii, jj = raw_match["det_indices"], raw_match["gt_indices"]
        before = raw_match["residuals_px"]
        # Keep the same raw assignment even when refinement moves a point outside the gate.
        after = np.linalg.norm(coords[ii] - gt[jj], axis=1)
    return dict(row=row, noisy=noisy, clean=clean, den=denoised, gaussian=gaussian,
                prob=probability, peaks=peaks, coords=coords, accepted=accepted,
                widths=widths, statuses=statuses, quality=quality, gt=gt, match=match,
                raw_match=raw_match, paired_before=before, paired_after=after,
                spacing=nn_spacings_px(coords))


# ---------------------------------------------------------------- figures

def _save_figure(fig, stem, dpi, show=False):
    fig.savefig(str(stem) + ".png", dpi=dpi, facecolor="white")
    fig.savefig(str(stem) + ".pdf", facecolor="white")
    if show:
        plt.show()
    plt.close(fig)


def plot_example(record, title, stem, options, show=False):
    with plt.rc_context({"font.size": 10, "axes.titlesize": 11,
                         "figure.facecolor": "white", "axes.grid": False,
                         "pdf.fonttype": 42}):
        fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.4), layout="constrained")
        row = record["row"]
        panels = [(record["noisy"], "Noisy input", "psnr_noisy"),
                  (record["gaussian"], "Gaussian baseline", "psnr_gaussian"),
                  (record["den"], "Network denoising", "psnr_denoised"),
                  (record["clean"], "Clean reference", None)]
        for ax, (im, label, metric) in zip(axes.flat, panels):
            if im is None:
                ax.text(.5, .5, "Reference unavailable", ha="center", va="center",
                        transform=ax.transAxes)
            else:
                ax.imshow(im, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            suffix = (f"  |  {_fmt(row[metric], 2)} dB"
                      if metric and row[metric] is not None else "")
            ax.set_title(label + suffix)
        im = axes[1, 1].imshow(record["prob"], cmap="magma", vmin=0, vmax=1,
                               interpolation="nearest")
        axes[1, 1].set_title(f"Atom probability  |  threshold {options.probability_threshold:.2f}")
        fig.colorbar(im, ax=axes[1, 1], shrink=.75, label="Probability")
        ax = axes[1, 2]
        ax.imshow(record["noisy"], cmap="gray", vmin=0, vmax=1, interpolation="nearest")

        def dots(points, colour, label, marker="o", size=24):
            if len(points):
                ax.scatter(points[:, 1], points[:, 0], s=size, c=colour, marker=marker,
                           linewidths=.9, label=label)

        m, xy = record["match"], record["coords"]
        if m is None:
            dots(xy, "#56b4e9", "Detected centres")
        else:
            dots(xy[m["det_indices"]], "#009e73", "Matched detections")
            dots(xy[m["unmatched_det"]], "#d55e00", "Unmatched detections", "+", 38)
            dots(record["gt"][m["unmatched_gt"]], "#f0e442", "Missed references", "x", 30)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(loc="upper center", bbox_to_anchor=(.5, -.015), fontsize=8, frameon=False)
        ax.set_title("Centre matching" + (f"  |  F1 {_fmt(row.get('f1'))}" if m is not None else ""))
        for ax in axes.flat:
            ax.set_xticks([])
            ax.set_yticks([])
        fig.suptitle(title + "\n" + str(row["sample"]), fontsize=14, weight="bold")
        _save_figure(fig, stem, options.dpi, show)


def plot_overview(frame, records, summary, stem, options, show=False):
    colour = ("#7a8794", "#e69f00", "#0072b2")
    with plt.rc_context({"font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "pdf.fonttype": 42}):
        fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5), layout="constrained")
        ax = axes[0, 0]
        cols = ["psnr_noisy", "psnr_gaussian", "psnr_denoised"]
        vals = frame[cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        finite = np.isfinite(vals).all(axis=1)
        if finite.any():
            for values in vals[finite]:
                ax.plot(range(3), values, color="#a9b1b8", alpha=.25, lw=.7)
            for k, c in enumerate(colour):
                ax.scatter(np.full(finite.sum(), k), vals[finite, k], color=c, s=12, alpha=.45)
                mean = vals[finite, k].mean()
                ax.scatter(k, mean, s=85, color=c, edgecolor="white", zorder=5)
                ax.annotate(f"{mean:.2f}", (k, mean), xytext=(8, 0), textcoords="offset points")
        else:
            ax.text(.5, .5, "No finite paired PSNR values", transform=ax.transAxes, ha="center")
        ax.set_xticks(range(3), ["Noisy", "Gaussian", "Network"])
        ax.set_ylabel("PSNR (dB)")
        ax.set_title("Denoising on the same images")

        ax = axes[0, 1]
        metrics = [summary.get(k) for k in ("gt_precision", "gt_recall", "gt_f1")]
        ax.bar(range(3), [x if x is not None else 0 for x in metrics],
               color=["#0072b2", "#009e73", "#cc79a7"], width=.55)
        for i, v in enumerate(metrics):
            ax.text(i, (v or 0) + .03, _fmt(v), ha="center")
        ax.set_xticks(range(3), ["Precision", "Recall", "F1"])
        ax.set_ylim(0, 1.12)
        ax.set_title(f"Marker matching  |  gate {options.match_tolerance_px:g} px")
        ax.set_ylabel("Pooled detection score")

        ax = axes[1, 0]
        for key, label, c in (("paired_before", "Before refinement", "#7a8794"),
                              ("paired_after", "After refinement", "#0072b2")):
            v = _cat(records, key)
            if v.size:
                v = np.sort(v)
                ax.plot(v, np.arange(1, len(v)+1)/len(v),
                        label=f"{label}: RMSE {_fmt(_rmse(v))}", color=c)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
        else:
            ax.text(.5, .5, "No matched pairs", transform=ax.transAxes, ha="center")
        ax.set_xlabel("Position error (px)")
        ax.set_ylabel("Cumulative fraction")
        ax.set_title("Refinement on identical raw-matched pairs")

        ax = axes[1, 1]
        status_parts = [np.asarray(r["statuses"], dtype=object) for r in records
                        if len(r["statuses"])]
        statuses = np.concatenate(status_parts) if status_parts else np.empty(0, dtype=object)
        counts = pd.Series(statuses, dtype=object).value_counts()
        if len(counts):
            labels = [str(x).replace("_", " ") for x in counts.index]
            ax.barh(labels, counts.values,
                    color=["#009e73" if x == "accepted" else "#a9b1b8" for x in counts.index])
            for i, v in enumerate(counts.values):
                ax.text(v, i, f"  {v}", va="center", fontsize=9)
            ax.margins(x=.15)
        else:
            ax.text(.5, .5, "No detections", transform=ax.transAxes, ha="center")
        ax.set_xlabel("Detected centres")
        ax.set_title("Fit outcomes - failed fits remain in evaluation")

        fig.suptitle(f"{summary['arch']} · {summary.get('condition', '')} · "
                     f"fold {summary['best_fold']} · {len(frame)} validation images\n"
                     "Preliminary localization and paired denoising diagnostics",
                     fontsize=14, weight="bold")
        _save_figure(fig, stem, options.dpi, show)


def plot_geometry(records, stem, options, show=False):
    scale = options.pixel_size_A or 1.0
    unit = "A" if options.pixel_size_A is not None else "px"
    spacing = _cat(records, "spacing") * scale
    width_parts = [r["widths"][r["accepted"]] for r in records
                   if len(r["accepted"]) and r["accepted"].any()]
    widths = np.concatenate(width_parts) if width_parts else np.empty((0, 2))
    fwhm = (2 * np.sqrt(2 * np.log(2)) * np.sqrt(np.prod(widths, axis=1)) * scale
            if len(widths) else np.empty(0))
    with plt.rc_context({"font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "pdf.fonttype": 42}):
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), layout="constrained")
        for ax, values, title, label, colour in zip(
                axes, (spacing, fwhm),
                ("Nearest-neighbour distances", "Fitted feature widths"),
                (f"Detected-centre separation ({unit})", f"Apparent Gaussian FWHM ({unit})"),
                ("#0072b2", "#e69f00")):
            if values.size:
                ax.hist(values, bins=min(40, max(5, int(np.sqrt(values.size)))),
                        color=colour, alpha=.8)
                median = float(np.median(values))
                ax.axvline(median, color="#333333", lw=1.2, ls="--",
                           label=f"median {median:.2f} {unit}")
                ax.legend(fontsize=8)
            else:
                ax.text(.5, .5, "No eligible measurements", transform=ax.transAxes, ha="center")
            ax.set_title(f"{title}  |  n={values.size}")
            ax.set_xlabel(label)
            ax.set_ylabel("Count")
        fig.suptitle("Geometry diagnostics: spacing uses all detections; "
                     "widths use accepted fits", fontsize=12)
        _save_figure(fig, stem, options.dpi, show)


# ---------------------------------------------------------------- export

def export_localization(records, out_dir, options, metadata, show=False):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    arch = str(metadata.get("arch", "model"))
    frame = pd.DataFrame([r["row"] for r in records])
    if frame.empty:
        raise ValueError("No images were evaluated.")
    evaluated = frame[frame["reference_available"]]
    counts = evaluated[["tp", "fp", "fn"]].sum().astype(int).tolist() if len(evaluated) else [0, 0, 0]
    metric = detection_metrics(*counts)

    spacing = _cat(records, "spacing")
    residual_parts = [np.asarray(r["match"]["residuals_px"]) for r in records
                      if r["match"] is not None and len(r["match"]["residuals_px"])]
    residuals = np.concatenate(residual_parts) if residual_parts else np.empty(0)
    before, after = _cat(records, "paired_before"), _cat(records, "paired_after")
    accepted = _cat(records, "accepted").astype(bool)
    widths = _cat(records, "widths", width=2)
    fwhm = (2 * np.sqrt(2 * np.log(2)) * np.sqrt(np.prod(widths[accepted], axis=1))
            if len(widths) and accepted.any() else np.empty(0))

    summary = dict(metadata, options=asdict(options), n_samples=len(records),
                   n_samples_with_reference=len(evaluated), n_atoms=int(len(accepted)),
                   gt_n_det=int(counts[0]+counts[1]), gt_n_gt=int(counts[0]+counts[2]),
                   gt_n_match=int(counts[0]), gt_precision=metric["precision"],
                   gt_recall=metric["recall"], gt_f1=metric["f1"],
                   gt_rmse_px=_rmse(residuals), refine_success_rate=_mean(accepted),
                   spacing_px_mean=_mean(spacing),
                   spacing_px_median=float(np.median(spacing)) if spacing.size else None,
                   column_fwhm_px_mean=_mean(fwhm), pixel_size_A=options.pixel_size_A,
                   match_tol_px=options.match_tolerance_px,
                   paired_refinement=dict(
                       n=len(before), raw_rmse_px=_rmse(before), refined_rmse_px=_rmse(after),
                       pairing="Fixed raw-detection assignments; no post-refinement "
                               "distance censoring."),
                   reference_sources=frame["gt_source"].value_counts().to_dict())
    for metric_name in ("psnr_noisy", "psnr_gaussian", "psnr_denoised"):
        v = pd.to_numeric(frame[metric_name], errors="coerce").to_numpy(float)
        finite = v[np.isfinite(v)]
        summary[metric_name] = dict(mean=_mean(finite),
                                    std=float(finite.std(ddof=1)) if len(finite) > 1 else None,
                                    n_finite=len(finite),
                                    n_exact_matches=int(np.isposinf(v).sum()))
    scale = options.pixel_size_A
    summary.update(spacing_A_mean=_mean(spacing*scale) if scale else None,
                   spacing_A_std=float(np.std(spacing*scale)) if scale and spacing.size else None,
                   column_fwhm_A_mean=_mean(fwhm*scale) if scale else None,
                   gt_rmse_A=_rmse(residuals*scale) if scale else None)

    notes = ["The selected validation fold/checkpoint is not an independent final test.",
             "Marker-derived references are approximate; exact coordinates require a "
             "verified coordinate convention.",
             "Position RMSE is conditional on a successful gated match; interpret it "
             "with precision and recall.",
             "Nearest-neighbour spacing includes all detections and is a diagnostic, "
             "not a validated lattice parameter.",
             "Normalized-image PSNR does not measure preservation of original detector "
             "intensities.",
             "Settings were not tuned on the evaluated reference markers."]
    if str(metadata.get("condition", "")).lower() == "with_n2v":
        notes.append("N2V pretraining runs before the CV split; validation exposure "
                     "must be disclosed.")
    if scale is None:
        notes.append("No verified calibration supplied: all spatial measurements remain in pixels.")
    if (frame["prediction_coverage"] > .6).any():
        notes.append("Some masks cover more than 60% of the image; inspect "
                     "transfer/calibration before trusting centres.")
    summary["notes"] = notes

    atom_rows, match_rows, gt_rows, spacing_rows = [], [], [], []
    for rec in records:
        name = rec["row"]["sample"]
        for i, (xy, pk, sig) in enumerate(zip(rec["coords"], rec["peaks"], rec["widths"])):
            atom_rows.append(dict(sample=name, detection_id=i, y_px=xy[0], x_px=xy[1],
                                  peak_y_px=pk[0], peak_x_px=pk[1], refined=rec["accepted"][i],
                                  fit_status=rec["statuses"][i], fit_r2=rec["quality"][i],
                                  sigma_r_px=sig[0], sigma_c_px=sig[1]))
        if rec["gt"] is not None:
            for i, xy in enumerate(rec["gt"]):
                gt_rows.append(dict(sample=name, reference_id=i, y_px=xy[0], x_px=xy[1],
                                    source=rec["row"]["gt_source"]))
        if rec["match"] is not None:
            m = rec["match"]
            for i, j, e in zip(m["det_indices"], m["gt_indices"], m["residuals_px"]):
                match_rows.append(dict(sample=name, detection_id=i, reference_id=j, error_px=e))
        # nn_spacings_px returns nothing for a single detection, so detection_id
        # only means anything when the counts line up.
        sp = rec["spacing"]
        aligned = len(sp) == len(rec["coords"])
        for i, d in enumerate(sp):
            spacing_rows.append(dict(sample=name, detection_id=i if aligned else None,
                                     spacing_px=d, spacing_A=d*scale if scale else None))

    tables = {
        "per_image": frame,
        "localization_atoms": pd.DataFrame(atom_rows, columns=[
            "sample", "detection_id", "y_px", "x_px", "peak_y_px", "peak_x_px",
            "refined", "fit_status", "fit_r2", "sigma_r_px", "sigma_c_px"]),
        "reference_centres": pd.DataFrame(gt_rows, columns=[
            "sample", "reference_id", "y_px", "x_px", "source"]),
        "matched_centres": pd.DataFrame(match_rows, columns=[
            "sample", "detection_id", "reference_id", "error_px"]),
        "localization_spacings": pd.DataFrame(spacing_rows, columns=[
            "sample", "detection_id", "spacing_px", "spacing_A"])}
    for name, table in tables.items():
        _write_csv(table, out / f"{arch}_{name}.csv")

    plot_overview(frame, records, summary, out / f"{arch}_localization_overview",
                  options, show)
    plot_geometry(records, out / f"{arch}_geometry_diagnostics", options, show)
    for i, rec in enumerate(records[:options.n_examples], 1):
        plot_example(rec, f"{arch} · {summary.get('condition', '')} · "
                          f"fold {summary['best_fold']} · example {i}",
                     out / f"{arch}_example_{i:02d}", options, show)

    text = [f"# {arch}: localization diagnostics", "",
            f"Condition: {summary.get('condition', 'unknown')}  |  "
            f"fold: {summary['best_fold']}  |  images: {len(frame)}", "",
            "| Measurement | Value |", "|---|---|",
            f"| Detection precision / recall / F1 | {_fmt(metric['precision'])} / "
            f"{_fmt(metric['recall'])} / {_fmt(metric['f1'])} |",
            f"| Matched-position RMSE | {_fmt(summary['gt_rmse_px'])} px |",
            f"| Accepted Gaussian fits | {_fmt(summary['refine_success_rate'])} |",
            f"| Raw / refined RMSE on fixed pairs | {_fmt(_rmse(before))} / "
            f"{_fmt(_rmse(after))} px |", "",
            "## Denoising on the same selected images", "",
            "| Method | Mean finite PSNR (dB) | Finite images |", "|---|---:|---:|"]
    for key in ("psnr_noisy", "psnr_gaussian", "psnr_denoised"):
        v = summary[key]
        text.append(f"| {key.removeprefix('psnr_')} | {_fmt(v['mean'], 2)} | {v['n_finite']} |")
    text += ["", "## Interpretation", ""] + [f"- {n}" for n in notes]
    if any(summary[k]["n_exact_matches"] for k in ("psnr_noisy", "psnr_gaussian", "psnr_denoised")):
        text += ["- Exact image matches have infinite PSNR and are counted separately "
                 "in JSON; the table reports finite means."]
    text += ["", "## Figures", "", f"![Overview]({arch}_localization_overview.png)", "",
             f"![Geometry diagnostics]({arch}_geometry_diagnostics.png)", ""]
    text += [f"![Example {i}]({arch}_example_{i:02d}.png)"
             for i in range(1, min(options.n_examples, len(records))+1)]
    (out / f"{arch}_localization_report.md").write_text("\n".join(text)+"\n", encoding="utf-8")
    write_json_atomic(summary, out / f"{arch}_localization_summary.json")
    print(f"      {len(frame)} images, {summary['n_atoms']} detections, "
          f"{summary['gt_n_gt']} references | P {_fmt(metric['precision'])} "
          f"R {_fmt(metric['recall'])} F1 {_fmt(metric['f1'])} | "
          f"RMSE {_fmt(summary['gt_rmse_px'])} px | fits {_fmt(summary['refine_success_rate'])}")
    return summary, tables


# ---------------------------------------------------------------- data access

class NotebookSamples:
    """Read original files, avoiding the old cache's unverified preprocessing identity."""

    def __init__(self, namespace, options):
        self.ns, self.options = namespace, options
        self.size = int(namespace["CFG"].IMG_SIZE)
        self.exact = namespace.get("LOC_EXACT_COORDS", {})

    def load(self, name):
        ns, opt, s = self.ns, self.options, self.size
        raw = read_gray_file(ns["noisy_map"][name])
        noisy = resize_image(legacy_normalize(raw), s)
        clean = None
        path = ns.get("clean_map", {}).get(name)
        if path is not None:
            target = read_gray_file(path)
            if target.shape != raw.shape:
                raise ValueError(f"Noisy/clean image sizes differ for {name}; verify registration.")
            clean = resize_image(legacy_normalize(target), s)
        if opt.quantize_like_training_cache:
            def quantize(x):
                return np.floor(x*255+.5).clip(0, 255).astype(np.uint8).astype(np.float32)/255
            noisy = quantize(noisy)
            clean = quantize(clean) if clean is not None else None
        if name in self.exact:
            record = self.exact[name]
            if not isinstance(record, dict) or not {"coords_rc", "shape"} <= record.keys():
                raise ValueError("LOC_EXACT_COORDS entries need coords_rc and original "
                                 "shape; use zero-based pixel centres.")
            shape = np.asarray(record["shape"], dtype=float)
            if shape.shape != (2,) or not np.isfinite(shape).all() or np.any(shape <= 0):
                raise ValueError("Exact-reference original shape must be a positive (height, width).")
            gt = (_points(record["coords_rc"]) + .5) * (s / shape) - .5
            source = "verified_coordinate_input"
        else:
            gt, source = None, "unavailable"
            for key, label, binary in (("gauss_map", "gaussian_marker_peaks", False),
                                       ("mask_map", "binary_mask_centres_approximate", True)):
                path = ns.get(key, {}).get(name)
                if path is not None:
                    gt = marker_coordinates(read_gray_file(path), (s, s), opt, binary=binary)
                    source = label
                    break
        return noisy.astype(np.float32), clean, gt, source

    def fingerprint(self, names):
        records = []
        for name in names:
            record = {"sample": name}
            for key in ("noisy_map", "clean_map", "gauss_map", "mask_map"):
                p = self.ns.get(key, {}).get(name)
                if p is not None:
                    record[key] = dict(path=str(p), sha256=_sha256(p))
            if name in self.exact:
                record["exact_coords"] = _json_safe(self.exact[name])
            records.append(record)
        return records


def _canonical_state_dict(state):
    """Strip known wrapper PREFIXES only.

    The previous version removed any key component equal to "mod" at arbitrary
    depth, which silently renames a legitimate submodule and then fails
    strict loading with a misleading message.
    """
    if not isinstance(state, dict):
        raise TypeError("Checkpoint does not contain a state dictionary.")
    for key in ("model", "state_dict", "model_state_dict", "raw", "ema"):
        if isinstance(state.get(key), dict):
            state = state[key]
            break
    canonical = {}
    for key, value in state.items():
        if not hasattr(value, "shape"):        # step counters, EMA bookkeeping
            continue
        parts = str(key).split(".")
        while parts and parts[0] in ("module", "_orig_mod", "mod"):
            parts.pop(0)
        name = ".".join(parts)
        if name in canonical:
            raise ValueError(f"Checkpoint key collision while removing known wrappers: {name}")
        canonical[name] = value
    return canonical


class TorchPredictor:
    def __init__(self, model, device, options):
        import torch
        self.torch, self.device, self.options = torch, torch.device(device), options
        if options.inference_precision != "fp32" and self.device.type != "cuda":
            raise ValueError("Reduced-precision inference is configured only for CUDA.")
        if options.inference_precision == "bf16" and not torch.cuda.is_bf16_supported():
            raise ValueError("This GPU does not support the requested BF16 inference.")
        self.model = model.to(self.device).float().to(
            memory_format=torch.channels_last).eval()

    def __call__(self, images):
        torch, opt = self.torch, self.options
        x = torch.from_numpy(np.stack(images).astype(np.float32))[:, None]
        x = x.to(self.device, memory_format=torch.channels_last)
        dtype = {"fp16": torch.float16, "bf16": torch.bfloat16}.get(
            opt.inference_precision, torch.float32)
        with torch.inference_mode(), torch.autocast(
                self.device.type, enabled=opt.inference_precision != "fp32", dtype=dtype):
            den, logits = self.model(x)
        # Sigmoid in FP32 avoids adding further probability plateaus to BF16 logits.
        with torch.inference_mode(), torch.autocast(self.device.type, enabled=False):
            den, prob = den.float().clamp(0, 1), logits.float().sigmoid()
        if den.shape != x.shape or prob.shape != x.shape:
            raise ValueError("Model must return denoised images and binary logits of shape (B,1,H,W).")
        result = den[:, 0].cpu().numpy(), prob[:, 0].cpu().numpy()
        if not all(np.isfinite(v).all() for v in result):
            raise FloatingPointError("The model produced nonfinite outputs; evaluation stopped.")
        return result


# ---------------------------------------------------------------- fold sweep

def _fold_val_names(namespace, fold, base):
    """Same KFold arguments and same `common` order as training."""
    from sklearn.model_selection import KFold
    cfg, common, seed = namespace["CFG"], namespace["common"], namespace["SEED"]
    kf = KFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=seed)
    for k, (_, va) in enumerate(kf.split(common)):
        if k == int(fold) - base:
            return [common[i] for i in va]
    return None


def _discover_jobs(namespace, archs=None, conditions=None, folds=None):
    """(condition, arch, fold, checkpoint, fold_base) for everything on disk."""
    cond_use = namespace["CONDITION_USE_N2V"]
    if not namespace.get("CKPT_INDEX"):
        namespace["rebuild_ckpt_index"]()
    index = namespace["CKPT_INDEX"]
    archs = archs or namespace.get("RUN_ARCHS") or list(namespace["CFG"].ARCHS)
    conditions = conditions or [c for c in cond_use if index.get(c)]
    jobs = []
    for cond in conditions:
        base = namespace["fold_base_on_disk"](cond)
        for (arch, fold), ckpt in sorted(index.get(cond, {}).items()):
            if arch not in archs:
                continue
            if folds is not None and int(fold) not in {int(f) for f in folds}:
                continue
            jobs.append((cond, arch, int(fold), Path(ckpt), base))
    return jobs


def _load_model(namespace, arch, ckpt):
    """strict=True first; report exactly what differs before relaxing."""
    import torch
    state = torch.load(ckpt, map_location="cpu", weights_only=False)
    model = namespace["ARCH_REGISTRY"][arch]()
    sd = _canonical_state_dict(state)
    del state
    try:
        model.load_state_dict(sd, strict=True)
        return model, "strict"
    except RuntimeError as exc:
        res = model.load_state_dict(sd, strict=False)
        missing, unexpected = list(res.missing_keys), list(res.unexpected_keys)
        if len(missing) > 0.2 * len(model.state_dict()):
            raise RuntimeError(
                f"{Path(ckpt).name}: {len(missing)} of {len(model.state_dict())} "
                f"parameters absent - wrong architecture, not a wrapper mismatch."
            ) from exc
        print(f"      non-strict load: {len(missing)} missing, {len(unexpected)} unexpected")
        return model, f"non_strict({len(missing)}m/{len(unexpected)}u)"


def run_fold_localization(namespace, options, cond, arch, fold, ckpt, base, out_dir,
                          show=False):
    """One fold. Returns (summary, tables, records) or raises."""
    import torch
    names = _fold_val_names(namespace, fold, base)
    if not names:
        raise RuntimeError(f"fold {fold} split could not be reconstructed (base {base})")
    rng = np.random.default_rng(options.seed)
    count = len(names) if options.n_pool is None else min(options.n_pool, len(names))
    selected = [names[i] for i in rng.choice(len(names), size=count, replace=False)]

    model, load_mode = _load_model(namespace, arch, ckpt)
    predictor = TorchPredictor(model, namespace["DEVICE"], options)
    data = NotebookSamples(namespace, options)

    manifest = dict(checkpoint=str(ckpt), checkpoint_sha256=_sha256(ckpt),
                    load_mode=load_mode, condition=cond, arch=arch, fold=fold,
                    fold_base=base, options=asdict(options), selected_samples=selected,
                    input_files=data.fingerprint(selected),
                    python=platform.python_version(), torch=torch.__version__,
                    numpy=np.__version__)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    write_json_atomic(manifest, out_dir / "input_manifest.json")

    records, offset, batch = [], 0, options.batch_size
    start = time.perf_counter()
    while offset < len(selected):
        group = selected[offset:offset + batch]
        loaded = [data.load(n) for n in group]
        try:
            den, prob = predictor([v[0] for v in loaded])
        except torch.cuda.OutOfMemoryError:
            if batch == 1:
                raise RuntimeError("A single inference image does not fit. "
                                   "Release other GPU models and rerun.") from None
            batch = max(1, batch // 2)
            if predictor.device.type == "cuda":
                torch.cuda.empty_cache()
            print(f"      memory limit: retrying the same images at batch {batch}")
            continue
        for nm, (noisy, clean, gt, src), d, p in zip(group, loaded, den, prob):
            rec = evaluate_frame(nm, noisy, clean, d, p, gt, src, options)
            if len(records) >= options.n_examples:
                for k in ("noisy", "clean", "den", "gaussian", "prob"):
                    rec.pop(k)
            records.append(rec)
        offset += len(group)
    elapsed = time.perf_counter() - start

    metadata = dict(arch=arch, condition=cond, best_fold=fold, fold_base=base,
                    checkpoint=str(ckpt), checkpoint_sha256=manifest["checkpoint_sha256"],
                    load_mode=load_mode, elapsed_localization_s=elapsed,
                    actual_inference_batch=batch, n_val_total=len(names),
                    sampling="Fixed random subset chosen before inference",
                    output_directory=str(out_dir))
    summary, tables = export_localization(records, out_dir, options, metadata, show)
    del predictor, model
    if namespace["DEVICE"].type == "cuda":
        torch.cuda.empty_cache()
    return summary, tables, records


def plot_fold_comparison(frame, stem, options, show=False):
    """Per-fold detection quality and localization error, grouped by condition."""
    if frame.empty:
        return
    candidates = [("gt_f1", "Detection F1"), ("gt_precision", "Precision"),
                  ("gt_recall", "Recall"), ("gt_rmse_px", "Matched RMSE (px)"),
                  ("refine_success_rate", "Accepted fits"),
                  ("psnr_denoised_mean", "Denoised PSNR (dB)")]
    metrics = [(k, l) for k, l in candidates
               if k in frame.columns and frame[k].notna().any()]
    if not metrics:
        return
    conds = sorted(frame["condition"].unique())
    archs = list(dict.fromkeys(frame["arch"]))
    ncol = min(3, len(metrics))
    nrow = int(np.ceil(len(metrics) / ncol))
    with plt.rc_context({"font.size": 9, "axes.spines.top": False,
                         "axes.spines.right": False, "pdf.fonttype": 42}):
        fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.6 * nrow),
                                 squeeze=False, layout="constrained")
        for ax, (key, label) in zip(axes.flat, metrics):
            for arch in archs:
                for cond, ls in zip(conds, ("-", "--", ":")):
                    g = frame[(frame["arch"] == arch) &
                              (frame["condition"] == cond)].sort_values("fold")
                    if len(g) and g[key].notna().any():
                        ax.plot(g["fold"], g[key], marker="o", ms=4, ls=ls,
                                label=f"{arch} · {cond}")
            ax.set_xlabel("fold")
            ax.set_title(label, fontsize=10)
            ax.set_xticks(sorted(frame["fold"].unique()))
        for ax in axes.flat[len(metrics):]:
            ax.axis("off")
        handles, labels = axes.flat[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc="lower center",
                       ncol=min(4, len(labels)), fontsize=8, frameon=False)
        fig.suptitle("Localization across folds - each point is one held-out fold",
                     fontsize=13, weight="bold")
        _save_figure(fig, stem, options.dpi, show)


def aggregate_folds(rows, out_dir, options, show=False):
    """Mean and sd across folds. The sd is the fold-to-fold spread of a single CV
    sweep, which is the only uncertainty this design supports."""
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise ValueError("No fold completed.")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    _write_csv(frame, out_dir / "localization_per_fold.csv")

    keys = [k for k in ("gt_precision", "gt_recall", "gt_f1", "gt_rmse_px",
                        "refine_success_rate", "spacing_px_median",
                        "column_fwhm_px_mean", "psnr_denoised_mean",
                        "psnr_gaussian_mean", "psnr_noisy_mean")
            if k in frame.columns]
    agg = []
    for (cond, arch), g in frame.groupby(["condition", "arch"]):
        row = dict(condition=cond, arch=arch, n_folds=int(g["fold"].nunique()),
                   folds=",".join(str(int(f)) for f in sorted(g["fold"])),
                   n_images=int(g["n_samples"].sum()),
                   n_detections=int(g["n_atoms"].sum()))
        for k in keys:
            v = pd.to_numeric(g[k], errors="coerce").dropna()
            row[k] = float(v.mean()) if len(v) else None
            row[f"{k}_sd"] = float(v.std(ddof=1)) if len(v) > 1 else None
        agg.append(row)
    agg = pd.DataFrame(agg).sort_values(["condition", "arch"]).reset_index(drop=True)
    _write_csv(agg, out_dir / "localization_across_folds.csv")
    plot_fold_comparison(frame, out_dir / "localization_fold_comparison", options, show)

    lines = ["# Localization across all available folds", "",
             f"Folds evaluated: {len(frame)}  |  images per fold: "
             f"{options.n_pool or 'all'}  |  match gate: "
             f"{options.match_tolerance_px:g} px  |  threshold: "
             f"{options.probability_threshold:.2f}", "",
             "| Condition | Arch | Folds | F1 | Precision | Recall | RMSE (px) | Accepted fits |",
             "|---|---|---:|---|---|---|---|---|"]

    def pm(r, k, d=3):
        m, s = r.get(k), r.get(f"{k}_sd")
        if m is None or pd.isna(m):
            return "n/a"
        return f"{m:.{d}f}" + ("" if s is None or pd.isna(s) else f" ± {s:.{d}f}")

    for _, r in agg.iterrows():
        lines.append(f"| {r['condition']} | {r['arch']} | {r['n_folds']} | "
                     f"{pm(r, 'gt_f1')} | {pm(r, 'gt_precision')} | "
                     f"{pm(r, 'gt_recall')} | {pm(r, 'gt_rmse_px', 2)} | "
                     f"{pm(r, 'refine_success_rate')} |")
    lines += ["", "## Per fold", "",
              "| Condition | Arch | Fold | Images | Detections | F1 | RMSE (px) | Accepted fits |",
              "|---|---|---:|---:|---:|---|---|---|"]
    for _, r in frame.sort_values(["condition", "arch", "fold"]).iterrows():
        lines.append(f"| {r['condition']} | {r['arch']} | {int(r['fold'])} | "
                     f"{int(r['n_samples'])} | {int(r['n_atoms'])} | "
                     f"{_fmt(r.get('gt_f1'))} | {_fmt(r.get('gt_rmse_px'), 2)} | "
                     f"{_fmt(r.get('refine_success_rate'))} |")
    lines += ["", "## Interpretation", "",
              "- The spread across folds is the fold-to-fold variability of a single "
              "CV sweep, not a confidence interval on a population.",
              "- Every fold is a held-out validation split, not an independent test set.",
              "- Reference centres come from markers unless a verified coordinate input "
              "was supplied; see each fold's summary JSON for its source.",
              "- Position RMSE is conditional on a gated match, so it must be read "
              "alongside precision and recall.",
              "", "![Across folds](localization_fold_comparison.png)"]
    (out_dir / "localization_across_folds.md").write_text("\n".join(lines) + "\n",
                                                          encoding="utf-8")
    write_json_atomic(dict(per_fold=frame.to_dict("records"),
                           across_folds=agg.to_dict("records"),
                           options=asdict(options)),
                      out_dir / "localization_across_folds.json")
    return frame, agg


def run_all_folds_localization(namespace, options, archs=None, conditions=None,
                               folds=None, show=False):
    """Localize on every fold that has weights, in every enabled condition."""
    required = ("CFG", "ARCH_REGISTRY", "DEVICE", "common", "SEED", "noisy_map",
                "CONDITION_USE_N2V", "arch_dir", "fold_base_on_disk",
                "rebuild_ckpt_index")
    missing = [k for k in required if k not in namespace]
    if missing:
        raise RuntimeError("Run sections 1-9a first. Missing: " + ", ".join(missing))

    jobs = _discover_jobs(namespace, archs, conditions, folds)
    if not jobs:
        raise RuntimeError("No fold checkpoints found. Check CKPT_INDEX in section 9a.")

    run_id = time.strftime("%Y%m%dT%H%M%S")
    root = Path(namespace["CFG"].RESULTS_ROOT) / "localization_v2" / run_id
    print(f"Localization run {run_id}: {len(jobs)} fold(s)")
    for cond, arch, fold, ckpt, base in jobs:
        print(f"  {cond:9s} {arch:11s} fold {fold}  {Path(ckpt).name}")

    rows, per_fold, failures = [], {}, []
    for i, (cond, arch, fold, ckpt, base) in enumerate(jobs, 1):
        print(f"\n[{i}/{len(jobs)}] {cond} · {arch} · fold {fold}", flush=True)
        out = (Path(namespace["arch_dir"](arch, namespace["CONDITION_USE_N2V"][cond]))
               / "localization_v2" / run_id / f"fold_{fold:02d}")
        try:
            summary, tables, records = run_fold_localization(
                namespace, options, cond, arch, fold, ckpt, base, out, show)
        except Exception as exc:
            print(f"      failed: {exc.__class__.__name__}: {exc}", flush=True)
            failures.append(dict(condition=cond, arch=arch, fold=fold,
                                 error=f"{exc.__class__.__name__}: {exc}"))
            continue
        row = {k: v for k, v in summary.items() if not isinstance(v, (dict, list))}
        for m in ("psnr_noisy", "psnr_gaussian", "psnr_denoised"):
            row[f"{m}_mean"] = summary[m]["mean"]
        row.update(condition=cond, arch=arch, fold=int(fold), output_directory=str(out))
        rows.append(row)
        per_fold[(cond, arch, fold)] = dict(summary=summary, tables=tables,
                                            directory=str(out))

    if not rows:
        raise RuntimeError("Every fold failed; see the messages above.")
    frame, agg = aggregate_folds(rows, root, options, show)
    if failures:
        write_json_atomic(failures, root / "localization_failures.json")
        print(f"\n{len(failures)} fold(s) failed; see localization_failures.json")

    print("\n=== across folds ===")
    cols = [c for c in ("condition", "arch", "n_folds", "gt_f1", "gt_f1_sd",
                        "gt_precision", "gt_recall", "gt_rmse_px", "gt_rmse_px_sd",
                        "refine_success_rate") if c in agg.columns]
    print(agg[cols].round(4).to_string(index=False))
    print(f"\n-> {root}")

    best = max(rows, key=lambda r: (r.get("gt_f1") if r.get("gt_f1") is not None else -1))
    namespace.update(LOC_RUN_ID=run_id, LOC_ROOT=str(root), LOC_PER_FOLD=per_fold,
                     LOC_FOLD_FRAME=frame, LOC_ACROSS_FOLDS=agg, LOC_OPTIONS=options,
                     LOC_FAILURES=failures, loc_summary=best,
                     LOC_DIR=best["output_directory"],
                     LOC_BEST=(best["condition"], best["arch"], best["fold"]))
    return frame, agg


if __name__ == "__main__":
    # Edit these explicit settings once, then freeze them for the evaluation run.
    LOC_OPTIONS = LocalizationOptions(
        n_pool=100, batch_size=4, probability_threshold=.50,
        min_distance_px=4., match_tolerance_px=3.,
        inference_precision="fp32", fit_source="denoised",
        n_examples=2,                     # per fold, so keep this small for a sweep
        pixel_size_A=None, calibration_source="")

    # Start narrow to confirm discovery and loading, e.g.
    #   run_all_folds_localization(globals(), LOC_OPTIONS, folds=[1])
    run_all_folds_localization(globals(), LOC_OPTIONS)

## 11. Sub-pixel localization precision (synthetic shift test)

A noisy held-out frame is shifted by known sub-pixel offsets, the pipeline is
rerun, and the recovered atom centres are compared against the imposed shift.
Shift 0.0 is a repeatability control and shift 1.0 is an integer-shift
equivariance control; the fractional values are the actual measurement.

In [ ]:
# ============================================================
# 11. Sub-pixel localization precision (shift recovery, all folds)
# ============================================================
FORCE_PREC   = False
PREC_ARCHS   = list(CFG.ARCHS)
PREC_CONDS   = [k for k, on in (("no_n2v", RUN_NO_N2V),
                                ("with_n2v", RUN_WITH_N2V)) if on] or ["with_n2v"]
PREC_FOLDS   = "all"          # "all", "best", or e.g. [1, 2]
N_PREC       = 6              # images per fold; each costs 17 forward passes
SHIFTS       = (0.0, 0.25, -0.25, 0.5, -0.5, 0.75, -0.75, 1.0)
EDGE_M       = 8              # px, drop atoms near borders (patch clearance)
PAIR_TOL     = 2.0            # px, = min_distance_px / 2
PREC_BATCH   = 8              # shifted variants per forward pass

for _n in ("detect_centres", "refine_centres", "LocalizationOptions"):
    if _n not in globals():
        raise RuntimeError("Run the section 10 file first; it defines "
                           "detect_centres, refine_centres and LocalizationOptions.")

PREC_OPTIONS = globals().get("LOC_OPTIONS") or LocalizationOptions(
    n_pool=None, probability_threshold=0.50, min_distance_px=4.0,
    fit_source="denoised", inference_precision="fp32", pixel_size_A=None)


def shift_image(img, dx, dy):
    """Content moves by (+dx cols, +dy rows). INTER_CUBIC for sub-pixel
    fidelity. Fractional shifts interpolate and so smooth the noise slightly,
    making fractional-shift errors marginally optimistic; integer shifts are
    exact translations and are reported separately as a control."""
    H, W = img.shape
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    return np.clip(cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_CUBIC,
                                  borderMode=cv2.BORDER_REFLECT), 0.0, 1.0)


def match_pairs(a, b, tol_px):
    """Hungarian one-to-one match of rows of a to rows of b within tol_px."""
    if len(a) == 0 or len(b) == 0:
        return np.array([], int), np.array([], int)
    D = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)
    ri, ci = linear_sum_assignment(D)
    k = D[ri, ci] <= tol_px
    return ri[k], ci[k]


@torch.inference_mode()
def _forward(model, images):
    x = torch.from_numpy(np.stack(images).astype(np.float32))[:, None].to(
        DEVICE, memory_format=torch.channels_last)
    den, lg = model(x)
    return (den.float().clamp(0, 1)[:, 0].cpu().numpy(),
            torch.sigmoid(lg.float())[:, 0].cpu().numpy())


def _localize(model, images, opt):
    """Peaks and refined centres for a batch of images."""
    den, prob = _forward(model, images)
    out = []
    for k, img in enumerate(images):
        peaks = detect_centres(prob[k], opt)
        src = den[k] if opt.fit_source == "denoised" else img
        refined, accepted, _, _, _ = refine_centres(src, peaks, opt)
        out.append((peaks.astype(float), refined, accepted))
    return out


def _clean(v):
    """NaN is not valid JSON; None is."""
    if isinstance(v, dict):
        return {k: _clean(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_clean(x) for x in v]
    if isinstance(v, (float, np.floating)):
        return float(v) if np.isfinite(v) else None
    if isinstance(v, (np.integer,)):
        return int(v)
    return v


def _stats(g):
    """std over fewer than two pairs is undefined; NaN rather than a number
    that looks like a measurement."""
    if len(g) == 0:
        return dict(n=0, bias_x=None, bias_y=None, sigma_x=None, sigma_y=None,
                    rmse_2d=None)
    ex, ey = g["err_col"].to_numpy(), g["err_row"].to_numpy()
    return dict(n=int(len(g)), bias_x=float(ex.mean()), bias_y=float(ey.mean()),
                sigma_x=float(ex.std(ddof=1)) if len(g) > 1 else float("nan"),
                sigma_y=float(ey.std(ddof=1)) if len(g) > 1 else float("nan"),
                rmse_2d=float(np.sqrt(np.mean(ex**2 + ey**2))))


def precision_one_fold(arch, cond, fold, ckpt, base, opt=PREC_OPTIONS,
                       n_images=N_PREC):
    """Shift-recovery errors for one checkpoint. Returns (errors_df, matches_df)."""
    names = fold_val_names(fold, base)
    if not names:
        raise RuntimeError(f"fold {fold} split could not be reconstructed")
    picks = random.Random(SEED).sample(names, min(n_images, len(names)))

    # (dx, dy, axis) with the duplicate zero shift removed.
    variants = []
    for axis in ("x", "y"):
        for s in SHIFTS:
            if s == 0.0 and axis == "y":
                continue
            variants.append(((s, 0.0) if axis == "x" else (0.0, s), axis, s))

    model = None
    err_rows, match_rows = [], []
    try:
        model = prep_model(ARCH_REGISTRY[arch]())
        load_model_weights(ckpt, model)
        model.eval()
        for nm in picks:
            noisy = center_or_resize(norm01(imread_gray(noisy_map[nm])), CFG.IMG_SIZE)
            pk0, rf0, _ = _localize(model, [noisy], opt)[0]
            if len(rf0) < 5:
                continue
            det0 = {"refined": rf0, "raw": pk0}

            for i in range(0, len(variants), PREC_BATCH):
                chunk = variants[i:i + PREC_BATCH]
                imgs = [shift_image(noisy, dx, dy) for (dx, dy), _, _ in chunk]
                for ((dx, dy), axis, s), (pks, rfs, _) in zip(chunk,
                                                              _localize(model, imgs, opt)):
                    for meth, d0 in det0.items():
                        ds = rfs if meth == "refined" else pks
                        if len(ds) == 0:
                            continue
                        # Detections are (row, col); dy moves rows, dx moves cols.
                        ia, ib = match_pairs(ds - np.array([dy, dx]), d0, PAIR_TOL)
                        match_rows.append(dict(arch=arch, condition=cond, fold=fold,
                                               method=meth, axis=axis, shift=s,
                                               n_det=len(ds), n_match=len(ia)))
                        if len(ia) == 0:
                            continue
                        keep = np.all((d0[ib] >= EDGE_M)
                                      & (d0[ib] < CFG.IMG_SIZE - EDGE_M), axis=1)
                        err = (ds[ia] - np.array([dy, dx]) - d0[ib])[keep]
                        frac = abs(s - round(s)) > 1e-9
                        for ey, ex in err:
                            err_rows.append(dict(arch=arch, condition=cond, fold=fold,
                                                 sample=nm, axis=axis, shift=s,
                                                 method=meth, is_frac=frac,
                                                 err_row=ey, err_col=ex))
    finally:
        if model is not None:
            del model
        free_gpu()

    ef, mf = pd.DataFrame(err_rows), pd.DataFrame(match_rows)
    if not ef.empty:
        ef["err_along"] = np.where(ef["axis"] == "x", ef["err_col"], ef["err_row"])
    return ef, mf


def summarize_precision(ef, mf, arch, cond, fold):
    frac_ref = _stats(ef[(ef.method == "refined") & ef.is_frac])
    frac_raw = _stats(ef[(ef.method == "raw") & ef.is_frac])
    ctrl0 = ef[(ef["shift"] == 0.0) & (ef["method"] == "refined")]
    equiv = _stats(ef[(ef["shift"] == 1.0) & (ef["method"] == "refined")])
    rate = float("nan")
    if not mf.empty:
        g = mf[mf.method == "refined"]
        rate = float(g.n_match.sum() / max(g.n_det.sum(), 1)) if len(g) else float("nan")
    row = dict(arch=arch, condition=cond, fold=fold,
               n_pairs=frac_ref["n"], match_rate_refined=rate,
               rmse_2d_px=frac_ref["rmse_2d"],
               # A pair error is the difference of two measurements; if roughly
               # independent (true at fractional phase), single-shot RMSE is
               # about pair RMSE / sqrt(2).
               single_shot_rmse_px=(frac_ref["rmse_2d"] / np.sqrt(2)
                                    if frac_ref["rmse_2d"] is not None else None),
               sigma_x_px=frac_ref["sigma_x"], sigma_y_px=frac_ref["sigma_y"],
               bias_x_px=frac_ref["bias_x"], bias_y_px=frac_ref["bias_y"],
               raw_rmse_2d_px=frac_raw["rmse_2d"],
               control_shift0_max_abs_px=(
                   float(np.abs(ctrl0[["err_row", "err_col"]].to_numpy()).max())
                   if len(ctrl0) else None),
               control_shift1_rmse_px=equiv["rmse_2d"])
    return row


def plot_precision(ef, stem, title):
    if ef.empty:
        return
    lock = (ef[ef.is_frac | (ef.shift == 1.0)]
            .groupby(["method", "shift"])["err_along"]
            .agg(["mean", "std", "count"]).reset_index())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    bins = np.linspace(-1.0, 1.0, 61)
    for meth, col in (("raw", "gray"), ("refined", "steelblue")):
        g = ef[(ef.method == meth) & ef.is_frac]
        if len(g) < 2:
            continue
        axes[0].hist(g["err_along"], bins=bins, alpha=0.6, color=col,
                     label=f"{meth}: sigma={g['err_along'].std(ddof=1):.3f} px")
    axes[0].axvline(0, c="k", lw=0.8)
    axes[0].set_xlabel("per-axis shift-recovery error (px)")
    axes[0].set_ylabel("atoms")
    axes[0].legend(fontsize=8)
    axes[0].set_title("Fractional shifts, pooled over both axes")
    for meth, col in (("raw", "gray"), ("refined", "steelblue")):
        g = lock[lock.method == meth]
        if g.empty:
            continue
        axes[1].errorbar(g["shift"], g["mean"], yerr=g["std"], marker="o", ms=4,
                         capsize=3, color=col, label=meth)
    axes[1].axhline(0, c="k", lw=0.8)
    axes[1].set_xlabel("imposed shift (px)")
    axes[1].set_ylabel("mean recovery error along shift axis (px)")
    axes[1].set_title("Bias vs sub-pixel phase (pixel locking shows as a sawtooth)")
    axes[1].legend(fontsize=8)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.savefig(str(stem) + ".png", dpi=150, bbox_inches="tight")
    plt.show()


# ---------------- run over every fold ----------------
PREC_ROOT = CFG.RESULTS_ROOT / "precision"
PREC_ROOT.mkdir(parents=True, exist_ok=True)
PREC_ALL_CSV = PREC_ROOT / "precision_errors_all.csv"
PREC_SUM_CSV = PREC_ROOT / "precision_per_fold.csv"

if PREC_SUM_CSV.is_file() and not FORCE_PREC:
    prec_per_fold = pd.read_csv(PREC_SUM_CSV)
    print("precision results already on disk (FORCE_PREC=True to recompute):")
    print(prec_per_fold.round(4).to_string(index=False))
else:
    if not CKPT_INDEX:
        rebuild_ckpt_index()
    jobs = []
    for cond in PREC_CONDS:
        base = fold_base_on_disk(cond)
        idx = CKPT_INDEX.get(cond, {})
        for arch in PREC_ARCHS:
            folds = sorted(f for (a, f) in idx if a == arch)
            if PREC_FOLDS == "best":
                res, _ = read_cv_results(cond)
                got, _ = arch_folds(res, arch, cond, which="best") if not res.empty else ([], None)
                folds = [f for f, _, _ in got] or folds[:1]
            elif isinstance(PREC_FOLDS, (list, tuple, set)):
                folds = [f for f in folds if f in {int(x) for x in PREC_FOLDS}]
            jobs += [(cond, arch, f, idx[(arch, f)], base) for f in folds]

    if not jobs:
        raise RuntimeError("No checkpoints found; check CKPT_INDEX in section 9a.")
    print(f"{len(jobs)} fold(s), {N_PREC} images each, "
          f"{len(SHIFTS)*2-1} shifted variants per image")

    all_err, all_sum = [], []
    for i, (cond, arch, fold, ckpt, base) in enumerate(jobs, 1):
        print(f"\n[{i}/{len(jobs)}] {cond} · {arch} · fold {fold}", flush=True)
        t0 = time.time()
        try:
            ef, mf = precision_one_fold(arch, cond, fold, ckpt, base)
        except Exception as exc:
            print(f"    failed: {exc.__class__.__name__}: {exc}")
            continue
        if ef.empty:
            print("    no matched pairs; the model may detect too few atoms here, "
                  "or probability_threshold/min_distance_px need retuning")
            continue
        row = summarize_precision(ef, mf, arch, cond, fold)
        row["seconds"] = round(time.time() - t0, 1)
        all_err.append(ef)
        all_sum.append(row)
        print(f"    pair RMSE {row['rmse_2d_px']:.3f} px "
              f"(~{row['single_shot_rmse_px']:.3f} single-shot)  "
              f"sigma {row['sigma_x_px']:.3f}/{row['sigma_y_px']:.3f}  "
              f"bias {row['bias_x_px']:+.3f}/{row['bias_y_px']:+.3f}  "
              f"match {row['match_rate_refined']:.3f}  n={row['n_pairs']}  "
              f"[{row['seconds']:.0f}s]")

    if not all_sum:
        raise RuntimeError("Every fold produced zero matched pairs.")
    prec_errors = pd.concat(all_err, ignore_index=True)
    prec_per_fold = pd.DataFrame(all_sum)
    prec_errors.to_csv(PREC_ALL_CSV, index=False)
    prec_per_fold.round(6).to_csv(PREC_SUM_CSV, index=False)

    # ---------------- aggregate ----------------
    keys = ["rmse_2d_px", "single_shot_rmse_px", "sigma_x_px", "sigma_y_px",
            "bias_x_px", "bias_y_px", "raw_rmse_2d_px", "match_rate_refined",
            "control_shift0_max_abs_px", "control_shift1_rmse_px"]
    agg = []
    for (cond, arch), g in prec_per_fold.groupby(["condition", "arch"]):
        r = dict(condition=cond, arch=arch, n_folds=len(g),
                 n_pairs=int(g["n_pairs"].sum()))
        for k in keys:
            v = pd.to_numeric(g[k], errors="coerce").dropna()
            r[k] = float(v.mean()) if len(v) else None
            r[f"{k}_sd"] = float(v.std(ddof=1)) if len(v) > 1 else None
        agg.append(r)
    prec_across = pd.DataFrame(agg).sort_values(["condition", "arch"]).reset_index(drop=True)
    prec_across.round(6).to_csv(PREC_ROOT / "precision_across_folds.csv", index=False)

    print("\n=== across folds (mean ± sd over folds) ===")
    for _, r in prec_across.iterrows():
        def pm(k, d=3):
            m, s = r[k], r.get(f"{k}_sd")
            return "n/a" if m is None or pd.isna(m) else (
                f"{m:.{d}f}" + ("" if s is None or pd.isna(s) else f" ± {s:.{d}f}"))
        print(f"  {r['condition']:9s} {r['arch']:11s} folds {r['n_folds']}  "
              f"pair RMSE {pm('rmse_2d_px')} px  "
              f"single-shot {pm('single_shot_rmse_px')} px  "
              f"bias {pm('bias_x_px')}/{pm('bias_y_px')}  "
              f"match {pm('match_rate_refined')}")

    # Per-fold RMSE is what supports or refuses the claim, not the pooled mean.
    for _, r in prec_across.iterrows():
        sub = prec_per_fold[(prec_per_fold.condition == r["condition"])
                            & (prec_per_fold.arch == r["arch"])]
        n_ok = int((sub["rmse_2d_px"] < 0.5).sum())
        print(f"  {r['condition']:9s} {r['arch']:11s} pair RMSE < 0.5 px on "
              f"{n_ok}/{len(sub)} fold(s)"
              + ("  [raw quantization floor per axis is 1/sqrt(12) = 0.289 px]"
                 if n_ok == len(sub) else ""))

    # ---------------- figures ----------------
    for (cond, arch), g in prec_errors.groupby(["condition", "arch"]):
        plot_precision(g, arch_dir(arch, CONDITION_USE_N2V[cond]) /
                       f"{arch}_precision_pooled",
                       f"{arch} · {cond} · pooled over "
                       f"{g['fold'].nunique()} fold(s)")

    if prec_per_fold["fold"].nunique() > 1:
        fig, ax = plt.subplots(figsize=(7.5, 4.2))
        for (cond, arch), g in prec_per_fold.groupby(["condition", "arch"]):
            g = g.sort_values("fold")
            ax.plot(g["fold"], g["rmse_2d_px"], marker="o", ms=5,
                    ls="-" if cond == "no_n2v" else "--", label=f"{arch} · {cond}")
        ax.axhline(0.5, c="k", lw=0.8, ls=":", label="0.5 px")
        ax.set_xlabel("fold")
        ax.set_ylabel("pair RMSE (px)")
        ax.set_xticks(sorted(prec_per_fold["fold"].unique()))
        ax.set_title("Shift-recovery precision across folds")
        ax.legend(fontsize=8, ncol=2)
        plt.tight_layout()
        plt.savefig(PREC_ROOT / "precision_across_folds.png", dpi=150,
                    bbox_inches="tight")
        plt.show()

    write_json_atomic(_clean(dict(
        options=asdict(PREC_OPTIONS), shifts=list(SHIFTS), n_images=N_PREC,
        edge_margin_px=EDGE_M, pair_tol_px=PAIR_TOL,
        dataset_hash=getattr(CFG, "DATASET_HASH", None),
        pixel_size_A=PREC_OPTIONS.pixel_size_A,
        calibration_source=PREC_OPTIONS.calibration_source,
        per_fold=prec_per_fold.to_dict("records"),
        across_folds=prec_across.to_dict("records"),
        notes=[
            "Pair errors compare two independent measurements of the same atom; "
            "single-shot RMSE is estimated as pair RMSE / sqrt(2).",
            "Fractional shifts interpolate the image and slightly smooth the noise, "
            "so fractional-shift errors are marginally optimistic.",
            "Shift 0.0 measures repeatability including cuDNN nondeterminism; "
            "shift 1.0 measures translation equivariance without interpolation.",
            "No verified pixel calibration is applied, so all values are in pixels."
            if PREC_OPTIONS.pixel_size_A is None else
            f"Angstrom values use {PREC_OPTIONS.pixel_size_A} A/px from "
            f"{PREC_OPTIONS.calibration_source}."])),
        PREC_ROOT / "precision_summary.json")
    print(f"\n-> {PREC_ROOT}")

In [ ]:
"""Section 13: displacement and strain mapping from detected atomic columns.

Paste into a notebook cell, or run it there with `%run -i section13_strain_mapping.py`.
Requires section 10 (detect_centres, refine_centres, LocalizationOptions) and
section 9a (CKPT_INDEX, fold_base_on_disk, arch_dir, CONDITION_USE_N2V).

Method
------
1. Estimate a reference lattice basis from nearest-neighbour vectors, using
   circular statistics on the DOUBLED angle so that v and -v are one direction
   and the +-pi wrap is handled correctly.
2. Lagrange-reduce the basis, assign integer (m, n) indices, and refine the
   basis and origin by alternating least squares.
3. Displacement u_i = r_i - (A n_i + r0).
4. Strain from a locally weighted least-squares fit of the displacement
   gradient, decomposed into symmetric strain and rigid rotation.

What this does and does not measure
-----------------------------------
The reference lattice is fitted to the same image, so the mean strain is zero
by construction. Only spatial VARIATION of strain is measurable this way; an
absolute value requires an external reference region or a calibrated lattice
parameter. Strain is dimensionless, so it does not depend on the pixel size at
all; only the reported lattice parameter does.

On unstrained simulated data the spread of the measured strain is the noise
floor of the whole detect-refine-fit chain, not physical strain.
"""
from __future__ import annotations

import json
import os
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree


# ---------------------------------------------------------------- options

@dataclass(frozen=True)
class StrainOptions:
    # Robust lattice candidate search.  The old single-angle-peak selection was
    # too brittle when detections contained missing atoms, false positives,
    # multiple neighbour shells, or a multi-atom basis.
    vector_k: int = 10                 # neighbours sampled per detected centre
    max_shell_factor: float = 3.0      # search vectors out to this x median NN
    min_vector_factor: float = 0.55    # reject near-duplicate detections below this x NN
    vector_angle_bin_deg: float = 3.0  # joint angle/length clustering
    vector_length_bin_factor: float = 0.12
    min_vector_cluster_count: int = 4
    max_vector_clusters: int = 28
    max_candidate_pairs: int = 360
    min_sin_between: float = 0.25      # reject nearly collinear bases
    origin_trials: int = 24            # deterministic candidate lattice phases
    lattice_inlier_factor: float = 0.24 # radial residual / shortest basis
    min_lattice_inlier_fraction: float = 0.25
    robust_refine_iterations: int = 8
    max_raw_collision_rate: float = 0.25

    # Legacy names retained so older notebook cells do not break if they inspect
    # the object.  They are no longer used to choose the lattice basis.
    nn_radius_factor: float = 1.8
    n_angle_bins: int = 180
    angle_smooth_bins: int = 5
    min_direction_count: int = 12
    refine_iterations: int = 8

    # strain fit
    neighbour_k: int = 12
    weight_sigma_factor: float = 1.0
    min_neighbours: int = 6
    max_condition_number: float = 50.0
    edge_margin_factor: float = 1.5

    # final sanity gates.  Do not loosen these to force a result.
    max_displacement_factor: float = 0.35
    min_atoms: int = 30
    max_index_collision_rate: float = 0.02

    # reporting
    robust_percentile: float = 2.0
    lattice_parameter_A: float | None = None
    calibration_source: str = ""

    def __post_init__(self):
        if not 3 <= self.vector_k <= 64:
            raise ValueError("vector_k must lie in [3, 64].")
        if not 1.2 <= self.max_shell_factor <= 6.0:
            raise ValueError("max_shell_factor must lie in [1.2, 6].")
        if not 0.1 < self.min_vector_factor < 1.0:
            raise ValueError("min_vector_factor must lie in (0.1, 1).")
        if not 0 < self.vector_angle_bin_deg <= 15:
            raise ValueError("vector_angle_bin_deg must lie in (0, 15].")
        if not 0.03 <= self.vector_length_bin_factor <= 0.5:
            raise ValueError("vector_length_bin_factor must lie in [0.03, 0.5].")
        if self.min_vector_cluster_count < 2:
            raise ValueError("min_vector_cluster_count must be >= 2.")
        if self.max_vector_clusters < 4 or self.max_candidate_pairs < 1:
            raise ValueError("Too few lattice candidates requested.")
        if not 0 < self.min_sin_between < 1:
            raise ValueError("min_sin_between must lie in (0, 1).")
        if self.origin_trials < 1:
            raise ValueError("origin_trials must be positive.")
        if not 0.05 <= self.lattice_inlier_factor < 0.5:
            raise ValueError("lattice_inlier_factor must lie in [0.05, 0.5).")
        if not 0 < self.min_lattice_inlier_fraction <= 1:
            raise ValueError("min_lattice_inlier_fraction must lie in (0, 1].")
        if not 0 <= self.max_raw_collision_rate < 1:
            raise ValueError("max_raw_collision_rate must lie in [0, 1).")
        if self.neighbour_k < 3 or self.min_neighbours < 3:
            raise ValueError("A plane fit needs at least three neighbours.")
        if self.min_neighbours > self.neighbour_k:
            raise ValueError("min_neighbours cannot exceed neighbour_k.")
        if not 0 < self.max_displacement_factor < 0.5:
            raise ValueError("max_displacement_factor must be below 0.5.")
        if self.lattice_parameter_A is not None and not self.calibration_source.strip():
            raise ValueError("A lattice parameter in Angstrom needs a calibration_source.")


class LatticeError(RuntimeError):
    """Raised when no trustworthy reference lattice can be fitted."""


# ---------------------------------------------------------------- geometry

def to_xy(coords_rc):
    """Detections are (row, col). Everything below works in (x, y) = (col, row),
    so the conversion happens once, here, rather than being reasoned about at
    every use site."""
    a = np.asarray(coords_rc, dtype=float)
    if a.ndim != 2 or a.shape[1] != 2:
        raise ValueError("Coordinates must be an (N, 2) array.")
    return np.column_stack((a[:, 1], a[:, 0]))


def _fold_half_plane(v):
    """v and -v are the same lattice direction, so fold onto y > 0 (and x > 0
    along the y = 0 line). Without this the angle histogram has every direction
    duplicated 180 degrees apart and the two 'strongest' peaks can be the same
    direction twice."""
    out = np.asarray(v, dtype=float).copy()
    flip = (out[:, 1] < 0) | ((np.abs(out[:, 1]) < 1e-12) & (out[:, 0] < 0))
    out[flip] *= -1.0
    return out


def _circular_boxcar(counts, width):
    if width <= 1:
        return counts.astype(float)
    k = np.ones(int(width)) / float(width)
    padded = np.concatenate([counts[-len(k):], counts, counts[:len(k)]])
    smoothed = np.convolve(padded, k, mode="same")
    return smoothed[len(k):len(k) + len(counts)]


def candidate_directions(points, options):
    """Lattice directions from the pooled neighbour vectors.

    Angles are folded to [0, pi) and then DOUBLED before any circular average,
    so 179 degrees and 1 degree average to 0, not to 90. Averaging the folded
    angles directly is the classic failure here and puts a phantom direction
    halfway between two real ones.
    """
    pts = np.asarray(points, dtype=float)
    if len(pts) < options.min_atoms:
        raise LatticeError(f"only {len(pts)} atoms; need {options.min_atoms}")
    tree = cKDTree(pts)
    nn = tree.query(pts, k=2)[0][:, 1]
    d0 = float(np.median(nn))
    if not np.isfinite(d0) or d0 <= 0:
        raise LatticeError("degenerate nearest-neighbour distance")

    pairs = tree.query_pairs(d0 * options.nn_radius_factor, output_type="ndarray")
    if len(pairs) < options.min_direction_count:
        raise LatticeError(f"only {len(pairs)} neighbour vectors within "
                           f"{options.nn_radius_factor:.2f} x d0")
    vec = _fold_half_plane(pts[pairs[:, 1]] - pts[pairs[:, 0]])
    length = np.linalg.norm(vec, axis=1)
    keep = length > 1e-9
    vec, length = vec[keep], length[keep]
    theta = np.mod(np.arctan2(vec[:, 1], vec[:, 0]), np.pi)     # [0, pi)

    doubled = 2.0 * theta                                        # [0, 2pi)
    bins = int(options.n_angle_bins)
    counts, edges = np.histogram(doubled, bins=bins, range=(0.0, 2 * np.pi))
    smooth = _circular_boxcar(counts, options.angle_smooth_bins)
    left, right = np.roll(smooth, 1), np.roll(smooth, -1)
    peak = (smooth >= left) & (smooth > right) & (smooth > 0)
    peak_bins = np.flatnonzero(peak)
    if not len(peak_bins):
        raise LatticeError("no angular peak in the neighbour-vector distribution")

    half_width = np.pi / bins * max(options.angle_smooth_bins, 3)
    directions = []
    for b in peak_bins[np.argsort(smooth[peak_bins])[::-1]]:
        centre = 0.5 * (edges[b] + edges[b + 1])                 # doubled angle
        delta = np.abs(np.angle(np.exp(1j * (doubled - centre))))
        member = delta <= 2 * half_width
        if member.sum() < options.min_direction_count:
            continue
        # Circular mean on the doubled angle, then halve to recover [0, pi).
        mean_angle = 0.5 * np.mod(np.angle(np.mean(np.exp(1j * doubled[member]))),
                                  2 * np.pi)
        # Median length within the cluster resists the second-shell vectors
        # that leak in at the radius cut.
        med_len = float(np.median(length[member]))
        directions.append(dict(angle=float(mean_angle), length=med_len,
                               count=int(member.sum()),
                               vector=np.array([med_len * np.cos(mean_angle),
                                                med_len * np.sin(mean_angle)])))
    if not directions:
        raise LatticeError("no angular cluster met min_direction_count")
    directions.sort(key=lambda d: d["length"])
    return directions, d0



def _cross2(a, b):
    """Scalar z-component of the cross product for two 2-D vectors.

    Avoids np.cross(a, b), whose 2-vector behaviour is deprecated/removed in
    newer NumPy versions.
    """
    a = np.asarray(a, dtype=float).reshape(2)
    b = np.asarray(b, dtype=float).reshape(2)
    return float(a[0] * b[1] - a[1] * b[0])


def _reduce_basis(a1, a2, max_steps=64):
    """Lagrange-Gauss reduction: the shortest equivalent basis of the same
    lattice. Two runs on the same lattice then yield the same basis, so strain
    is comparable between images."""
    a1, a2 = np.asarray(a1, float).copy(), np.asarray(a2, float).copy()
    for _ in range(max_steps):
        if a2 @ a2 < a1 @ a1:
            a1, a2 = a2, a1
        mu = np.round((a1 @ a2) / (a1 @ a1))
        if mu == 0:
            break
        a2 = a2 - mu * a1
    if _cross2(a1, a2) < 0:            # keep it right-handed
        a2 = -a2
    return a1, a2


def initial_basis(points, options):
    directions, d0 = candidate_directions(points, options)
    a1 = directions[0]["vector"]
    a2 = None
    for cand in directions[1:]:
        v = cand["vector"]
        sin_between = abs(_cross2(a1, v)) / (np.linalg.norm(a1) * np.linalg.norm(v))
        if sin_between >= options.min_sin_between:
            a2 = v
            break
    if a2 is None:
        raise LatticeError("only one independent lattice direction found; the "
                           "detections may lie on a line or a single row")
    return _reduce_basis(a1, a2) + (d0, directions)


def refine_lattice(points, a1, a2, options):
    """Alternate integer indexing and least-squares refinement of (A, r0).

    r_i ~ A @ [m_i, n_i] + r0.  Indices are recomputed from the refined basis
    each round; the loop stops when no index changes.
    """
    pts = np.asarray(points, float)
    A = np.column_stack((a1, a2))
    if abs(np.linalg.det(A)) < 1e-9:
        raise LatticeError("degenerate basis (zero determinant)")
    r0 = pts[np.argmin(np.linalg.norm(pts - pts.mean(axis=0), axis=1))]

    idx = None
    for _ in range(options.refine_iterations):
        new_idx = np.rint(np.linalg.solve(A, (pts - r0).T).T)
        if idx is not None and np.array_equal(new_idx, idx):
            break
        idx = new_idx
        design = np.column_stack((idx, np.ones(len(pts))))
        sol, *_ = np.linalg.lstsq(design, pts, rcond=None)   # (3, 2)
        A = sol[:2].T                                        # columns a1, a2
        r0 = sol[2]
        if abs(np.linalg.det(A)) < 1e-9:
            raise LatticeError("basis collapsed during refinement")

    # Two atoms on one lattice site means the basis is too small (a superlattice
    # was picked up) or the detections contain duplicates.
    keys = np.ascontiguousarray(idx).view([("m", idx.dtype), ("n", idx.dtype)])
    _, counts = np.unique(keys, return_counts=True)
    collisions = int((counts - 1).sum())
    rate = collisions / max(len(pts), 1)
    if rate > options.max_index_collision_rate:
        raise LatticeError(f"{collisions} of {len(pts)} atoms share a lattice index "
                           f"({rate:.1%}); the fitted cell is too small")

    residual = pts - (idx @ A.T + r0)
    shortest = min(np.linalg.norm(A[:, 0]), np.linalg.norm(A[:, 1]))
    max_disp = float(np.abs(residual).max()) if len(residual) else 0.0
    if max_disp > options.max_displacement_factor * shortest:
        raise LatticeError(
            f"max displacement {max_disp:.2f} px exceeds "
            f"{options.max_displacement_factor:.2f} x the {shortest:.2f} px basis "
            "vector; indices alias to neighbouring sites and the strain would be "
            "meaningless")
    return dict(A=A, r0=r0, indices=idx.astype(int), displacement=residual,
                collisions=collisions, collision_rate=rate,
                rmse_px=float(np.sqrt(np.mean(np.sum(residual ** 2, axis=1)))),
                a1_px=float(np.linalg.norm(A[:, 0])), a2_px=float(np.linalg.norm(A[:, 1])),
                angle_deg=float(np.degrees(np.arccos(
                    np.clip((A[:, 0] @ A[:, 1]) /
                            (np.linalg.norm(A[:, 0]) * np.linalg.norm(A[:, 1])),
                            -1, 1)))),
                cell_area_px2=float(abs(np.linalg.det(A))))



def _robust_nn_scale(points):
    """Median nearest-neighbour scale with pathological duplicates trimmed.

    This is only a search scale; the final lattice vectors are inferred from
    repeated pair vectors and need not equal the nearest-neighbour distance.
    """
    pts = np.asarray(points, float)
    if len(pts) < 3:
        raise LatticeError("fewer than three finite detections")
    d = cKDTree(pts).query(pts, k=2)[0][:, 1]
    d = d[np.isfinite(d) & (d > 1e-6)]
    if len(d) < 3:
        raise LatticeError("degenerate nearest-neighbour distances")
    lo, hi = np.percentile(d, (10, 90))
    core = d[(d >= lo) & (d <= hi)]
    d0 = float(np.median(core if len(core) else d))
    if not np.isfinite(d0) or d0 <= 0:
        raise LatticeError("invalid nearest-neighbour scale")
    return d0, d


def _vector_clusters(points, d0, options):
    """Repeated neighbour vectors clustered jointly by direction and length."""
    pts = np.asarray(points, float)
    n = len(pts)
    k = min(options.vector_k + 1, n)
    dist, nbr = cKDTree(pts).query(pts, k=k)

    vectors = []
    min_len = options.min_vector_factor * d0
    max_len = options.max_shell_factor * d0
    seen = set()
    for i in range(n):
        for dd, j in zip(np.atleast_1d(dist[i])[1:], np.atleast_1d(nbr[i])[1:]):
            j = int(j)
            if j == i or not np.isfinite(dd) or dd < min_len or dd > max_len:
                continue
            key = (min(i, j), max(i, j))
            if key in seen:
                continue
            seen.add(key)
            vectors.append(pts[j] - pts[i])

    if len(vectors) < 2 * options.min_vector_cluster_count:
        raise LatticeError(
            f"only {len(vectors)} usable neighbour vectors within "
            f"[{options.min_vector_factor:.2f}, {options.max_shell_factor:.2f}] x d0")

    vec = _fold_half_plane(np.asarray(vectors, float))
    length = np.linalg.norm(vec, axis=1)
    theta = np.mod(np.arctan2(vec[:, 1], vec[:, 0]), np.pi)
    angle_step = np.deg2rad(options.vector_angle_bin_deg)
    length_step = max(options.vector_length_bin_factor * d0, 1e-6)

    # Quantise both polar coordinates.  Direction-only clustering was the main
    # reason the old code mixed first and second neighbour shells.
    ab = np.floor(theta / angle_step + 0.5).astype(int)
    lb = np.floor(length / length_step + 0.5).astype(int)
    groups = {}
    for i, key in enumerate(zip(ab, lb)):
        groups.setdefault(key, []).append(i)

    clusters = []
    for ids in groups.values():
        if len(ids) < options.min_vector_cluster_count:
            continue
        ids = np.asarray(ids, int)
        # Median Cartesian vector is stable inside a narrow angle/length bin.
        v = np.median(vec[ids], axis=0)
        L = float(np.linalg.norm(v))
        if not np.isfinite(L) or L <= 1e-6:
            continue
        ang = float(np.mod(np.arctan2(v[1], v[0]), np.pi))
        clusters.append(dict(vector=v, length=L, angle=ang, count=int(len(ids))))

    if len(clusters) < 2:
        raise LatticeError("fewer than two repeated vector clusters were found")

    # Frequency first, then short vectors.  Keep several shells because the
    # primitive translation can be longer than the nearest atom-atom distance.
    clusters.sort(key=lambda c: (-c["count"], c["length"]))
    return clusters[:options.max_vector_clusters], len(vectors)


def _unique_best_indices(idx, residual_norm, mask):
    """Keep at most one detected point per integer lattice site."""
    ids = np.flatnonzero(mask)
    if not len(ids):
        return np.zeros(len(mask), dtype=bool), 0
    keys = np.asarray(idx[ids], dtype=np.int64)
    _, inv = np.unique(keys, axis=0, return_inverse=True)
    keep_ids = []
    for g in range(int(inv.max()) + 1):
        members = ids[inv == g]
        keep_ids.append(members[np.argmin(residual_norm[members])])
    keep = np.zeros(len(mask), dtype=bool)
    keep[np.asarray(keep_ids, int)] = True
    collisions = int(len(ids) - len(keep_ids))
    return keep, collisions


def _assign_to_basis(points, A, r0, inlier_factor):
    """Round to integer sites and return residuals plus unique geometric inliers."""
    pts = np.asarray(points, float)
    if abs(np.linalg.det(A)) < 1e-10:
        return None
    frac = np.linalg.solve(A, (pts - r0).T).T
    idx = np.rint(frac).astype(np.int64)
    pred = idx @ A.T + r0
    residual = pts - pred
    rnorm = np.linalg.norm(residual, axis=1)
    shortest = float(min(np.linalg.norm(A[:, 0]), np.linalg.norm(A[:, 1])))
    if not np.isfinite(shortest) or shortest <= 1e-9:
        return None
    tol = inlier_factor * shortest
    raw = np.isfinite(rnorm) & (rnorm <= tol)
    unique, collisions = _unique_best_indices(idx, rnorm, raw)
    raw_n = int(raw.sum())
    collision_rate = collisions / max(raw_n, 1)
    return dict(idx=idx, residual=residual, rnorm=rnorm, raw=raw, keep=unique,
                collisions=collisions, raw_n=raw_n, collision_rate=collision_rate,
                shortest=shortest, tol=tol)


def _candidate_origins(points, n_trials):
    """Deterministic phase seeds spread through the field of view."""
    pts = np.asarray(points, float)
    if len(pts) <= n_trials:
        return pts
    # Sorting on an irrational-ish projection avoids taking all seeds from one row.
    order = np.argsort(pts[:, 0] + 0.61803398875 * pts[:, 1])
    pick = np.linspace(0, len(order) - 1, n_trials).round().astype(int)
    return pts[order[pick]]


def _score_basis(points, a1, a2, d0, options):
    """Best lattice phase for one candidate pair.

    Score prioritises the number of UNIQUE geometrical inliers and heavily
    penalises duplicate site assignments.  This naturally rejects a basis that
    is too large/small for the detected lattice and can select one dominant
    sublattice when the image contains a multi-atom basis.
    """
    a1, a2 = _reduce_basis(a1, a2)
    A = np.column_stack((a1, a2))
    det = float(abs(np.linalg.det(A)))
    if det < 0.15 * d0 * d0:
        return None
    L1, L2 = np.linalg.norm(a1), np.linalg.norm(a2)
    if min(L1, L2) < options.min_vector_factor * d0:
        return None
    if max(L1, L2) > 1.25 * options.max_shell_factor * d0:
        return None
    sinang = abs(_cross2(a1, a2)) / max(L1 * L2, 1e-12)
    if sinang < options.min_sin_between:
        return None

    best = None
    for r0 in _candidate_origins(points, options.origin_trials):
        assn = _assign_to_basis(points, A, r0, options.lattice_inlier_factor)
        if assn is None or not assn["keep"].any():
            continue
        n_keep = int(assn["keep"].sum())
        med = float(np.median(assn["rnorm"][assn["keep"]]))
        p90 = float(np.percentile(assn["rnorm"][assn["keep"]], 90))
        # Collision penalty is intentionally severe.  A valid sublattice may
        # explain fewer points, but should not stack many detections on one site.
        score = (n_keep
                 - 3.0 * assn["collisions"]
                 - 0.35 * med / max(assn["tol"], 1e-9)
                 - 0.10 * p90 / max(assn["tol"], 1e-9))
        rec = dict(score=float(score), A=A.copy(), r0=np.asarray(r0, float).copy(),
                   n_keep=n_keep, median_residual=med, p90_residual=p90,
                   collision_rate=float(assn["collision_rate"]),
                   collisions=int(assn["collisions"]), cell_area=det)
        if best is None or rec["score"] > best["score"]:
            best = rec
    return best


def robust_initial_lattice(points, options):
    """Search many repeated vector pairs instead of trusting the first two peaks."""
    pts = np.asarray(points, float)
    if len(pts) < options.min_atoms:
        raise LatticeError(f"only {len(pts)} candidate atoms; need {options.min_atoms}")

    d0, nn = _robust_nn_scale(pts)
    clusters, n_vectors = _vector_clusters(pts, d0, options)

    pairs = []
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            a, b = clusters[i], clusters[j]
            sinang = abs(_cross2(a["vector"], b["vector"])) / max(a["length"] * b["length"], 1e-12)
            if sinang < options.min_sin_between:
                continue
            # Search frequent pairs first without hard-coding a neighbour shell.
            merit = a["count"] + b["count"] - 0.02 * (a["length"] + b["length"]) / d0
            pairs.append((merit, a, b))
    pairs.sort(key=lambda x: x[0], reverse=True)
    pairs = pairs[:options.max_candidate_pairs]
    if not pairs:
        raise LatticeError("no non-collinear repeated vector pair was found")

    scored = []
    for _, ca, cb in pairs:
        rec = _score_basis(pts, ca["vector"], cb["vector"], d0, options)
        if rec is not None:
            rec["source_count_a"] = ca["count"]
            rec["source_count_b"] = cb["count"]
            scored.append(rec)
    if not scored:
        raise LatticeError("all candidate lattice bases were geometrically degenerate")

    scored.sort(key=lambda r: r["score"], reverse=True)
    best = scored[0]
    required = max(options.min_atoms,
                   int(np.ceil(options.min_lattice_inlier_fraction * len(pts))))
    if best["n_keep"] < required:
        top = ", ".join(
            f"{r['n_keep']} inliers @ {np.linalg.norm(r['A'][:,0]):.2f}/{np.linalg.norm(r['A'][:,1]):.2f}px"
            for r in scored[:3])
        raise LatticeError(
            f"best robust basis explains only {best['n_keep']}/{len(pts)} unique atoms; "
            f"need {required}. Top candidates: {top}")
    if best["collision_rate"] > options.max_raw_collision_rate:
        raise LatticeError(
            f"best basis still has {best['collision_rate']:.1%} raw index collisions; "
            "detections are incompatible with one dominant lattice/sublattice")

    best.update(d0=d0, nn_median=float(np.median(nn)),
                n_vector_clusters=len(clusters), n_pair_vectors=n_vectors,
                n_basis_candidates=len(scored))
    return best


def robust_refine_lattice(points, initial, options):
    """Iteratively re-index, reject outliers/collisions, and refine A and r0."""
    pts = np.asarray(points, float)
    A = np.asarray(initial["A"], float).copy()
    r0 = np.asarray(initial["r0"], float).copy()
    previous = None

    for _ in range(options.robust_refine_iterations):
        assn = _assign_to_basis(pts, A, r0, options.lattice_inlier_factor)
        if assn is None:
            raise LatticeError("basis became singular during robust refinement")
        keep = assn["keep"]
        if int(keep.sum()) < options.min_atoms:
            raise LatticeError(
                f"robust refinement retained only {int(keep.sum())} unique atoms")

        idx = assn["idx"][keep]
        p = pts[keep]
        design = np.column_stack((idx, np.ones(len(idx))))
        sol, *_ = np.linalg.lstsq(design, p, rcond=None)
        new_A = sol[:2].T
        new_r0 = sol[2]
        if abs(np.linalg.det(new_A)) < 1e-10:
            raise LatticeError("basis collapsed during robust refinement")
        if _cross2(new_A[:, 0], new_A[:, 1]) < 0:
            new_A[:, 1] *= -1
            # Recompute the phase rather than attempting to transform old indices.

        signature = (tuple(np.flatnonzero(keep)),
                     tuple(np.round(new_A.ravel(), 5)),
                     tuple(np.round(new_r0, 5)))
        A, r0 = new_A, new_r0
        if previous == signature:
            break
        previous = signature

    assn = _assign_to_basis(pts, A, r0, options.lattice_inlier_factor)
    if assn is None:
        raise LatticeError("final basis is singular")
    keep = assn["keep"]
    n_keep = int(keep.sum())
    required = max(options.min_atoms,
                   int(np.ceil(options.min_lattice_inlier_fraction * len(pts))))
    if n_keep < required:
        raise LatticeError(
            f"final lattice retains {n_keep}/{len(pts)} unique inliers; need {required}")

    # Refit once more on the exact final unique inliers, then reassign once.
    idx0 = assn["idx"][keep]
    design = np.column_stack((idx0, np.ones(n_keep)))
    sol, *_ = np.linalg.lstsq(design, pts[keep], rcond=None)
    A = sol[:2].T
    r0 = sol[2]
    assn = _assign_to_basis(pts, A, r0, options.lattice_inlier_factor)
    keep = assn["keep"]
    n_keep = int(keep.sum())
    if n_keep < required:
        raise LatticeError(
            f"final reassignment retains {n_keep}/{len(pts)} unique inliers; need {required}")

    # Remove isolated residual catastrophes without clipping a broad physical
    # displacement field.  The MAD gate only tightens the fixed geometric gate
    # when a small number of points sit far outside the main residual cloud.
    rr = assn["rnorm"][keep]
    med = float(np.median(rr))
    mad = float(1.4826 * np.median(np.abs(rr - med))) if len(rr) else 0.0
    adaptive_tol = min(assn["tol"],
                       max(med + 6.0 * max(mad, 1e-9), 0.08 * assn["shortest"]))
    adaptive_raw = assn["raw"] & (assn["rnorm"] <= adaptive_tol)
    adaptive_keep, _ = _unique_best_indices(assn["idx"], assn["rnorm"], adaptive_raw)
    if int(adaptive_keep.sum()) >= required:
        keep = adaptive_keep
        n_keep = int(keep.sum())

    kept_points = pts[keep]
    kept_idx = assn["idx"][keep].astype(int)
    kept_residual = assn["residual"][keep]
    kept_norm = assn["rnorm"][keep]
    shortest = float(assn["shortest"])

    # Scientific gate is applied to the robust inlier set, not to detections that
    # have already been identified as geometrical outliers.
    max_disp = float(kept_norm.max()) if len(kept_norm) else 0.0
    if max_disp > options.max_displacement_factor * shortest:
        raise LatticeError(
            f"robust inlier max displacement {max_disp:.2f}px exceeds "
            f"{options.max_displacement_factor:.2f} x {shortest:.2f}px basis")

    L1, L2 = np.linalg.norm(A[:, 0]), np.linalg.norm(A[:, 1])
    angle = float(np.degrees(np.arccos(np.clip(
        (A[:, 0] @ A[:, 1]) / max(L1 * L2, 1e-12), -1, 1))))
    return dict(
        A=A, r0=r0, indices=kept_idx, displacement=kept_residual,
        source_indices=np.flatnonzero(keep), inlier_mask=keep,
        collisions=0, collision_rate=0.0,
        raw_collisions=int(assn["collisions"]),
        raw_collision_rate=float(assn["collision_rate"]),
        n_input_atoms=int(len(pts)), n_inliers=n_keep,
        inlier_fraction=float(n_keep / max(len(pts), 1)),
        n_dropped=int(len(pts) - n_keep),
        rmse_px=float(np.sqrt(np.mean(kept_norm ** 2))),
        median_residual_px=float(np.median(kept_norm)),
        p90_residual_px=float(np.percentile(kept_norm, 90)),
        max_residual_px=max_disp,
        a1_px=float(L1), a2_px=float(L2), angle_deg=angle,
        cell_area_px2=float(abs(np.linalg.det(A))),
        n_vector_clusters=int(initial["n_vector_clusters"]),
        n_basis_candidates=int(initial["n_basis_candidates"]),
        initial_score=float(initial["score"]),
    )


def strain_from_displacement(points, displacement, d0, shape, options):
    """Locally weighted least squares for the displacement gradient.

    Per atom, fit  u = c + G . dr  over its k nearest neighbours with Gaussian
    weights, then split G into the symmetric strain and the rigid rotation.
    Sign convention: rotation w = 0.5 (du_y/dx - du_x/dy), positive counter-
    clockwise in image (x right, y down) coordinates.
    """
    pts, u = np.asarray(points, float), np.asarray(displacement, float)
    n = len(pts)
    out = {k: np.full(n, np.nan) for k in
           ("exx", "eyy", "exy", "rotation", "dilatation", "shear_max",
            "fit_residual_px", "condition")}
    out["n_neighbours"] = np.zeros(n, dtype=int)
    if n < options.min_neighbours + 1:
        return out

    sigma = max(options.weight_sigma_factor * d0, 1e-6)
    k = min(options.neighbour_k + 1, n)
    tree = cKDTree(pts)
    dist, nbr = tree.query(pts, k=k)

    for i in range(n):
        j = nbr[i][1:]                      # drop self
        dj = dist[i][1:]
        keep = np.isfinite(dj)
        j, dj = j[keep], dj[keep]
        if len(j) < options.min_neighbours:
            continue
        dr = pts[j] - pts[i]
        du = u[j] - u[i]
        w = np.exp(-0.5 * (dj / sigma) ** 2)
        if w.sum() <= 1e-12:
            continue
        design = np.column_stack((np.ones(len(j)), dr[:, 0], dr[:, 1]))
        sw = np.sqrt(w)[:, None]
        Aw, Bw = design * sw, du * sw
        # A collinear neighbourhood (an atom on a single row) gives a rank-2
        # design matrix; the gradient across that row is then unconstrained.
        s = np.linalg.svd(Aw, compute_uv=False)
        cond = float(s[0] / s[-1]) if s[-1] > 1e-12 else np.inf
        if not np.isfinite(cond) or cond > options.max_condition_number:
            out["condition"][i] = cond
            out["n_neighbours"][i] = len(j)
            continue
        sol, *_ = np.linalg.lstsq(Aw, Bw, rcond=None)        # (3, 2)
        gxx, gxy = sol[1, 0], sol[2, 0]      # d ux/dx, d ux/dy
        gyx, gyy = sol[1, 1], sol[2, 1]      # d uy/dx, d uy/dy
        exy = 0.5 * (gxy + gyx)
        out["exx"][i], out["eyy"][i], out["exy"][i] = gxx, gyy, exy
        out["rotation"][i] = 0.5 * (gyx - gxy)
        out["dilatation"][i] = gxx + gyy
        out["shear_max"][i] = np.hypot(0.5 * (gxx - gyy), exy)
        resid = Bw - Aw @ sol
        out["fit_residual_px"][i] = float(np.sqrt(np.mean(np.sum(resid ** 2, axis=1))))
        out["condition"][i] = cond
        out["n_neighbours"][i] = len(j)

    margin = options.edge_margin_factor * d0
    h, w_img = shape
    out["is_edge"] = ~((pts[:, 0] >= margin) & (pts[:, 0] < w_img - margin)
                       & (pts[:, 1] >= margin) & (pts[:, 1] < h - margin))
    return out


def analyse_frame(coords_rc, shape, options):
    """Robust full chain for one image.

    Candidate detections are allowed to contain false positives, missed sites,
    and multiple neighbour shells.  Only the dominant geometrically consistent
    lattice/sublattice is retained for displacement and strain.
    """
    pts_all = to_xy(coords_rc)
    initial = robust_initial_lattice(pts_all, options)
    lat = robust_refine_lattice(pts_all, initial, options)
    pts = pts_all[lat["source_indices"]]
    strain = strain_from_displacement(pts, lat["displacement"], initial["d0"], shape, options)
    return dict(points=pts, all_points=pts_all, d0=initial["d0"], lattice=lat,
                strain=strain, n_directions=initial["n_vector_clusters"])


def frame_summary(result, name, options):
    """Robust statistics only. A single mis-indexed atom produces a huge strain,
    and a mean would carry it into the summary; the median and the IQR-derived
    sigma do not."""
    s, lat = result["strain"], result["lattice"]
    valid = np.isfinite(s["exx"]) & ~s["is_edge"]
    row = dict(sample=name, n_atoms=len(result["points"]),
               n_input_atoms=lat.get("n_input_atoms", len(result["points"])),
               n_lattice_inliers=lat.get("n_inliers", len(result["points"])),
               lattice_inlier_fraction=lat.get("inlier_fraction", 1.0),
               n_dropped_lattice_outliers=lat.get("n_dropped", 0),
               n_valid=int(valid.sum()), d0_px=result["d0"],
               a1_px=lat["a1_px"], a2_px=lat["a2_px"],
               cell_angle_deg=lat["angle_deg"], cell_area_px2=lat["cell_area_px2"],
               lattice_rmse_px=lat["rmse_px"], index_collisions=lat["collisions"],
               raw_index_collision_rate=lat.get("raw_collision_rate", 0.0),
               lattice_p90_residual_px=lat.get("p90_residual_px", np.nan),
               lattice_max_residual_px=lat.get("max_residual_px", np.nan),
               n_vector_clusters=lat.get("n_vector_clusters", np.nan),
               n_basis_candidates=lat.get("n_basis_candidates", np.nan),
               displacement_rms_px=float(np.sqrt(np.mean(
                   np.sum(lat["displacement"] ** 2, axis=1)))))
    for key in ("exx", "eyy", "exy", "rotation", "dilatation", "shear_max"):
        v = s[key][valid]
        v = v[np.isfinite(v)]
        if len(v) < 2:
            row[f"{key}_median"] = row[f"{key}_sigma"] = None
            continue
        q1, q3 = np.percentile(v, (25, 75))
        row[f"{key}_median"] = float(np.median(v))
        row[f"{key}_sigma"] = float(0.7413 * (q3 - q1))   # IQR -> sigma, Gaussian
    if options.lattice_parameter_A is not None:
        row["px_per_A"] = None if not lat["a1_px"] else \
            lat["a1_px"] / options.lattice_parameter_A
    return row


# ---------------------------------------------------------------- figures

def _sym_limits(v, pct):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if not len(v):
        return -1e-3, 1e-3
    lim = float(np.percentile(np.abs(v), 100 - pct))
    lim = max(lim, 1e-6)
    return -lim, lim


def plot_strain_frame(image, result, title, out_png, options, show=False):
    pts, s, lat = result["points"], result["strain"], result["lattice"]
    u = lat["displacement"]
    panels = [("exx", r"$\varepsilon_{xx}$"), ("eyy", r"$\varepsilon_{yy}$"),
              ("exy", r"$\varepsilon_{xy}$"), ("rotation", r"rotation $\omega$"),
              ("dilatation", r"dilatation $\varepsilon_{xx}+\varepsilon_{yy}$"),
              ("shear_max", "max shear")]
    fig, axes = plt.subplots(2, 4, figsize=(19, 8.6))
    ax = axes[0, 0]
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.scatter(pts[:, 0], pts[:, 1], s=6, c="#00d0a0", linewidths=0)
    ax.set_title(f"detections (n={len(pts)})", fontsize=10)

    ax = axes[0, 1]
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    scale = 0.25 * result["d0"] / max(np.abs(u).max(), 1e-9)
    ax.quiver(pts[:, 0], pts[:, 1], u[:, 0] * scale, -u[:, 1] * scale,
              color="#ff5030", angles="xy", scale_units="xy", scale=1, width=.003)
    ax.set_title(f"displacement x{scale:.0f}  (RMS "
                 f"{np.sqrt(np.mean(np.sum(u**2, axis=1))):.3f} px)", fontsize=10)

    for ax, (key, label) in zip(list(axes.flat)[2:], panels):
        v = s[key]
        lo, hi = _sym_limits(v[~s["is_edge"]], options.robust_percentile)
        sc = ax.scatter(pts[:, 0], pts[:, 1], c=v, s=14, cmap="RdBu_r",
                        vmin=lo, vmax=hi, linewidths=0)
        ax.set_facecolor("#f2f2f2")
        fig.colorbar(sc, ax=ax, shrink=.82)
        ax.set_title(label, fontsize=10)
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlim(0, image.shape[1])
        ax.set_ylim(image.shape[0], 0)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show() if show else plt.close(fig)


def plot_strain_across_folds(frame, out_png, show=False):
    """Per-fold strain sigma. On unstrained simulated data this is the noise
    floor of the measurement chain, so lower is better and the comparison
    between architectures is meaningful."""
    keys = [("exx_sigma", r"$\sigma(\varepsilon_{xx})$"),
            ("eyy_sigma", r"$\sigma(\varepsilon_{yy})$"),
            ("exy_sigma", r"$\sigma(\varepsilon_{xy})$"),
            ("displacement_rms_px", "displacement RMS (px)")]
    keys = [(k, l) for k, l in keys if k in frame.columns and frame[k].notna().any()]
    if not keys:
        return
    fig, axes = plt.subplots(1, len(keys), figsize=(4.7 * len(keys), 3.9),
                             squeeze=False)
    for ax, (key, label) in zip(axes[0], keys):
        for (cond, arch), g in frame.groupby(["condition", "arch"]):
            g = g.groupby("fold")[key].median().sort_index()
            ax.plot(g.index, g.values, marker="o", ms=4,
                    ls="-" if cond == "no_n2v" else "--", label=f"{arch} · {cond}")
        ax.set_xlabel("fold")
        ax.set_title(label, fontsize=10)
        ax.set_xticks(sorted(frame["fold"].unique()))
    axes[0, 0].legend(fontsize=7, ncol=2)
    fig.suptitle("Strain measurement noise floor across folds", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show() if show else plt.close(fig)


# ---------------------------------------------------------------- driver

def strain_one_fold(ns, arch, cond, fold, ckpt, base, opts, loc_opts,
                    n_images, out_dir, n_figures, show):
    """Detect, refine, fit the lattice and map strain for one checkpoint."""
    import torch
    names = ns["fold_val_names"](fold, base)
    if not names:
        raise RuntimeError(f"fold {fold} split could not be reconstructed")
    picks = ns["random"].Random(ns["SEED"]).sample(names, min(n_images, len(names)))
    cfg, size = ns["CFG"], ns["CFG"].IMG_SIZE
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model, rows, atom_rows, failures = None, [], [], []
    try:
        model = ns["prep_model"](ns["ARCH_REGISTRY"][arch]())
        ns["load_model_weights"](ckpt, model)
        model.eval()
        for i, nm in enumerate(picks):
            noisy = ns["center_or_resize"](
                ns["norm01"](ns["imread_gray"](ns["noisy_map"][nm])), size)
            x = torch.from_numpy(noisy)[None, None].float().to(
                ns["DEVICE"], memory_format=torch.channels_last)
            with torch.inference_mode():
                den, lg = model(x)
            den = den[0, 0].float().clamp(0, 1).cpu().numpy()
            prob = torch.sigmoid(lg.float())[0, 0].cpu().numpy()
            peaks = ns["detect_centres"](prob, loc_opts)
            src = den if loc_opts.fit_source == "denoised" else noisy
            refined, accepted, _, _, _ = ns["refine_centres"](src, peaks, loc_opts)

            # refine_centres() keeps the original detected centre whenever a
            # sub-pixel Gaussian fit is rejected. Therefore use ALL finite
            # refined centres here: accepted detections get sub-pixel positions,
            # rejected fits retain their original peak coordinates. Filtering
            # to accepted-only centres can collapse a perfectly usable lattice
            # from dozens of detections to only a handful.
            refined = np.asarray(refined, dtype=float)
            accepted = np.asarray(accepted, dtype=bool)
            finite = np.isfinite(refined).all(axis=1)
            coords = refined[finite]

            try:
                res = analyse_frame(coords, noisy.shape, opts)
            except LatticeError as exc:
                failures.append(dict(
                    sample=nm,
                    reason=str(exc),
                    n_atoms=int(len(coords)),
                    n_peaks=int(len(peaks)),
                    n_accepted_fits=int(accepted.sum()),
                ))
                continue
            row = frame_summary(res, nm, opts)
            row.update(arch=arch, condition=cond, fold=fold,
                       n_accepted_fits=int(accepted.sum()), n_peaks=int(len(peaks)))
            rows.append(row)

            s, lat = res["strain"], res["lattice"]
            for j in range(len(res["points"])):
                atom_rows.append(dict(
                    sample=nm, arch=arch, condition=cond, fold=fold, atom_id=j,
                    x_px=res["points"][j, 0], y_px=res["points"][j, 1],
                    m=lat["indices"][j, 0], n=lat["indices"][j, 1],
                    ux_px=lat["displacement"][j, 0], uy_px=lat["displacement"][j, 1],
                    exx=s["exx"][j], eyy=s["eyy"][j], exy=s["exy"][j],
                    rotation=s["rotation"][j], dilatation=s["dilatation"][j],
                    shear_max=s["shear_max"][j], n_neighbours=s["n_neighbours"][j],
                    fit_residual_px=s["fit_residual_px"][j], is_edge=s["is_edge"][j]))

            if i < n_figures:
                plot_strain_frame(
                    noisy, res,
                    f"{arch} · {cond} · fold {fold} · {nm}\n"
                    f"a1 {lat['a1_px']:.2f} px, a2 {lat['a2_px']:.2f} px, "
                    f"angle {lat['angle_deg']:.1f} deg, "
                    f"lattice RMSE {lat['rmse_px']:.3f} px",
                    out_dir / f"{arch}_fold{fold}_{nm}_strain.png", opts, show)
    finally:
        if model is not None:
            del model
        ns["free_gpu"]()

    if failures:
        print(f"      {len(failures)} frame(s) had no trustworthy lattice: "
              f"{failures[0]['reason']}"
              + (f" (and {len(failures)-1} more)" if len(failures) > 1 else ""))
    return pd.DataFrame(rows), pd.DataFrame(atom_rows), failures


def run_strain_all_folds(ns, opts=None, loc_opts=None, archs=None, conditions=None,
                         folds=None, n_images=4, n_figures=1, show=False):
    """Strain mapping on every fold that has a checkpoint."""
    required = ("CFG", "ARCH_REGISTRY", "DEVICE", "common", "SEED", "noisy_map",
                "CONDITION_USE_N2V", "arch_dir", "fold_base_on_disk",
                "rebuild_ckpt_index", "detect_centres", "refine_centres",
                "prep_model", "load_model_weights", "center_or_resize",
                "norm01", "imread_gray", "free_gpu", "fold_val_names",
                "random", "LocalizationOptions")
    missing = [k for k in required if k not in ns]
    if missing:
        raise RuntimeError("Run sections 1-10 first. Missing: " + ", ".join(missing))

    opts = opts or StrainOptions()
    loc_opts = loc_opts or ns.get("LOC_OPTIONS") or ns["LocalizationOptions"](
        n_pool=None, probability_threshold=0.50, min_distance_px=4.0,
        fit_source="denoised", inference_precision="fp32", pixel_size_A=None)

    if not ns.get("CKPT_INDEX"):
        ns["rebuild_ckpt_index"]()
    index = ns["CKPT_INDEX"]
    archs = archs or ns.get("RUN_ARCHS") or list(ns["CFG"].ARCHS)
    conditions = conditions or [c for c in ns["CONDITION_USE_N2V"] if index.get(c)]

    jobs = []
    for cond in conditions:
        base = ns["fold_base_on_disk"](cond)
        for (arch, fold), ckpt in sorted(index.get(cond, {}).items()):
            if arch not in archs:
                continue
            if folds is not None and int(fold) not in {int(f) for f in folds}:
                continue
            jobs.append((cond, arch, int(fold), Path(ckpt), base))
    if not jobs:
        raise RuntimeError("No fold checkpoints found; check CKPT_INDEX in section 9a.")

    run_id = time.strftime("%Y%m%dT%H%M%S")
    root = Path(ns["CFG"].RESULTS_ROOT) / "strain" / run_id
    root.mkdir(parents=True, exist_ok=True)
    print(f"Strain run {run_id}: {len(jobs)} fold(s), {n_images} image(s) each")

    all_rows, all_atoms, all_fail = [], [], []
    for i, (cond, arch, fold, ckpt, base) in enumerate(jobs, 1):
        print(f"\n[{i}/{len(jobs)}] {cond} · {arch} · fold {fold}", flush=True)
        out = (Path(ns["arch_dir"](arch, ns["CONDITION_USE_N2V"][cond]))
               / "strain" / run_id / f"fold_{fold:02d}")
        t0 = time.time()
        try:
            rows, atoms, fails = strain_one_fold(
                ns, arch, cond, fold, ckpt, base, opts, loc_opts,
                n_images, out, n_figures, show)
        except Exception as exc:
            print(f"      failed: {exc.__class__.__name__}: {exc}")
            all_fail.append(dict(condition=cond, arch=arch, fold=fold,
                                 error=f"{exc.__class__.__name__}: {exc}"))
            continue
        all_fail += [dict(f, condition=cond, arch=arch, fold=fold) for f in fails]
        if rows.empty:
            print("      no frame yielded a trustworthy lattice")
            continue
        all_rows.append(rows)
        all_atoms.append(atoms)
        print(f"      {len(rows)} frame(s)  a1 {rows['a1_px'].median():.2f} px  "
              f"inliers {100*rows['lattice_inlier_fraction'].median():.0f}%  "
              f"lattice RMSE {rows['lattice_rmse_px'].median():.3f} px  "
              f"sigma(exx) {rows['exx_sigma'].median():.2e}  "
              f"[{time.time()-t0:.0f}s]")

    if not all_rows:
        # Preserve diagnostics even when every frame fails. Previously the
        # exception happened before strain_failures.csv could be written.
        if all_fail:
            pd.DataFrame(all_fail).to_csv(root / "strain_failures.csv", index=False)
        raise RuntimeError(
            "No fold produced a usable lattice fit. "
            f"Diagnostics saved to {root / 'strain_failures.csv'}"
        )
    frames = pd.concat(all_rows, ignore_index=True)
    atoms = pd.concat(all_atoms, ignore_index=True)
    frames.to_csv(root / "strain_per_frame.csv", index=False)
    atoms.to_csv(root / "strain_per_atom.csv", index=False)
    if all_fail:
        pd.DataFrame(all_fail).to_csv(root / "strain_failures.csv", index=False)

    keys = [k for k in frames.columns if k.endswith(("_sigma", "_median"))] + \
           ["lattice_rmse_px", "displacement_rms_px", "a1_px", "a2_px",
            "cell_angle_deg", "n_atoms"]
    agg = []
    for (cond, arch), g in frames.groupby(["condition", "arch"]):
        r = dict(condition=cond, arch=arch, n_folds=int(g["fold"].nunique()),
                 n_frames=len(g), n_atoms=int(g["n_atoms"].sum()))
        for k in keys:
            v = pd.to_numeric(g[k], errors="coerce").dropna()
            r[k] = float(v.median()) if len(v) else None
        agg.append(r)
    across = pd.DataFrame(agg).sort_values(["condition", "arch"]).reset_index(drop=True)
    across.to_csv(root / "strain_across_folds.csv", index=False)
    plot_strain_across_folds(frames, root / "strain_across_folds.png", show)

    notes = [
        "The reference lattice is fitted to each image, so the mean strain is zero "
        "by construction; only spatial variation within an image is measured.",
        "Strain is dimensionless and does not depend on the pixel size; only a "
        "lattice parameter in Angstrom would.",
        "On unstrained simulated data the reported sigma is the noise floor of the "
        "detect-refine-fit chain, not physical strain.",
        "Frames whose lattice fit failed are excluded and listed in "
        "strain_failures.csv; excluding them biases the summary toward images the "
        "model segmented well.",
        "Edge atoms are flagged and excluded from the per-frame statistics because "
        "their gradient fits are one-sided.",
    ]
    if opts.lattice_parameter_A is None:
        notes.append("No lattice parameter supplied, so all lengths are in pixels.")
    with open(root / "strain_summary.json", "w", encoding="utf-8") as f:
        json.dump(dict(run_id=run_id, options=asdict(opts),
                       localization_options=asdict(loc_opts),
                       dataset_hash=getattr(ns["CFG"], "DATASET_HASH", None),
                       n_images_per_fold=n_images,
                       per_condition_arch=across.to_dict("records"),
                       n_failures=len(all_fail), notes=notes),
                  f, indent=2, default=str)

    print("\n=== across folds (median over frames) ===")
    cols = [c for c in ("condition", "arch", "n_folds", "n_frames", "a1_px",
                        "cell_angle_deg", "lattice_rmse_px", "displacement_rms_px",
                        "exx_sigma", "eyy_sigma", "exy_sigma") if c in across.columns]
    print(across[cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print(f"\n-> {root}")
    ns.update(STRAIN_RUN_ID=run_id, STRAIN_ROOT=str(root), STRAIN_FRAMES=frames,
              STRAIN_ATOMS=atoms, STRAIN_ACROSS=across, STRAIN_OPTIONS=opts)
    return frames, across


if __name__ == "__main__":
    STRAIN_OPTIONS = StrainOptions(
        neighbour_k=12, weight_sigma_factor=1.0,
        lattice_parameter_A=None, calibration_source="")

    # Diagnostic first.  Do not burn through all folds until the inferred basis
    # is visibly sensible on at least one representative checkpoint.
    run_strain_all_folds(
        globals(), STRAIN_OPTIONS,
        folds=[1], conditions=["no_n2v"],
        n_images=4, n_figures=2, show=True)

    # Full sweep after the diagnostic passes:
    # run_strain_all_folds(globals(), STRAIN_OPTIONS,
    #                      n_images=4, n_figures=1, show=False)


In [ ]:
# ============================================================================ #
# SECTION 14 — Reference-space robust strain refinement + validation
#
# Run AFTER Section 13 robust lattice mapping.
#
# Improvements:
#   1. Strain gradient is evaluated in ideal/reference lattice coordinates.
#   2. Neighbour distance is based on fitted lattice spacing, not noisy raw NN.
#   3. One-sided / boundary neighbourhoods are rejected geometrically.
#   4. Local fit uses robust Huber reweighting.
#   5. Coordinate scaling improves LS conditioning.
#   6. Max shear gets a physically correct non-negative colour scale.
#   7. Extra per-frame quality metrics are reported.
#
# Existing run_strain_all_folds() is reused unchanged.
# ============================================================================ #

from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree


# --------------------------------------------------------------------------- #
# Tunable Section-14 constants
# --------------------------------------------------------------------------- #

S14_MAX_ANGULAR_GAP_DEG = 190.0
S14_HUBER_C = 1.5
S14_ROBUST_ITERS = 2
S14_MIN_VALID_FRACTION = 0.15


# --------------------------------------------------------------------------- #
# Utilities
# --------------------------------------------------------------------------- #

def _s14_robust_sigma(x):
    """MAD-based robust scale."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) < 2:
        return np.nan

    med = np.median(x)
    mad = np.median(np.abs(x - med))

    return float(1.4826 * mad)


def _s14_angular_gap(dr):
    """
    Largest empty angular sector around a central atom.

    Interior 2-D lattice sites should have neighbours surrounding them.
    Boundary sites typically have a very large empty sector.
    """
    dr = np.asarray(dr, dtype=float)

    if len(dr) < 3:
        return np.inf

    theta = np.mod(
        np.arctan2(dr[:, 1], dr[:, 0]),
        2.0 * np.pi,
    )

    theta = np.sort(theta)

    gaps = np.diff(
        np.r_[theta, theta[0] + 2.0 * np.pi]
    )

    return float(np.max(gaps))


def _s14_weighted_solve(X, Y, weights):
    """Weighted least squares with safe failure handling."""
    weights = np.asarray(weights, dtype=float)

    good = (
        np.isfinite(weights)
        & (weights > 1e-12)
        & np.isfinite(X).all(axis=1)
        & np.isfinite(Y).all(axis=1)
    )

    if good.sum() < 3:
        return None, np.inf, None

    X = X[good]
    Y = Y[good]
    weights = weights[good]

    sw = np.sqrt(weights)[:, None]

    Xw = X * sw
    Yw = Y * sw

    s = np.linalg.svd(
        Xw,
        compute_uv=False,
    )

    cond = (
        float(s[0] / s[-1])
        if len(s) and s[-1] > 1e-12
        else np.inf
    )

    try:
        coef, *_ = np.linalg.lstsq(
            Xw,
            Yw,
            rcond=None,
        )
    except np.linalg.LinAlgError:
        return None, np.inf, None

    residual = Y - X @ coef

    return coef, cond, residual


# --------------------------------------------------------------------------- #
# Improved strain calculation
# --------------------------------------------------------------------------- #

def strain_from_displacement(
    reference,
    displacement,
    lattice_scale,
    shape,
    options,
):
    """
    Compute local strain from displacement u(R).

    IMPORTANT:
        reference = ideal fitted lattice positions R
        displacement = measured position - reference position

    Therefore the derivative is

        du / dR

    rather than du/d(measured coordinate).

    This avoids coupling localization noise into both the independent and
    dependent variables of the local gradient regression.
    """

    ref = np.asarray(reference, dtype=float)
    u = np.asarray(displacement, dtype=float)

    if (
        ref.ndim != 2
        or ref.shape[1] != 2
        or u.shape != ref.shape
    ):
        raise ValueError(
            "reference and displacement must both have shape (N, 2)"
        )

    n = len(ref)

    keys = (
        "exx",
        "eyy",
        "exy",
        "rotation",
        "dilatation",
        "shear_max",
        "fit_residual_px",
        "condition",
        "max_angular_gap_deg",
    )

    out = {
        k: np.full(n, np.nan, dtype=float)
        for k in keys
    }

    out["n_neighbours"] = np.zeros(n, dtype=int)
    out["is_edge"] = np.ones(n, dtype=bool)
    out["fit_valid"] = np.zeros(n, dtype=bool)

    if n < options.min_neighbours + 1:
        return out

    lattice_scale = float(lattice_scale)

    if (
        not np.isfinite(lattice_scale)
        or lattice_scale <= 0
    ):
        raise ValueError("lattice_scale must be positive")

    # Neighbourhood geometry comes from the IDEAL reference lattice.
    tree = cKDTree(ref)

    k = min(
        int(options.neighbour_k) + 1,
        n,
    )

    dist, nbr = tree.query(
        ref,
        k=k,
    )

    # Gaussian width should be tied to the fitted translation vector,
    # not the potentially corrupted raw nearest-neighbour statistic.
    sigma = max(
        float(options.weight_sigma_factor) * lattice_scale,
        1e-6,
    )

    h, w_img = shape

    image_margin = (
        float(options.edge_margin_factor)
        * lattice_scale
    )

    max_gap_allowed = np.deg2rad(
        S14_MAX_ANGULAR_GAP_DEG
    )

    for i in range(n):

        # --------------------------------------------------------------- #
        # Gather neighbours
        # --------------------------------------------------------------- #

        j = np.asarray(
            nbr[i, 1:],
            dtype=int,
        )

        dj = np.asarray(
            dist[i, 1:],
            dtype=float,
        )

        keep = (
            np.isfinite(dj)
            & (dj > 1e-9)
        )

        j = j[keep]
        dj = dj[keep]

        out["n_neighbours"][i] = len(j)

        if len(j) < options.min_neighbours:
            continue

        dr = ref[j] - ref[i]
        du = u[j] - u[i]

        # --------------------------------------------------------------- #
        # Reject one-sided neighbourhoods
        # --------------------------------------------------------------- #

        max_gap = _s14_angular_gap(dr)

        out["max_angular_gap_deg"][i] = np.degrees(
            max_gap
        )

        if (
            not np.isfinite(max_gap)
            or max_gap > max_gap_allowed
        ):
            continue

        # --------------------------------------------------------------- #
        # Reject actual image-boundary sites too
        # --------------------------------------------------------------- #

        x0, y0 = ref[i]

        if (
            x0 < image_margin
            or x0 >= w_img - image_margin
            or y0 < image_margin
            or y0 >= h - image_margin
        ):
            continue

        out["is_edge"][i] = False

        # --------------------------------------------------------------- #
        # Dimensionless reference coordinates improve conditioning.
        #
        # q = dr / a
        #
        # Fitting:
        #     du = B q
        #
        # gives:
        #     du/dr = B/a
        # --------------------------------------------------------------- #

        q = dr / lattice_scale

        X = np.column_stack(
            (
                np.ones(len(j)),
                q[:, 0],
                q[:, 1],
            )
        )

        spatial_w = np.exp(
            -0.5 * (dj / sigma) ** 2
        )

        weights = spatial_w.copy()

        coef = None
        residual = None
        cond = np.inf

        # --------------------------------------------------------------- #
        # Robust local regression
        # --------------------------------------------------------------- #

        for robust_iter in range(
            max(1, S14_ROBUST_ITERS)
        ):

            coef, cond, residual = _s14_weighted_solve(
                X,
                du,
                weights,
            )

            if coef is None:
                break

            if robust_iter == S14_ROBUST_ITERS - 1:
                break

            rmag = np.linalg.norm(
                residual,
                axis=1,
            )

            rmed = np.median(rmag)
            scale = _s14_robust_sigma(rmag)

            if (
                not np.isfinite(scale)
                or scale < 1e-8
            ):
                break

            threshold = (
                S14_HUBER_C * scale
            )

            delta = np.abs(
                rmag - rmed
            )

            huber = np.ones_like(delta)

            bad = delta > threshold

            huber[bad] = (
                threshold
                / np.maximum(delta[bad], 1e-12)
            )

            weights = spatial_w * huber

        if coef is None:
            continue

        out["condition"][i] = cond

        if (
            not np.isfinite(cond)
            or cond > options.max_condition_number
        ):
            continue

        # --------------------------------------------------------------- #
        # Convert dimensionless-coordinate coefficients into du/dR
        # --------------------------------------------------------------- #

        grad = coef[1:, :].T / lattice_scale

        # grad =
        #
        # [ du_x/dx   du_x/dy ]
        # [ du_y/dx   du_y/dy ]

        dux_dx = float(grad[0, 0])
        dux_dy = float(grad[0, 1])

        duy_dx = float(grad[1, 0])
        duy_dy = float(grad[1, 1])

        exx = dux_dx
        eyy = duy_dy

        exy = 0.5 * (
            dux_dy + duy_dx
        )

        omega = 0.5 * (
            duy_dx - dux_dy
        )

        dilatation = (
            exx + eyy
        )

        shear_max = np.hypot(
            0.5 * (exx - eyy),
            exy,
        )

        out["exx"][i] = exx
        out["eyy"][i] = eyy
        out["exy"][i] = exy

        out["rotation"][i] = omega
        out["dilatation"][i] = dilatation
        out["shear_max"][i] = shear_max

        if residual is not None:
            out["fit_residual_px"][i] = float(
                np.sqrt(
                    np.mean(
                        np.sum(
                            residual**2,
                            axis=1,
                        )
                    )
                )
            )

        out["fit_valid"][i] = True

    return out


# --------------------------------------------------------------------------- #
# Override analyse_frame from Section 13
# --------------------------------------------------------------------------- #

def analyse_frame(
    coords_rc,
    shape,
    options,
):
    """
    Robust lattice extraction from Section 13 +
    reference-space strain refinement from Section 14.
    """

    pts_all = to_xy(coords_rc)

    initial = robust_initial_lattice(
        pts_all,
        options,
    )

    lat = robust_refine_lattice(
        pts_all,
        initial,
        options,
    )

    source_indices = np.asarray(
        lat["source_indices"],
        dtype=int,
    )

    pts = pts_all[source_indices]

    indices = np.asarray(
        lat["indices"],
        dtype=float,
    )

    A = np.asarray(
        lat["A"],
        dtype=float,
    )

    r0 = np.asarray(
        lat["r0"],
        dtype=float,
    )

    # Ideal/reference lattice coordinate:
    #
    # R = A n + r0
    #
    reference = (
        indices @ A.T
        + r0
    )

    displacement = (
        pts - reference
    )

    # Protect against any stale displacement stored by Section 13.
    lat["displacement"] = displacement
    lat["reference_points"] = reference

    a1 = np.linalg.norm(
        A[:, 0]
    )

    a2 = np.linalg.norm(
        A[:, 1]
    )

    lattice_scale = float(
        min(a1, a2)
    )

    strain = strain_from_displacement(
        reference,
        displacement,
        lattice_scale,
        shape,
        options,
    )

    return dict(
        points=pts,
        reference_points=reference,
        all_points=pts_all,
        d0=initial["d0"],
        lattice_scale=lattice_scale,
        lattice=lat,
        strain=strain,
        n_directions=initial["n_vector_clusters"],
    )


# --------------------------------------------------------------------------- #
# Improved frame summary
# --------------------------------------------------------------------------- #

def frame_summary(
    result,
    name,
    options,
):
    """
    Robust per-frame statistics.

    Median + IQR sigma are retained for compatibility with Section 13.
    """

    s = result["strain"]
    lat = result["lattice"]

    valid = (
        np.asarray(s["fit_valid"], dtype=bool)
        & np.isfinite(s["exx"])
        & np.isfinite(s["eyy"])
        & np.isfinite(s["exy"])
        & ~np.asarray(s["is_edge"], dtype=bool)
    )

    n_atoms = len(result["points"])
    n_valid = int(valid.sum())

    displacement = np.asarray(
        lat["displacement"],
        dtype=float,
    )

    row = dict(
        sample=name,

        n_atoms=n_atoms,

        n_input_atoms=lat.get(
            "n_input_atoms",
            n_atoms,
        ),

        n_lattice_inliers=lat.get(
            "n_inliers",
            n_atoms,
        ),

        lattice_inlier_fraction=lat.get(
            "inlier_fraction",
            1.0,
        ),

        n_dropped_lattice_outliers=lat.get(
            "n_dropped",
            0,
        ),

        n_valid=n_valid,

        valid_strain_fraction=(
            n_valid / max(n_atoms, 1)
        ),

        edge_fraction=float(
            np.mean(s["is_edge"])
        ) if n_atoms else np.nan,

        d0_px=result["d0"],

        lattice_scale_px=result[
            "lattice_scale"
        ],

        a1_px=lat["a1_px"],
        a2_px=lat["a2_px"],

        cell_angle_deg=lat[
            "angle_deg"
        ],

        cell_area_px2=lat[
            "cell_area_px2"
        ],

        lattice_rmse_px=lat[
            "rmse_px"
        ],

        lattice_rmse_fraction=(
            lat["rmse_px"]
            / max(
                result["lattice_scale"],
                1e-12,
            )
        ),

        index_collisions=lat[
            "collisions"
        ],

        raw_index_collision_rate=lat.get(
            "raw_collision_rate",
            0.0,
        ),

        lattice_p90_residual_px=lat.get(
            "p90_residual_px",
            np.nan,
        ),

        lattice_max_residual_px=lat.get(
            "max_residual_px",
            np.nan,
        ),

        n_vector_clusters=lat.get(
            "n_vector_clusters",
            np.nan,
        ),

        n_basis_candidates=lat.get(
            "n_basis_candidates",
            np.nan,
        ),

        displacement_rms_px=float(
            np.sqrt(
                np.mean(
                    np.sum(
                        displacement**2,
                        axis=1,
                    )
                )
            )
        ),
    )

    # Median local-fit residual
    local_resid = np.asarray(
        s["fit_residual_px"],
        dtype=float,
    )

    local_resid = local_resid[
        valid
        & np.isfinite(local_resid)
    ]

    row["local_fit_residual_median_px"] = (
        float(np.median(local_resid))
        if len(local_resid)
        else None
    )

    # Strain and rotation statistics
    for key in (
        "exx",
        "eyy",
        "exy",
        "rotation",
        "dilatation",
        "shear_max",
    ):

        v = np.asarray(
            s[key],
            dtype=float,
        )[valid]

        v = v[
            np.isfinite(v)
        ]

        if len(v) < 2:
            row[f"{key}_median"] = None
            row[f"{key}_sigma"] = None
            continue

        q1, q3 = np.percentile(
            v,
            (25, 75),
        )

        row[f"{key}_median"] = float(
            np.median(v)
        )

        row[f"{key}_sigma"] = float(
            0.7413 * (q3 - q1)
        )

    if (
        options.lattice_parameter_A
        is not None
    ):
        row["px_per_A"] = (
            lat["a1_px"]
            / options.lattice_parameter_A
        )

    return row


# --------------------------------------------------------------------------- #
# Improved plot
# --------------------------------------------------------------------------- #

def plot_strain_frame(
    image,
    result,
    title,
    out_png,
    options,
    show=False,
):
    """
    Section-14 strain diagnostic plot.

    Diverging maps:
        exx, eyy, exy, rotation, dilatation

    Sequential map:
        max shear >= 0
    """

    pts = np.asarray(
        result["points"],
        dtype=float,
    )

    ref = np.asarray(
        result["reference_points"],
        dtype=float,
    )

    s = result["strain"]
    lat = result["lattice"]

    u = np.asarray(
        lat["displacement"],
        dtype=float,
    )

    valid = (
        np.asarray(s["fit_valid"], dtype=bool)
        & ~np.asarray(s["is_edge"], dtype=bool)
    )

    panels = [
        (
            "exx",
            r"$\varepsilon_{xx}$",
        ),
        (
            "eyy",
            r"$\varepsilon_{yy}$",
        ),
        (
            "exy",
            r"$\varepsilon_{xy}$",
        ),
        (
            "rotation",
            r"rotation $\omega$",
        ),
        (
            "dilatation",
            r"dilatation "
            r"$\varepsilon_{xx}+\varepsilon_{yy}$",
        ),
        (
            "shear_max",
            "max shear",
        ),
    ]

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(19, 8.8),
    )

    # --------------------------------------------------------------- #
    # Detection panel
    # --------------------------------------------------------------- #

    ax = axes[0, 0]

    ax.imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    # geometrically valid strain sites
    if valid.any():
        ax.scatter(
            pts[valid, 0],
            pts[valid, 1],
            s=10,
            c="lime",
            linewidths=0,
            label="strain-valid",
        )

    # lattice sites excluded from local strain
    rejected = ~valid

    if rejected.any():
        ax.scatter(
            pts[rejected, 0],
            pts[rejected, 1],
            s=10,
            c="orange",
            linewidths=0,
            alpha=0.65,
            label="edge/local reject",
        )

    ax.set_title(
        f"lattice atoms n={len(pts)}, "
        f"strain-valid n={valid.sum()}",
        fontsize=10,
    )

    # --------------------------------------------------------------- #
    # Displacement panel
    # --------------------------------------------------------------- #

    ax = axes[0, 1]

    ax.imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    max_u = (
        np.max(
            np.linalg.norm(
                u,
                axis=1,
            )
        )
        if len(u)
        else 0.0
    )

    scale = (
        0.30
        * result["lattice_scale"]
        / max(max_u, 1e-9)
    )

    ax.quiver(
        ref[:, 0],
        ref[:, 1],
        u[:, 0] * scale,
        -u[:, 1] * scale,
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.003,
    )

    displacement_rms = float(
        np.sqrt(
            np.mean(
                np.sum(
                    u**2,
                    axis=1,
                )
            )
        )
    )

    ax.set_title(
        f"displacement ×{scale:.1f}  "
        f"(RMS {displacement_rms:.3f} px)",
        fontsize=10,
    )

    # --------------------------------------------------------------- #
    # Field panels
    # --------------------------------------------------------------- #

    for ax, (
        key,
        label,
    ) in zip(
        list(axes.flat)[2:],
        panels,
    ):

        v = np.asarray(
            s[key],
            dtype=float,
        )

        finite_valid = (
            valid
            & np.isfinite(v)
        )

        # Plot rejected sites faintly first
        if (~finite_valid).any():
            ax.scatter(
                pts[~finite_valid, 0],
                pts[~finite_valid, 1],
                s=8,
                c="lightgray",
                linewidths=0,
                alpha=0.45,
            )

        if not finite_valid.any():
            ax.set_title(
                label + " (no valid sites)",
                fontsize=10,
            )
            continue

        vv = v[finite_valid]

        if key == "shear_max":

            hi = float(
                np.percentile(
                    vv,
                    98,
                )
            )

            hi = max(
                hi,
                1e-6,
            )

            sc = ax.scatter(
                pts[finite_valid, 0],
                pts[finite_valid, 1],
                c=vv,
                s=18,
                cmap="viridis",
                vmin=0.0,
                vmax=hi,
                linewidths=0,
            )

        else:

            lim = float(
                np.percentile(
                    np.abs(vv),
                    98,
                )
            )

            lim = max(
                lim,
                1e-6,
            )

            sc = ax.scatter(
                pts[finite_valid, 0],
                pts[finite_valid, 1],
                c=vv,
                s=18,
                cmap="RdBu_r",
                vmin=-lim,
                vmax=lim,
                linewidths=0,
            )

        fig.colorbar(
            sc,
            ax=ax,
            shrink=0.82,
        )

        ax.set_title(
            label,
            fontsize=10,
        )

    # --------------------------------------------------------------- #
    # Common formatting
    # --------------------------------------------------------------- #

    for ax in axes.flat:

        ax.set_xticks([])
        ax.set_yticks([])

        ax.set_xlim(
            0,
            image.shape[1],
        )

        ax.set_ylim(
            image.shape[0],
            0,
        )

    a1 = lat["a1_px"]
    a2 = lat["a2_px"]

    fig.suptitle(
        title
        + "\n"
        + f"reference-space strain · "
        + f"valid {valid.sum()}/{len(valid)} · "
        + f"a1={a1:.2f}px · "
        + f"a2={a2:.2f}px",
        fontsize=12,
    )

    plt.tight_layout()

    plt.savefig(
        out_png,
        dpi=180,
        bbox_inches="tight",
    )

    if show:
        plt.show()
    else:
        plt.close(fig)


# --------------------------------------------------------------------------- #
# Section-14 convenience runner
# --------------------------------------------------------------------------- #

def run_section14(
    *,
    folds=None,
    conditions=None,
    archs=None,
    n_images=12,
    n_figures=2,
    show=False,
):
    """
    Re-run Section 13 driver using Section-14 strain functions.

    Examples
    --------
    Smoke test:
        run_section14(
            folds=[1],
            conditions=["no_n2v"],
            n_images=12,
            n_figures=3,
            show=True,
        )

    Full run:
        run_section14(
            n_images=20,
            n_figures=1,
            show=False,
        )
    """

    required = (
        "STRAIN_OPTIONS",
        "run_strain_all_folds",
        "robust_initial_lattice",
        "robust_refine_lattice",
        "to_xy",
    )

    missing = [
        x
        for x in required
        if x not in globals()
    ]

    if missing:
        raise RuntimeError(
            "Run Section 13 robust-v2 first. "
            "Missing: "
            + ", ".join(missing)
        )

    print(
        "\n"
        "============================================================\n"
        "SECTION 14 — reference-space robust strain\n"
        "============================================================"
    )

    frames, across = run_strain_all_folds(
        globals(),
        STRAIN_OPTIONS,
        archs=archs,
        conditions=conditions,
        folds=folds,
        n_images=n_images,
        n_figures=n_figures,
        show=show,
    )

    # --------------------------------------------------------------- #
    # Additional quality summary
    # --------------------------------------------------------------- #

    if len(frames):

        quality_cols = [
            c
            for c in (
                "condition",
                "arch",
                "fold",
                "sample",
                "n_input_atoms",
                "n_atoms",
                "n_valid",
                "valid_strain_fraction",
                "lattice_inlier_fraction",
                "lattice_rmse_px",
                "lattice_rmse_fraction",
                "displacement_rms_px",
                "local_fit_residual_median_px",
                "exx_sigma",
                "eyy_sigma",
                "exy_sigma",
            )
            if c in frames.columns
        ]

        print(
            "\n=== Section 14 quality diagnostics ==="
        )

        print(
            frames[quality_cols]
            .to_string(
                index=False,
                float_format=lambda x: f"{x:.4g}",
            )
        )

        group_cols = [
            "condition",
            "arch",
        ]

        metrics = [
            c
            for c in (
                "valid_strain_fraction",
                "lattice_inlier_fraction",
                "lattice_rmse_fraction",
                "displacement_rms_px",
                "local_fit_residual_median_px",
                "exx_sigma",
                "eyy_sigma",
                "exy_sigma",
            )
            if c in frames.columns
        ]

        summary14 = (
            frames
            .groupby(group_cols)[metrics]
            .median()
            .reset_index()
        )

        print(
            "\n=== Section 14 median quality by model ==="
        )

        print(
            summary14.to_string(
                index=False,
                float_format=lambda x: f"{x:.4g}",
            )
        )

        globals()[
            "STRAIN14_SUMMARY"
        ] = summary14

    globals()[
        "STRAIN14_FRAMES"
    ] = frames

    globals()[
        "STRAIN14_ACROSS"
    ] = across

    return frames, across


print(
    "Section 14 loaded.\n"
    "Run the smoke test below before the full 24-fold sweep."
)

In [ ]:
frames14, across14 = run_section14(
    folds=[1],
    conditions=["no_n2v"],
    n_images=12,
    n_figures=3,
    show=True,
)

In [ ]:
# ================================================================
# SVG + PNG figure export
# ================================================================

from pathlib import Path
import matplotlib.pyplot as plt


def save_figure_png_svg(fig, out_path, dpi=180):
    """
    Save the same matplotlib figure as:
        *.png  -> raster preview
        *.svg  -> publication-quality vector figure
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    png = out_path.with_suffix(".png")
    svg = out_path.with_suffix(".svg")

    fig.savefig(
        png,
        dpi=dpi,
        bbox_inches="tight",
    )

    fig.savefig(
        svg,
        format="svg",
        bbox_inches="tight",
    )

    return png, svg

## 12. Final report

In [ ]:
# ============================================================
# 12. Final report, assembled from on-disk artifacts (all archs, both conditions)
# ============================================================
REPORT_ROOT = CFG.RESULTS_ROOT
REPORT_CONDS = [k for k, on in (("no_n2v", RUN_NO_N2V),
                                ("with_n2v", RUN_WITH_N2V)) if on] or ["with_n2v"]


def _write_report_json(obj, path):
    """default=str because the assembled report mixes numpy scalars, bools,
    Paths and None; default=float would raise on those."""
    tmp = str(path) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)


def _read_json(path):
    path = Path(path)
    if path.is_file():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"unreadable: {path} ({exc})")
    return None


def _first_existing(*paths):
    for p in paths:
        if p and Path(p).is_file():
            return Path(p)
    return None


def _num(frame, cols):
    for c in cols:
        if c in frame.columns:
            frame[c] = pd.to_numeric(frame[c], errors="coerce")
    return frame


# ---------------- environment ----------------
env = dict(generated=time.strftime("%Y-%m-%d %H:%M:%S"),
           python=platform.python_version(), torch=torch.__version__,
           cuda=torch.version.cuda, numpy=np.__version__,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           amp=bool(globals().get("AMP_ON", False)), precision=CFG.PRECISION,
           cache=str(CFG.CACHE_DIR),
           cudnn_benchmark=bool(torch.backends.cudnn.benchmark), seed=SEED,
           dataset_hash=getattr(CFG, "DATASET_HASH", None))

# ---------------- cross-validation results ----------------
cv_frames = {}
for _cond in REPORT_CONDS:
    _src = _first_existing(json_dir(CONDITION_USE_N2V[_cond]) / "cv_results.csv",
                           _results_json(cond_dir(_cond)))
    if _src is None:
        print(f"{_cond}: no CV results on disk")
        continue
    _df = (pd.read_csv(_src) if _src.suffix == ".csv"
           else pd.DataFrame(_read_json_rows(_src)))
    if "condition" in _df.columns:
        _df = _df[_df["condition"] == _cond]
    if _df.empty:
        continue
    _df = _num(_df, ("psnr", "ssim", "iou", "gauss_psnr", "fold", "epochs_run", "time"))
    _core = [m for m in ("psnr", "ssim", "iou") if m in _df.columns]
    _bad = _df[_df[_core].isna().any(axis=1)] if _core else _df.iloc[0:0]
    if len(_bad):
        # Rows seeded from checkpoints carry no metrics and would poison the means.
        print(f"{_cond}: excluding {len(_bad)} row(s) with missing metrics")
        _df = _df.dropna(subset=_core)
    if not _df.empty:
        _df["condition"] = _cond
        _df["source"] = str(_src)
        cv_frames[_cond] = _df

cv_all = (pd.concat(cv_frames.values(), ignore_index=True) if cv_frames
          else pd.DataFrame())

# ---------------- per-architecture, per-condition summary ----------------
per_arch, coverage = [], {}
for (_cond, _arch), _g in (cv_all.groupby(["condition", "arch"])
                           if not cv_all.empty else []):
    _best = _g.sort_values("iou", ascending=False).iloc[0]
    _row = dict(condition=_cond, arch=str(_arch),
                n_folds=int(_g["fold"].nunique()),
                folds=sorted(int(f) for f in _g["fold"].unique()),
                mean_iou=float(_g["iou"].mean()),
                std_iou=float(_g["iou"].std(ddof=1)) if len(_g) > 1 else None,
                mean_psnr=float(_g["psnr"].mean()),
                std_psnr=float(_g["psnr"].std(ddof=1)) if len(_g) > 1 else None,
                mean_ssim=float(_g["ssim"].mean()),
                best_fold=int(_best["fold"]), best_iou=float(_best["iou"]),
                best_psnr=float(_best["psnr"]),
                checkpoint=str(find_fold_ckpt(str(_arch), int(_best["fold"]), _cond) or ""))
    if "gauss_psnr" in _g and _g["gauss_psnr"].notna().any():
        _row["mean_gauss_psnr"] = float(_g["gauss_psnr"].mean())
        _row["psnr_gain_over_gaussian_dB"] = _row["mean_psnr"] - _row["mean_gauss_psnr"]
    per_arch.append(_row)
    coverage[f"{_cond}/{_arch}"] = _row["n_folds"]

per_arch_df = (pd.DataFrame(per_arch).sort_values(["condition", "mean_iou"],
                                                  ascending=[True, False])
               .reset_index(drop=True) if per_arch else pd.DataFrame())

# ---------------- the N2V contrast, paired by arch and fold ----------------
n2v_effect = []
if {"no_n2v", "with_n2v"} <= set(cv_frames):
    _a, _b = cv_frames["no_n2v"], cv_frames["with_n2v"]
    for _arch in CFG.ARCHS:
        _ga = _a[_a["arch"].astype(str) == _arch].set_index("fold")
        _gb = _b[_b["arch"].astype(str) == _arch].set_index("fold")
        _shared = _ga.index.intersection(_gb.index)
        if len(_shared) < 2:
            continue
        for _m in ("iou", "psnr", "ssim"):
            if _m not in _ga or _m not in _gb:
                continue
            _d = _gb.loc[_shared, _m] - _ga.loc[_shared, _m]
            _t, _p = stats.ttest_rel(_gb.loc[_shared, _m], _ga.loc[_shared, _m])
            n2v_effect.append(dict(arch=_arch, metric=_m, n_folds=len(_shared),
                                   without=float(_ga.loc[_shared, _m].mean()),
                                   with_n2v=float(_gb.loc[_shared, _m].mean()),
                                   delta=float(_d.mean()),
                                   sd=float(_d.std(ddof=1)),
                                   wins=int((_d > 0).sum()), p=float(_p)))

# ---------------- localization and precision ----------------
loc_across = _read_json(next(iter(sorted(
    (CFG.RESULTS_ROOT / "localization_v2").glob("*/localization_across_folds.json"),
    reverse=True)), None)) if (CFG.RESULTS_ROOT / "localization_v2").is_dir() else None
prec_summary = _read_json(CFG.RESULTS_ROOT / "precision" / "precision_summary.json")
panel_csv = CFG.RESULTS_ROOT / "panel_samples.csv"

# ---------------- caveats, derived rather than asserted ----------------
caveats = [
    "Folds are seeded KFold over basenames; if filenames encode simulation series, "
    "sibling frames can occupy train and validation of the same fold.",
    "cudnn.benchmark is enabled, so reruns are not bitwise reproducible.",
    "All reported folds are held-out validation splits, not an independent test set.",
]
_scale = (prec_summary or {}).get("pixel_size_A") or (loc_across or {}).get(
    "options", {}).get("pixel_size_A")
caveats.append(
    f"Spatial values use {_scale} A/px from "
    f"{(prec_summary or {}).get('calibration_source', 'an unnamed source')}."
    if _scale else
    "No verified pixel calibration was supplied, so every spatial value is in pixels.")
if len(cv_frames) == 2:
    caveats.append("Both a from-scratch (no_n2v) and an N2V warm-started (with_n2v) "
                   "arm were run; the paired contrast below isolates the warm start "
                   "only on folds present in both arms.")
    caveats.append("N2V pretraining used all noisy images, including images that later "
                   "served as validation in the CV folds (transductive; labels never seen).")
elif cv_frames:
    caveats.append(f"Only the {list(cv_frames)[0]} arm has results, so nothing here "
                   "isolates the effect of N2V pretraining.")
_short = {k: v for k, v in coverage.items() if v < CFG.N_FOLDS}
if _short:
    caveats.append(f"Fold counts are unequal: {_short} of {CFG.N_FOLDS}. "
                   "Cross-architecture and cross-condition means are provisional.")
_mixed = (sorted(set(cv_all.get("metric_source", pd.Series(dtype=str)).dropna()))
          if "metric_source" in cv_all.columns else [])
if len(_mixed) > 1:
    caveats.append(f"Metrics come from more than one evaluator ({_mixed}); rows marked "
                   "backfill_eval were measured after training on a subsample.")
_unstable = [f"{r['condition']}/{r['arch']}" for r in per_arch
             if r.get("psnr_gain_over_gaussian_dB") is not None
             and r["psnr_gain_over_gaussian_dB"] < 0]
if _unstable:
    caveats.append(f"Mean PSNR is below a Gaussian sigma=1 filter for {_unstable}; "
                   "the denoising head did not converge on at least some folds and "
                   "those means should not be quoted as denoising performance.")
if prec_summary:
    caveats.append("Precision values are pair RMSE from shifted-vs-unshifted "
                   "comparisons; the single-shot estimate assumes independent errors "
                   "at fractional pixel phase.")

# ---------------- assemble ----------------
artifacts = {}
for _label, _p in [("cv_results_all.csv", CFG.RESULTS_ROOT / "cv_results_all.csv"),
                   ("benchmark_comparison.csv", CFG.RESULTS_ROOT / "benchmark_comparison.csv"),
                   ("benchmark_summary.json", CFG.RESULTS_ROOT / "benchmark_summary.json"),
                   ("panel_samples.csv", panel_csv),
                   ("precision_per_fold.csv", CFG.RESULTS_ROOT / "precision" / "precision_per_fold.csv"),
                   ("precision_across_folds.png", CFG.RESULTS_ROOT / "precision" / "precision_across_folds.png")]:
    artifacts[_label] = Path(_p).is_file()
for _cond in cv_frames:
    _v = CONDITION_USE_N2V[_cond]
    artifacts[f"{_cond}/benchmark_bars.png"] = (variant_root(_v) / "benchmark_bars.png").is_file()
    for _arch in CFG.ARCHS:
        artifacts[f"{_cond}/{_arch}/panels"] = (arch_dir(_arch, _v, create=False) / "panels").is_dir()

report = dict(
    environment=env,
    config=dict(dataset_root=CFG.DATA_ROOT, results_root=str(CFG.RESULTS_ROOT),
                conditions=list(cv_frames), archs=list(CFG.ARCHS),
                n_samples=len(common), image_size=CFG.IMG_SIZE, batch=CFG.BATCH,
                lr=CFG.LR, wd=CFG.WD, n_folds=CFG.N_FOLDS, epochs_max=CFG.EPOCHS,
                n2v=dict(ratio=CFG.N2V_MASK_RATIO, radius=CFG.N2V_RADIUS,
                         checkpoints={a: dict(path=p, present=os.path.isfile(p))
                                      for a, p in N2V_CKPTS.items()})),
    per_architecture=per_arch, fold_coverage=coverage, n2v_effect=n2v_effect,
    localization=loc_across, precision=prec_summary,
    est_total_gpu_hours=(float((cv_all["epochs_run"] * cv_all["time"]).sum() / 3600)
                         if {"epochs_run", "time"} <= set(cv_all.columns) else None),
    fold_results=cv_all.to_dict("records") if not cv_all.empty else None,
    caveats=caveats, artifacts=artifacts)
_write_report_json(report, REPORT_ROOT / "final_report.json")

# ---------------- markdown ----------------
L = ["# TEM denoising and atomic-column localization: run report", "",
     f"generated {env['generated']} on {env['gpu'] or 'CPU'}, torch {env['torch']} "
     f"(CUDA {env['cuda']}), seed {env['seed']}",
     f"dataset: {len(common)} paired samples at {CFG.IMG_SIZE} px, "
     f"{CFG.N_FOLDS}-fold CV, conditions {', '.join(cv_frames) or 'none'}"
     + (f", ~{report['est_total_gpu_hours']:.1f} GPU-hours"
        if report["est_total_gpu_hours"] else ""), ""]

if not per_arch_df.empty:
    L += ["## benchmark", "",
          "| Condition | Arch | Folds | IoU | PSNR (dB) | SSIM | vs Gauss (dB) |",
          "|---|---|---:|---|---|---|---|"]
    for _, r in per_arch_df.iterrows():
        _pm = lambda m, s, d: (f"{m:.{d}f}" + ("" if s is None or pd.isna(s)
                                               else f" ± {s:.{d}f}"))
        L.append(f"| {r['condition']} | {r['arch']} | {r['n_folds']} | "
                 f"{_pm(r['mean_iou'], r['std_iou'], 3)} | "
                 f"{_pm(r['mean_psnr'], r['std_psnr'], 2)} | "
                 f"{r['mean_ssim']:.3f} | "
                 f"{r.get('psnr_gain_over_gaussian_dB', float('nan')):+.2f} |"
                 if pd.notna(r.get("psnr_gain_over_gaussian_dB"))
                 else f"| {r['condition']} | {r['arch']} | {r['n_folds']} | "
                      f"{_pm(r['mean_iou'], r['std_iou'], 3)} | "
                      f"{_pm(r['mean_psnr'], r['std_psnr'], 2)} | "
                      f"{r['mean_ssim']:.3f} | n/a |")
    L += [""]

if n2v_effect:
    L += ["## effect of N2V pretraining (paired by architecture and fold)", "",
          "| Arch | Metric | Folds | without | with | delta | wins | p |",
          "|---|---|---:|---|---|---|---|---|"]
    for e in n2v_effect:
        L.append(f"| {e['arch']} | {e['metric']} | {e['n_folds']} | "
                 f"{e['without']:.4f} | {e['with_n2v']:.4f} | "
                 f"{e['delta']:+.4f} ± {e['sd']:.4f} | "
                 f"{e['wins']}/{e['n_folds']} | {e['p']:.3f} |")
    L += ["", "A negative delta means the warm start made that metric worse on "
          "those folds.", ""]

if loc_across and loc_across.get("across_folds"):
    L += ["## localization across folds", "",
          "| Condition | Arch | Folds | F1 | RMSE (px) | Accepted fits |",
          "|---|---|---:|---|---|---|"]
    for r in loc_across["across_folds"]:
        def _p(k, d=3):
            m, s = r.get(k), r.get(f"{k}_sd")
            return "n/a" if m is None else (
                f"{m:.{d}f}" + ("" if s is None else f" ± {s:.{d}f}"))
        L.append(f"| {r['condition']} | {r['arch']} | {r['n_folds']} | "
                 f"{_p('gt_f1')} | {_p('gt_rmse_px', 2)} | "
                 f"{_p('refine_success_rate')} |")
    L += [""]

if prec_summary and prec_summary.get("across_folds"):
    L += ["## sub-pixel precision (shift recovery on noisy input)", "",
          "| Condition | Arch | Folds | Pair RMSE (px) | Single-shot (px) | Match rate |",
          "|---|---|---:|---|---|---|"]
    for r in prec_summary["across_folds"]:
        def _p(k, d=3):
            m, s = r.get(k), r.get(f"{k}_sd")
            return "n/a" if m is None else (
                f"{m:.{d}f}" + ("" if s is None else f" ± {s:.{d}f}"))
        L.append(f"| {r['condition']} | {r['arch']} | {r['n_folds']} | "
                 f"{_p('rmse_2d_px')} | {_p('single_shot_rmse_px')} | "
                 f"{_p('match_rate_refined')} |")
    L += [""]

L += ["## caveats", ""] + [f"{i}. {c}" for i, c in enumerate(caveats, 1)] + ["", 
     "## artifacts", ""] + [f"- [{'x' if ok else ' '}] {n}" for n, ok in artifacts.items()]
(REPORT_ROOT / "final_report.md").write_text("\n".join(L) + "\n", encoding="utf-8")

# ---------------- console ----------------
if not per_arch_df.empty:
    print(per_arch_df[["condition", "arch", "n_folds", "mean_iou", "std_iou",
                       "mean_psnr", "best_fold"]].round(4).to_string(index=False))
if n2v_effect:
    print("\nN2V effect:")
    for e in n2v_effect:
        print(f"  {e['arch']:11s} {e['metric']:5s} {e['without']:.4f} -> "
              f"{e['with_n2v']:.4f}  {e['delta']:+.4f} ± {e['sd']:.4f}  "
              f"wins {e['wins']}/{e['n_folds']}  p={e['p']:.3f}")
print("\ncaveats:")
for i, c in enumerate(caveats, 1):
    print(f"  {i}. {c}")
print(f"\nwrote final_report.json and final_report.md in {REPORT_ROOT}")

## Notes

Folder names. TEM-ImageNet-v1.3 ships `image/`, `noNoise/`, `circularMask/`,
`gaussianMask/` and `position/`. Section 3 auto-detects common variants and
prints what it matched; if something is missing, set `CFG.SUBDIR_*` directly.

Ground truth for localization. `position/` stores unit-cell vectors, not
per-atom coordinates, so section 10 takes intensity-weighted blob centroids from
`gaussianMask/`. `circularMask/` is the coarser fallback.

Pixel calibration. `CFG.PIXEL_SIZE_A` is 0.20 A/px, a placeholder for simulated
data. Set it to the real detector pixel size before quoting any spacing or RMSE
figure in angstroms. Appendix R estimates it from the diamond lattice for the
real HRTEM frames.

Speed. bf16 autocast (Ampere and later), channels-last, and a uint8 memmap cache
on the local disk. Watch the `img/s` and `GB` figures in each epoch line: if
throughput is well below the loader probe from section 4, the GPU is starved and
the cache is not working.

Data loading. `CFG.NUM_WORKERS` is 0 on Windows because spawned DataLoader
workers cannot import a dataset class defined in a notebook cell. The memmap
cache is what makes single-process loading fast enough.

Resuming. Every long cell checkpoints after each epoch. Rerun the cell after an
interrupt or a crash and it continues from the last completed epoch. The
walltime branches are inert off Slurm.

---
# Appendix R. Real-data transfer check

Unlabeled diamond HRTEM frames. There are no labels here, so nothing below is an
accuracy measurement. The point is to catch the failure modes that would make
later numbers meaningless:

- contrast polarity. If training columns were bright and these are dark, the
  model segments the gaps and every downstream number is confident nonsense.
- an indecisive probability map, everything near 0.5.
- denoiser hallucination: periodicity in the output that is absent from the raw
  FFT.

R1 through R1c are the transfer gate. R2 onward calibrate A/px from the diamond
lattice. Several R2 cells are diagnostics that were written while debugging the
ring finder, and they are labelled as such.

In [ ]:
# ============================================================
# R1. Real-frame transfer across all folds, architectures and conditions
# ============================================================
import glob

REAL_DIR   = os.environ.get("TEM_REAL_DATA", str(CFG.PROJECT_ROOT / "data" / "experimental"))
REAL_GLOB  = "Diamante-*.tif"
REAL_OUT   = CFG.RESULTS_ROOT / "real_diamond"
REAL_ARCHS = list(CFG.ARCHS)
REAL_CONDS = [k for k, on in (("no_n2v", RUN_NO_N2V),
                              ("with_n2v", RUN_WITH_N2V)) if on] or ["with_n2v"]
REAL_FOLDS = "all"          # "all" or an explicit list, e.g. [1, 2]
REAL_MODE  = "crop"         # "crop" keeps the native sampling; "resize" does not
REAL_BATCH = 4
SIM_COVERAGE = 0.137        # positive-pixel fraction seen during training
REAL_OUT.mkdir(parents=True, exist_ok=True)

real_paths = sorted(glob.glob(os.path.join(REAL_DIR, REAL_GLOB)))
if not real_paths:
    raise FileNotFoundError(f"no files matching {REAL_GLOB} in {REAL_DIR}")
print(f"{len(real_paths)} real image(s)")


def read_real(path, size=CFG.IMG_SIZE, mode=REAL_MODE):
    """16-bit TIFF to float32 in [0,1].

    Normalization matches training (1st/99th percentile). The 0.1/99.9 window
    used earlier is a different transform, and a model is sensitive to the one
    it was trained under.

    mode='crop' takes the centre CFG.IMG_SIZE window, preserving the native
    sampling rate. Resizing a real micrograph changes the lattice spacing in
    pixels, so a model trained at one sampling sees features at the wrong
    scale — that alone can look like failed transfer.
    """
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        raise IOError(f"could not read {path}")
    if im.ndim == 3:
        im = im[..., 0]
    im = im.astype(np.float32)
    lo, hi = np.percentile(im, [1, 99])
    im = np.clip((im - lo) / max(hi - lo, 1e-8), 0, 1)
    h, w = im.shape
    if mode == "crop" and h >= size and w >= size:
        y, x = (h - size) // 2, (w - size) // 2
        im = im[y:y + size, x:x + size]
    elif im.shape != (size, size):
        im = center_or_resize(im, size, cv2.INTER_AREA)
    return np.ascontiguousarray(im, dtype=np.float32)


def _power(img):
    """Hann-windowed power spectrum, DC-centred. The window stops edge
    discontinuities smearing a cross through the transform."""
    h, w = img.shape
    win = np.outer(np.hanning(h), np.hanning(w))
    return np.abs(np.fft.fftshift(np.fft.fft2((img - img.mean()) * win)))


def radial_profile(P):
    h, w = P.shape
    yy, xx = np.mgrid[0:h, 0:w]
    r = np.hypot(yy - h // 2, xx - w // 2).astype(int)
    return np.bincount(r.ravel(), weights=P.ravel()) / np.maximum(np.bincount(r.ravel()), 1)


def lattice_peak(img, r_min=4):
    """Strongest non-DC radial peak and its strength over the local background.
    Denoising that destroys this peak has removed the lattice, whatever the
    image looks like."""
    prof = radial_profile(_power(img))
    prof = prof[r_min:len(prof) // 2]
    if len(prof) < 5:
        return None, None
    k = int(np.argmax(prof))
    lo = max(0, k - 6), max(0, k - 3)
    hi = min(len(prof), k + 4), min(len(prof), k + 7)
    bg = np.concatenate([prof[lo[0]:lo[1]], prof[hi[0]:hi[1]]])
    bg = float(np.median(bg)) if len(bg) else float(np.median(prof))
    return k + r_min, float(prof[k] / max(bg, 1e-12))


# ---------------- load images once ----------------
real_imgs, native = {}, {}
for p in real_paths:
    raw = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    native[os.path.basename(p)] = (raw.shape, str(raw.dtype))
    real_imgs[os.path.basename(p)] = read_real(p)
print(f"\nnative sizes ({REAL_MODE} to {CFG.IMG_SIZE} px):")
for n, (shp, dt) in native.items():
    print(f"  {n:26s} {shp} {dt}")

real_names = list(real_imgs)
stack = np.stack([real_imgs[n] for n in real_names])
raw_peak = {n: lattice_peak(real_imgs[n]) for n in real_names}


@torch.inference_mode()
def run_batch(model, imgs):
    x = torch.from_numpy(np.stack(imgs))[:, None].float().to(
        DEVICE, memory_format=torch.channels_last)
    with amp_autocast():
        den, seg = model(x)
    return (den.float().clamp(0, 1)[:, 0].cpu().numpy(),
            torch.sigmoid(seg.float())[:, 0].cpu().numpy())


# ---------------- jobs ----------------
if not CKPT_INDEX:
    rebuild_ckpt_index()
real_jobs = []
for _c in REAL_CONDS:
    for (_a, _f), _ck in sorted(CKPT_INDEX.get(_c, {}).items()):
        if _a not in REAL_ARCHS:
            continue
        if REAL_FOLDS != "all" and int(_f) not in {int(x) for x in REAL_FOLDS}:
            continue
        real_jobs.append((_c, _a, int(_f), Path(_ck)))
if not real_jobs:
    raise RuntimeError("no checkpoints; check CKPT_INDEX in section 9a")
print(f"\n{len(real_jobs)} model(s) x {len(real_names)} image(s)")

# Running sums for the cross-model consensus: mean probability and the
# per-pixel spread between independently trained folds.
psum = np.zeros((len(real_names), CFG.IMG_SIZE, CFG.IMG_SIZE), np.float64)
psq = np.zeros_like(psum)
nmod = 0
rows, per_model_mean = [], {}

for _i, (_cond, _arch, _fold, _ck) in enumerate(real_jobs, 1):
    _model = None
    try:
        _model = prep_model(ARCH_REGISTRY[_arch]())
        load_model_weights(_ck, _model)
        _model.eval()
        _dens, _probs = [], []
        for _b in range(0, len(stack), REAL_BATCH):
            _d, _p = run_batch(_model, stack[_b:_b + REAL_BATCH])
            _dens.append(_d)
            _probs.append(_p)
        _dens, _probs = np.concatenate(_dens), np.concatenate(_probs)
    except Exception as _e:
        print(f"[{_i}/{len(real_jobs)}] {_cond} {_arch} fold {_fold}: "
              f"{_e.__class__.__name__}: {_e}")
        continue
    finally:
        if _model is not None:
            del _model
        free_gpu()

    psum += _probs
    psq += _probs.astype(np.float64) ** 2
    nmod += 1
    per_model_mean[(_cond, _arch, _fold)] = _probs.mean(axis=0)

    for _j, _n in enumerate(real_names):
        _img, _prob, _den = real_imgs[_n], _probs[_j], _dens[_j]
        # Sign of the correlation between probability and raw intensity:
        # positive means the model calls BRIGHT features atoms.
        _corr = float(np.corrcoef(_prob.ravel(), _img.ravel())[0, 1])
        _kd, _sd = lattice_peak(_den)
        _kr, _sr = raw_peak[_n]
        rows.append(dict(
            condition=_cond, arch=_arch, fold=_fold, image=_n,
            frac_pos=float((_prob > 0.5).mean()), p_mean=float(_prob.mean()),
            p_frac_undecided=float(((_prob > 0.3) & (_prob < 0.7)).mean()),
            corr=_corr,
            polarity="bright" if _corr > 0.1 else ("dark" if _corr < -0.1 else "unclear"),
            fft_peak_raw_k=_kr, fft_peak_raw_strength=_sr,
            fft_peak_den_k=_kd, fft_peak_den_strength=_sd,
            fft_peak_kept=(None if _sr is None or _sd is None else bool(
                _kd is not None and _kr is not None and abs(_kd - _kr) <= 2
                and _sd >= 0.5 * _sr))))
    print(f"[{_i}/{len(real_jobs)}] {_cond:9s} {_arch:11s} fold {_fold}  "
          f"coverage {np.mean([(p > .5).mean() for p in _probs]):.3f}  "
          f"corr {np.mean([np.corrcoef(p.ravel(), real_imgs[n].ravel())[0,1] for p, n in zip(_probs, real_names)]):+.3f}",
          flush=True)

if not rows:
    raise RuntimeError("every model failed")
real_df = pd.DataFrame(rows)
real_df.to_csv(REAL_OUT / "transfer_per_model_image.csv", index=False)

# ---------------- consensus ----------------
pmean = psum / nmod
pstd = np.sqrt(np.maximum(psq / nmod - (psum / nmod) ** 2, 0))

# ---------------- summary ----------------
print("\n=== per condition and architecture (median over folds and images) ===")
_agg = (real_df.groupby(["condition", "arch"])
        .agg(folds=("fold", "nunique"), coverage=("frac_pos", "median"),
             p_mean=("p_mean", "median"), undecided=("p_frac_undecided", "median"),
             corr=("corr", "median"),
             fft_kept=("fft_peak_kept", lambda s: float(pd.Series(s).mean())))
        .reset_index())
print(_agg.round(4).to_string(index=False))
_agg.to_csv(REAL_OUT / "transfer_by_arch.csv", index=False)

_pol = real_df["polarity"].value_counts().to_dict()
print(f"\npolarity across all model-image pairs: {_pol}")
if _pol.get("unclear", 0) > len(real_df) / 2:
    print("  -> the mask does not track intensity either way. Either transfer is "
          "failing, or these frames sit near a defocus where column contrast is "
          "weak. Judge by the panels, not the number.")
if _pol.get("dark") and _pol.get("bright"):
    print("  -> polarity flips between images or models. Physically plausible "
          "across defocus, but one subset is segmented inverted relative to the "
          "other; do not pool them.")

_cov = real_df["frac_pos"]
print(f"coverage {_cov.min():.3f}-{_cov.max():.3f} against {SIM_COVERAGE:.3f} in "
      f"training" + ("  -> outside the trained regime, so the 0.5 threshold does "
                     "not carry over" if _cov.min() < 0.01 or _cov.max() > 0.5 else ""))
_disagree = float(pstd.mean())
print(f"cross-model disagreement: mean per-pixel sd {_disagree:.3f} over {nmod} "
      f"model(s)" + ("  -> the folds do not agree; treat any single-model mask on "
                     "these frames as unreliable" if _disagree > 0.15 else ""))
_kept = real_df["fft_peak_kept"].dropna()
if len(_kept):
    print(f"lattice FFT peak preserved after denoising in "
          f"{100*_kept.mean():.0f}% of model-image pairs")

# ---------------- figures ----------------
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
ax[0].hist(np.concatenate([r for r in pmean])[::11], bins=80,
           color="steelblue", edgecolor="k")
ax[0].set_yscale("log")
ax[0].set_xlabel("consensus probability")
ax[0].set_ylabel("pixels (log)")
ax[0].set_title("Pooled consensus histogram\n(bimodal = committing, "
                "one hump near 0.5 = guessing)")
for (c, a), g in real_df.groupby(["condition", "arch"]):
    m = g.groupby("image")["frac_pos"].median().reindex(real_names)
    ax[1].plot(range(len(real_names)), m.values, marker="o", ms=4,
               ls="-" if c == "no_n2v" else "--", label=f"{a} · {c}")
ax[1].axhline(SIM_COVERAGE, ls="--", c="r", lw=1, label="training coverage")
ax[1].set_xticks(range(len(real_names)))
ax[1].set_xticklabels([n.split("-")[-1].replace(".tif", "") for n in real_names],
                      rotation=90, fontsize=7)
ax[1].set_ylabel("fraction of pixels > 0.5")
ax[1].legend(fontsize=6, ncol=2)
ax[1].set_title("Predicted atom coverage")
sns.boxplot(data=real_df, x="arch", y="corr", hue="condition", ax=ax[2])
ax[2].axhline(0, c="k", lw=.8)
ax[2].set_ylabel("corr(probability, intensity)")
ax[2].set_title("Polarity per architecture\n(positive = bright features called atoms)")
plt.tight_layout()
plt.savefig(REAL_OUT / "transfer_probability.png", dpi=150, bbox_inches="tight")
plt.show()

for _j, _n in enumerate(real_names[:4]):
    _img = real_imgs[_n]
    fig, axes = plt.subplots(1, 6, figsize=(23, 4))
    for _a, _im, _t, _kw in (
            (axes[0], _img, "Raw", dict(cmap="gray")),
            (axes[1], pmean[_j], f"Consensus prob. ({nmod} models)",
             dict(cmap="magma", vmin=0, vmax=1)),
            (axes[2], pstd[_j], f"Model disagreement (sd, max {pstd[_j].max():.2f})",
             dict(cmap="viridis")),
            (axes[3], np.log1p(_power(_img)), "FFT raw", dict(cmap="inferno"))):
        _a.imshow(_im, **_kw)
        _a.set_title(_t, fontsize=10)
        _a.axis("off")
    axes[4].imshow(_img, cmap="gray")
    axes[4].contour(pmean[_j], levels=[0.5], colors="#00ff90", linewidths=.6)
    axes[4].set_title("Consensus mask outline at 0.5", fontsize=10)
    axes[4].axis("off")
    _pr, _pd_ = radial_profile(_power(_img)), None
    axes[5].semilogy(_pr[:len(_pr)//2], color="k", lw=1, label="raw")
    if raw_peak[_n][0]:
        axes[5].axvline(raw_peak[_n][0], ls="--", c="r", lw=1,
                        label=f"peak k={raw_peak[_n][0]}")
    axes[5].set_xlabel("radial spatial frequency (px$^{-1}$ index)")
    axes[5].set_ylabel("power")
    axes[5].legend(fontsize=7)
    axes[5].set_title("Radial power", fontsize=10)
    fig.suptitle(_n, fontsize=12)
    plt.tight_layout()
    plt.savefig(REAL_OUT / f"transfer_{_n}.png", dpi=150, bbox_inches="tight")
    plt.show()

np.savez_compressed(REAL_OUT / "consensus.npz", names=np.array(real_names),
                    mean=pmean.astype(np.float32), sd=pstd.astype(np.float32))
print(f"\n-> {REAL_OUT}")

In [ ]:
# ============================================================
# R1b. Automated transfer verdict
#
# Replaces by-eye judgement with measurements. Every image gets a per-check
# pass/fail and an overall verdict. Nothing downstream should run on an image
# that fails LATTICE or HALLUCINATION.
# ============================================================
# thresholds, deliberately loose: they separate "obviously fine" from
# "obviously broken", not fine gradations
LATTICE_SNR_MIN = 3.0    # FFT spot peak / local background
HALLUC_GAIN_MAX = 2.5    # denoised spot power / raw spot power at NEW freqs
POLARITY_MIN    = 0.10   # |corr(prob, intensity)|
STRUCT_CORR_MIN = 0.80   # corr(raw, denoised) after matched low-pass


def check_lattice(img, r_min=8, r_max_frac=0.45):
    """Is there ANY crystalline reflection? Compares the brightest isolated peak
    against the azimuthally averaged background at the same radius. An amorphous
    or low-magnification image has no peak above its own halo."""
    P = _power(img)
    h, w = P.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.mgrid[0:h, 0:w]
    rr = np.hypot(yy - cy, xx - cx)
    band = (rr > r_min) & (rr < r_max_frac * min(h, w))

    prof = _radial_profile(P)
    bg = prof[np.clip(rr.astype(int), 0, len(prof) - 1)]
    contrast = np.where(band, P / np.maximum(bg, 1e-9), 0)

    peaks = (contrast == maximum_filter(contrast, size=7)) & (contrast > 0)
    if not peaks.any():
        return dict(snr=0.0, ok=False, R_px=None)
    idx = np.argwhere(peaks)
    vals = contrast[idx[:, 0], idx[:, 1]]
    k = int(np.argmax(vals))
    return dict(snr=float(vals[k]), ok=bool(vals[k] >= LATTICE_SNR_MIN),
                R_px=float(np.hypot(idx[k, 0] - cy, idx[k, 1] - cx)))


def check_hallucination(raw, den, r_min=8):
    """Power the denoiser ADDED at frequencies where the raw image had none. A
    real denoiser raises existing spots and lowers the floor; a hallucinating one
    creates power at frequencies absent from the input."""
    Pr, Pd = _power(raw), _power(den)
    h, w = Pr.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.mgrid[0:h, 0:w]
    rr = np.hypot(yy - cy, xx - cx)
    band = rr > r_min

    pr = _radial_profile(Pr)
    bgr = pr[np.clip(rr.astype(int), 0, len(pr) - 1)]
    quiet = band & (Pr < 1.2 * bgr)     # at or below the raw radial background
    if quiet.sum() < 100:
        return dict(gain=float('nan'), ok=True, frac_new=0.0)

    gain = float(Pd[quiet].sum() / max(Pr[quiet].sum(), 1e-9))
    frac_new = float(Pd[quiet].sum() / max(Pd[band].sum(), 1e-9))
    return dict(gain=gain, ok=bool(gain <= HALLUC_GAIN_MAX), frac_new=frac_new)


def check_structure(raw, den, sigma=2.0):
    """Is the denoised image still the same specimen? Compare at a scale the
    denoiser is not supposed to touch. Low correlation means the coarse structure
    itself was rewritten, not just the noise."""
    c = float(np.corrcoef(gaussian_filter(raw, sigma).ravel(),
                          gaussian_filter(den, sigma).ravel())[0, 1])
    return dict(corr=c, ok=bool(c >= STRUCT_CORR_MIN))


def check_polarity(prob, img):
    c = float(np.corrcoef(prob.ravel(), img.ravel())[0, 1])
    pol = "bright" if c > POLARITY_MIN else ("dark" if c < -POLARITY_MIN else "unclear")
    return dict(corr=c, polarity=pol, ok=bool(abs(c) >= POLARITY_MIN))


verdicts = []
for r in rows:                      # rows comes from R1
    lat = check_lattice(r['img'])
    hal = check_hallucination(r['img'], r['den'])
    st = check_structure(r['img'], r['den'])
    pol = check_polarity(r['prob'], r['img'])

    # LATTICE is the gate. Without resolved fringes the other checks measure the
    # model's response to an image it was never meant to see.
    if not lat['ok']:
        verdict = "NOT LATTICE-RESOLVED"
    elif not hal['ok']:
        verdict = "HALLUCINATING"
    elif not st['ok']:
        verdict = "STRUCTURE ALTERED"
    elif not pol['ok']:
        verdict = "MASK UNINFORMATIVE"
    else:
        verdict = "USABLE"

    verdicts.append(dict(name=r['name'], lat_snr=lat['snr'], lat_R=lat['R_px'],
                         hal_gain=hal['gain'], hal_frac_new=hal['frac_new'],
                         struct_corr=st['corr'], pol_corr=pol['corr'],
                         polarity=pol['polarity'], verdict=verdict))

vdf = pd.DataFrame(verdicts)
vdf.to_csv(os.path.join(REAL_OUT, 'transfer_verdict.csv'), index=False)

print(f"{'image':24s} {'latSNR':>7s} {'R px':>7s} {'halGain':>8s} "
      f"{'struct':>7s} {'pol':>7s}  verdict")
for _, v in vdf.iterrows():
    print(f"{v['name']:24s} {v.lat_snr:7.2f} "
          f"{(v.lat_R if v.lat_R is not None else float('nan')):7.1f} "
          f"{v.hal_gain:8.2f} {v.struct_corr:7.3f} {v.pol_corr:+7.3f}  {v.verdict}")

USABLE = vdf.loc[vdf.verdict == "USABLE", 'name'].tolist()
print(f"\n{len(USABLE)}/{len(vdf)} usable: {USABLE or 'none'}")
print(vdf.verdict.value_counts().to_string())

if not USABLE:
    print("\nNo image passes. Nothing downstream (calibration, spacing, precision) "
          "should be run: on an image with no resolved lattice the FFT has no "
          "reflections to index, so any A/px it produces is fitted to noise.")
    print("Check the magnification: a 100 nm scale bar is ~1000x too coarse for the "
          "~0.2 A/px this model was trained at.")

In [ ]:
# ============================================================
# R1c. Stratify the usable set by lattice quality
#
# LATTICE_SNR_MIN=3.0 is a loose gate and several images passed at 3.7-3.9, the
# same range as images that failed on other checks. The SNR distribution is
# bimodal, so split on the gap rather than on the original threshold.
# ============================================================
STRONG_SNR    = 7.0     # the gap in this dataset sits between ~6.7 and ~14.5
HALLUC_STRICT = 0.9     # a denoiser should REMOVE power at quiet frequencies

u = vdf[vdf.verdict == "USABLE"].copy()
u['tier'] = np.where((u.lat_snr >= STRONG_SNR) & (u.hal_gain <= HALLUC_STRICT),
                     'primary', 'marginal')

print(f"SNR distribution of usable images: {sorted(u.lat_snr.round(1).tolist())}")
print(f"\nprimary  ({int((u.tier=='primary').sum())}): "
      f"{u.loc[u.tier=='primary','name'].tolist()}")
print(f"marginal ({int((u.tier=='marginal').sum())}): "
      f"{u.loc[u.tier=='marginal','name'].tolist()}")

_drop = u[(u.tier == 'marginal') & (u.hal_gain > HALLUC_STRICT)]
if len(_drop):
    print(f"\nadding power at quiet frequencies, excluded despite passing: "
          f"{_drop.name.tolist()} (gain {_drop.hal_gain.round(2).tolist()})")

PRIMARY = u.loc[u.tier == 'primary', 'name'].tolist()

# Is the primary set one magnification, or several?
pr_tier = u[u.tier == 'primary']
print(f"\nring radius across primary set: {pr_tier.lat_R.mean():.1f} ± "
      f"{pr_tier.lat_R.std(ddof=1):.1f} px "
      f"(range {pr_tier.lat_R.min():.0f}-{pr_tier.lat_R.max():.0f})")
if pr_tier.lat_R.std(ddof=1) / pr_tier.lat_R.mean() > 0.10:
    print("  -> spread above 10%. These frames are NOT all at one magnification or "
          "zone axis. Group them by radius before calibrating; a single pooled "
          "A/px would be wrong for most of them.")
    print(pr_tier[['name', 'lat_R', 'lat_snr']].sort_values('lat_R').to_string(index=False))
else:
    print("  -> consistent; one calibration covers the primary set.")

u.to_csv(os.path.join(REAL_OUT, 'usable_tiers.csv'), index=False)

In [ ]:
# ============================================================
# R2. Per-image ring indexing and A/px
#
# Indexing each image against its OWN internal ring ratios is assumption-free:
# the ratio 220/111 = sqrt(8/3) is a property of the diamond structure, not of
# the microscope. A strongest-ring-only approach cannot tell {111} from {220}.
# A/px is computed only for images whose indexing is self-consistent.
# ============================================================
A_DIAMOND = 3.567
# allowed reflections for the diamond structure (h,k,l all even or all odd, and
# h+k+l != 4n+2)
REFL = {"111": np.sqrt(3), "220": np.sqrt(8), "311": np.sqrt(11),
        "400": np.sqrt(16), "331": np.sqrt(19), "422": np.sqrt(24)}
D_OF = {k: A_DIAMOND / v for k, v in REFL.items()}
RATIO_TOL = 0.04          # fractional tolerance on a ring-ratio match

print("expected d-spacings (A):", {k: round(v, 3) for k, v in D_OF.items()})


def find_rings(img, n_rings=6, r_min=8, r_max_frac=0.45, min_sep=7, snr_min=2.0):
    """Radii of the distinct diffraction rings, strongest first. Peaks are scored
    against the azimuthal background at their own radius, so a bright
    low-frequency halo does not outrank a genuine high-order spot.

    NOTE: R2c below diagnoses why this version over-reaches on 1792 px frames,
    and R2d redefines find_rings with a bounded radius and a proportional merge
    tolerance. The indexing in this cell uses THIS version."""
    P = _power(img)
    h, w = P.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.mgrid[0:h, 0:w]
    rr = np.hypot(yy - cy, xx - cx)
    band = (rr > r_min) & (rr < r_max_frac * min(h, w))

    prof = _radial_profile(P)
    bg = prof[np.clip(rr.astype(int), 0, len(prof) - 1)]
    contrast = np.where(band, P / np.maximum(bg, 1e-9), 0)

    idx = np.argwhere((contrast == maximum_filter(contrast, size=min_sep))
                      & (contrast > snr_min))
    if len(idx) == 0:
        return []
    snr = contrast[idx[:, 0], idx[:, 1]]
    rad = np.hypot(idx[:, 0] - cy, idx[:, 1] - cx)

    # Friedel pairs and the several spots of one ring share a radius; merge them
    order = np.argsort(rad)
    rad, snr = rad[order], snr[order]
    rings, cur_r, cur_s = [], [rad[0]], [snr[0]]
    for r, s in zip(rad[1:], snr[1:]):
        if r - cur_r[-1] < 4:
            cur_r.append(r); cur_s.append(s)
        else:
            rings.append((float(np.mean(cur_r)), float(np.max(cur_s)), len(cur_r)))
            cur_r, cur_s = [r], [s]
    rings.append((float(np.mean(cur_r)), float(np.max(cur_s)), len(cur_r)))
    return sorted(rings, key=lambda t: -t[1])[:n_rings]


def index_image(rings):
    """Assign hkl labels by testing every hypothesis for what the STRONGEST ring
    is, and scoring how well the remaining rings then land on allowed
    reflections. With one ring only, no assignment is testable."""
    if len(rings) < 2:
        return None
    radii = np.array([r for r, _, _ in rings])
    strongest = radii[0]
    best = None
    for name, g in REFL.items():
        # if the strongest ring is `name`, radius scales as sqrt(h^2+k^2+l^2)
        scale = strongest / g
        matched, resid = [], []
        for r in radii:
            pred = {k: scale * v for k, v in REFL.items()}
            k = min(pred, key=lambda k: abs(pred[k] - r))
            err = abs(pred[k] - r) / r
            if err < RATIO_TOL:
                matched.append((float(r), k, float(err)))
                resid.append(err)
        if len(matched) < 2:
            continue
        score = len(matched) - 3 * float(np.mean(resid))
        if best is None or score > best['score']:
            best = dict(assume=name, scale=scale, matched=matched,
                        n_matched=len(matched), mean_resid=float(np.mean(resid)),
                        score=score)
    return best


print(f"\n{'image':24s} {'rings (px)':>34s}  {'strongest=':>10s} "
      f"{'n':>2s} {'resid':>6s} {'A/px':>7s}")
cal2 = []
for name in PRIMARY:
    raw = read_real(os.path.join(REAL_DIR, name))       # native size
    rings = find_rings(raw)
    ix = index_image(rings)
    rs = ", ".join(f"{r:.0f}" for r, _, _ in rings[:5])
    if ix is None:
        print(f"{name:24s} {rs:>34s}  {'-':>10s} {'-':>2s} {'-':>6s} {'unindexed':>7s}")
        cal2.append(dict(name=name, indexed=False))
        continue
    # A/px from EVERY matched ring, not just the strongest
    N = min(raw.shape)
    est = [D_OF[k] * r / N for r, k, _ in ix['matched']]
    apx = float(np.mean(est))
    print(f"{name:24s} {rs:>34s}  {ix['assume']:>10s} {ix['n_matched']:2d} "
          f"{ix['mean_resid']:6.3f} {apx:7.4f}")
    cal2.append(dict(name=name, indexed=True, assume=ix['assume'],
                     n_matched=ix['n_matched'], resid=ix['mean_resid'],
                     A_per_px=apx, A_spread=float(np.std(est)),
                     rings=[r for r, _, _ in rings[:5]]))

c2 = pd.DataFrame(cal2)
ok = c2[c2.indexed == True] if 'indexed' in c2 else c2.iloc[0:0]

if len(ok):
    print(f"\nindexed {len(ok)}/{len(PRIMARY)}")
    print(f"assignments of the strongest ring: {ok.assume.value_counts().to_dict()}")
    print(f"A/px: {ok.A_per_px.mean():.4f} ± {ok.A_per_px.std(ddof=1):.4f} "
          f"(range {ok.A_per_px.min():.4f}-{ok.A_per_px.max():.4f})")
    print(f"CFG.PIXEL_SIZE_A placeholder: {CFG.PIXEL_SIZE_A:.4f}")

    _cv = ok.A_per_px.std(ddof=1) / ok.A_per_px.mean()
    if ok.assume.nunique() > 1:
        print("\n  -> the strongest ring is NOT the same reflection in every image. "
              "That, not magnification, explains the radius spread. The A/px values "
              "above already account for it.")
    if _cv < 0.05:
        print(f"\n  -> A/px consistent to {100*_cv:.1f}% once indexed. Use the mean: "
              f"CFG.PIXEL_SIZE_A = {ok.A_per_px.mean():.4f}")
    else:
        print(f"\n  -> A/px still spreads {100*_cv:.1f}% after indexing, so "
              f"magnification genuinely differs between frames. Use the per-image "
              f"value from this table, not a pooled mean.")
    ok.to_csv(os.path.join(REAL_OUT, 'ring_indexing.csv'), index=False)
else:
    print("\nNo image could be indexed: fewer than two rings found anywhere. Lower "
          "snr_min in find_rings, or these frames really do show only one "
          "reflection, in which case the assignment cannot be determined from the "
          "data alone and you need the microscope's magnification.")

### R2c to R2f: diagnostics

These four cells were written while debugging the ring finder and the TIFF
metadata. They print and plot; they do not feed anything downstream, so they can
be skipped on a rerun.

In [ ]:
# ============================================================
# R2g. [superseded] Single-ring calibration
#
# Kept for comparison with R2. This assumes the strongest ring IS a given
# reflection instead of testing the assignment against the ring ratios, so it
# cannot tell {111} from {220}. Prefer the R2 numbers. Calibration is done on
# the RAW frame on purpose: the denoiser must not be in the loop that defines
# the ruler.
#
# Diamond a = 3.567 A, so the strong HRTEM reflections sit at known d:
#   111 -> 2.059 A, 220 -> 1.261 A, 311 -> 1.075 A
# ============================================================
DIAMOND_D = {k: D_OF[k] for k in ("111", "220", "311")}


def fft_peaks(img, n_peaks=12, min_sep=6, r_min=6, r_max_frac=0.45):
    """Brightest isolated spots in the power spectrum, as radii in px. DC and
    the low-frequency background are excluded."""
    P = _power(img)
    h, w = P.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.mgrid[0:h, 0:w]
    rr = np.hypot(yy - cy, xx - cx)
    P = np.where((rr > r_min) & (rr < r_max_frac * min(h, w)), P, 0)

    cand = np.argwhere((P == maximum_filter(P, size=min_sep)) & (P > 0))
    vals = P[cand[:, 0], cand[:, 1]]
    order = np.argsort(vals)[::-1][:n_peaks]
    cand, vals = cand[order], vals[order]
    return np.hypot(cand[:, 0] - cy, cand[:, 1] - cx), vals, cand, P


def calibrate(img, hkl="111"):
    """A/px assuming the strongest ring is the given reflection. A spot at radius
    R px in an N-px transform corresponds to a real-space spacing of N/R pixels,
    so A/px = d_hkl * R / N."""
    N = min(img.shape)
    radii, vals, cand, P = fft_peaks(img)
    if len(radii) == 0:
        return None
    # Friedel pairs put every spot at +-R, so cluster radii before choosing
    r_sorted = np.sort(radii)
    groups, cur = [], [r_sorted[0]]
    for r in r_sorted[1:]:
        if r - cur[-1] < 3:
            cur.append(r)
        else:
            groups.append(cur); cur = [r]
    groups.append(cur)
    best, best_w = None, -1
    for g in groups:                       # strongest group, intensity-weighted
        w = vals[np.isin(radii, g)].sum()
        if w > best_w:
            best, best_w = float(np.mean(g)), w
    return dict(R_px=best, N=N, A_per_px=DIAMOND_D[hkl] * best / N,
                radii=radii, vals=vals, cand=cand, P=P)


print(f"{'image':24s} {'R (px)':>8s} {'N':>6s} {'A/px (111)':>11s} {'A/px (220)':>11s}")
cal_rows = []
for p in real_paths:
    raw = read_real(p)                    # native size, NOT resized
    c = calibrate(raw, "111")
    if c is None:
        print(f"{os.path.basename(p):24s}   no usable FFT spots")
        continue
    a111 = c['A_per_px']
    a220 = DIAMOND_D['220'] * c['R_px'] / c['N']
    cal_rows.append(dict(name=os.path.basename(p), R_px=c['R_px'], N=c['N'],
                         a111=a111, a220=a220))
    print(f"{os.path.basename(p):24s} {c['R_px']:8.2f} {c['N']:6d} "
          f"{a111:11.4f} {a220:11.4f}")

if cal_rows:
    cal = pd.DataFrame(cal_rows)
    print(f"\nA/px assuming the strong ring is 111: "
          f"{cal.a111.mean():.4f} ± {cal.a111.std(ddof=1):.4f}")
    print(f"A/px assuming it is 220:              "
          f"{cal.a220.mean():.4f} ± {cal.a220.std(ddof=1):.4f}")
    print(f"CFG.PIXEL_SIZE_A placeholder is        {CFG.PIXEL_SIZE_A:.4f}")
    print("\nWhich assignment is right is decided in R2, not here: check the RATIO "
          "of the ring radii against sqrt(8/3) = 1.633 for 220/111, and confirm "
          "against the microscope's nominal magnification.")
    cal.to_csv(os.path.join(REAL_OUT, 'pixel_calibration.csv'), index=False)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].bar(range(len(cal)), cal.a111, color='steelblue')
    ax[0].axhline(CFG.PIXEL_SIZE_A, ls='--', c='r', label='CFG placeholder')
    ax[0].set_xticks(range(len(cal)))
    ax[0].set_xticklabels([n.split('-')[-1].replace('.tif', '') for n in cal.name],
                          rotation=90, fontsize=7)
    ax[0].set_ylabel('A/px (111 assumption)'); ax[0].legend(fontsize=8)
    ax[0].set_title('Per-image calibration')

    _c = calibrate(read_real(real_paths[0]), "111")
    ax[1].imshow(np.log1p(_c['P']), cmap='inferno')
    ax[1].scatter(_c['cand'][:, 1], _c['cand'][:, 0], s=40, facecolors='none',
                  edgecolors='cyan', linewidths=1)
    ax[1].set_title(f"Indexed spots: {os.path.basename(real_paths[0])}")
    ax[1].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(REAL_OUT, 'pixel_calibration.png'), dpi=150)
    plt.show()
else:
    print("\nNo image gave usable FFT spots. Either the frames are not "
          "lattice-resolved, or r_min/r_max_frac are cutting the rings out.")